# Masked tanh-GELU kernel for fp16 and fp32

The tanh approximation of GELU, used by GPT-2-style models:

$$\mathrm{gelu}(x) = \tfrac12\, x \left(1 + \tanh\!\bigl(\sqrt{2/\pi}\,(x + 0.044715\, x^3)\bigr)\right)$$

Write it as a Triton kernel that works for tensors of **any** shape and size, including sizes that are not a multiple of the block, and for both half and single precision.

Write `gelu_tanh(x)`.

## Inputs
- `x`: contiguous CUDA tensor of dtype `float16` or `float32`, any shape, possibly with 0 elements.

## Output
- A new tensor with the same shape and dtype as `x`.

## Requirements
- Treat the tensor as a flat array of `n = x.numel()` elements. A 1-D grid of `triton.cdiv(n, BLOCK)` programs, `BLOCK = 1024`; the last program must mask its out-of-range lanes on both load and store.
- Load, convert to `float32`, compute, and convert back to the output dtype on store. Do not compute in `float16`: $x^3$ overflows it for $|x| > 40$.
- $\tanh$ must not produce NaN for large $|x|$. Build it from `tl.exp` as $\tanh(u) = 1 - \dfrac{2}{e^{2u} + 1}$, which saturates cleanly to $\pm 1$.
- For `n == 0`, return an empty tensor without launching.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def gelu_tanh(x):
    """x: contiguous float16/float32 CUDA tensor. Return tanh-approximate GELU, same shape and dtype."""
    raise NotImplementedError


@triton.jit
def _gelu_kernel(x_ptr, y_ptr, n, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [-2.0, -1.0, 0.0, 1.0, 2.0], "dtype": "float32"}], "e": {"__nd__": [-0.04540230706334114, -0.15880800783634186, 0.0, 0.8411920070648193, 1.9545977115631104], "dtype": "float32"}}, {"a": [{"__nd__": [[0.5, -0.5], [3.0, -3.0]], "dtype": "float16"}], "e": {"__nd__": [[0.345703125, -0.154296875], [2.99609375, -0.0036373138427734375]], "dtype": "float16"}}, {"a": [{"__nd__": [0.5671601295471191, -1.5682452917099, -1.2391905784606934, -7.324402332305908, 5.39912223815918, 3.432497501373291, -0.9762685298919678, 2.3214197158813477, 0.8436319828033447, -1.661468505859375, 2.9327023029327393, -0.9316696524620056, -0.9864717125892639, -2.3764402866363525, 1.364874243736267, -0.2975941598415375, 1.6358661651611328, -1.8215570449829102, 0.3804835379123688, -2.6768221855163574, 2.524394989013672, 0.5641052722930908, 0.9917130470275879, 1.2315117120742798, -3.0322725772857666, 2.3495430946350098, 6.170108318328857, -4.915327548980713, -5.188234329223633, -4.5144944190979, 2.524376630783081, 0.3861469626426697, 3.235027313232422, 2.167292594909668, 0.6317154169082642, 0.8521144390106201, -0.5092815160751343, 2.6053805351257324, -3.389147996902466, -1.2655764818191528, 0.728816568851471, 5.404262542724609, -2.2933924198150635, -3.2371814250946045, -1.689861536026001, 2.9078166484832764, -0.7050164937973022, 3.97304105758667, -5.617588520050049, 3.3855693340301514, 3.1045987606048584, -4.255997180938721, 0.4607456624507904, 3.647272825241089, 0.2637571394443512, 2.9991042613983154, 7.124660968780518, 0.8217967748641968, -0.8411469459533691, -2.3131563663482666, 1.9441938400268555, -0.5901901721954346, -0.5362390875816345, -0.31577932834625244, 1.9496089220046997, -3.199018955230713, -4.589629650115967, -7.301630020141602, 3.596012592315674, 0.2213800847530365, 4.53035831451416, -0.026868894696235657, -2.226466417312622, 1.4337879419326782, -0.22976523637771606, -3.7625598907470703, -2.6551997661590576, 5.300337791442871, 1.0630518198013306, 1.249159574508667, -0.8296551704406738, -2.0691587924957275, 2.6749675273895264, -0.3138793706893921, -2.2773244380950928, -0.40238556265830994, -2.71681809425354, 0.5701375007629395, 3.3876829147338867, -2.5082554817199707, 4.285632133483887, -2.0028598308563232, 0.46023648977279663, -2.509159564971924, -0.6667054891586304, 0.14221324026584625, -1.304054856300354, -2.10866379737854, -2.033658504486084, -2.46347975730896, -4.710239887237549, -0.788908839225769, 1.2034746408462524, 2.725205659866333, 1.9413081407546997, 7.372009754180908, 0.9560449123382568, -1.3690016269683838, 5.614903926849365, -3.1416542530059814, 2.905432939529419, -2.865434408187866, 1.0623350143432617, -5.905190467834473, 2.6978211402893066, -0.4747428596019745, -2.9030444622039795, 5.035257339477539, 2.2960641384124756, 0.13742350041866302, -2.236330270767212, -0.13008776307106018, -0.49178630113601685, 2.1743295192718506, 2.394225597381592, -2.0032691955566406, -1.6481117010116577, -1.596195936203003, -4.047951698303223, -1.7747246026992798, -0.27722418308258057, 2.0728559494018555, 3.9611384868621826, -2.4250376224517822, 1.6529295444488525, -1.3333704471588135, 6.2255635261535645, -0.14617063105106354, 1.5063809156417847, -2.8084864616394043, -2.432068347930908, 0.6050859093666077, -1.1521633863449097, 1.0598658323287964, -4.68547248840332, 1.98601496219635, -2.6984410285949707, 5.167270183563232, -0.8654017448425293, 3.291795015335083, -4.18494987487793, 1.2868387699127197, -2.6640889644622803, -1.4606672525405884, 0.07378914952278137, -0.9512249231338501, 0.7927668690681458, 2.2429661750793457, 1.9583463668823242, -0.2624444365501404, -4.090291500091553, -2.413938283920288, -0.5872686505317688, -6.365478038787842, 2.256241798400879, -0.7189145684242249, -0.7539578676223755, 2.8451285362243652, 2.0037314891815186, 0.6630029082298279, -3.004000663757324, 0.9312647581100464, 1.0385326147079468, -2.654320001602173, 3.1299355030059814, 1.523661494255066, -5.908873081207275, 0.3658483624458313, -3.7940940856933594, 3.4089455604553223, 2.421083927154541, -3.310772180557251, -2.65461802482605, 0.30555984377861023, -0.7164324522018433, 4.57560920715332, 2.222921371459961, -0.8472817540168762, 1.7321877479553223, -6.096837997436523, 0.9295940399169922, 2.5992212295532227, -0.536857545375824, -2.1306426525115967, 1.6975003480911255, 5.8758440017700195, 0.6146067976951599, -3.362032890319824, 4.337414264678955, -5.6356425285339355, 1.5096145868301392, -1.745863914489746, 2.791795253753662, -2.5484230518341064, 3.971543312072754, 0.9008192420005798, -5.1805219650268555, -5.921439170837402, -0.9482865333557129, 3.8826348781585693, 0.38109010457992554, 0.20136210322380066, -0.7077115178108215, 2.213700532913208, 1.1319804191589355, 0.8410376310348511, -2.9541687965393066, 3.5635600090026855, 4.089241027832031, -3.9040606021881104, 6.90448522567749, 2.1675198078155518, -1.4640179872512817, -5.713067531585693, 1.9705392122268677, 1.6725341081619263, -1.786895513534546, -5.371659755706787, 3.3734302520751953, -1.7649568319320679, -1.4152352809906006, 8.536934852600098, 6.38391637802124, 2.8612101078033447, -1.3662630319595337, 1.9483838081359863, -6.844409465789795, 0.31959810853004456, -2.2794549465179443, -3.8568153381347656, -4.748413562774658, 1.5754247903823853, 0.26790112257003784, 0.4791709780693054, 2.527128219604492, -2.5382513999938965, -3.7067532539367676, -5.32909631729126, -1.0055897235870361, 2.280468702316284, -0.4482705891132355, 1.9233845472335815, -3.543161153793335, -1.407364010810852, -2.2978100776672363, -3.745133399963379, 5.578420639038086, 2.556422233581543, -2.2854015827178955, -2.298051595687866, -0.17159056663513184, -4.326161861419678, 1.3074815273284912, -3.882716417312622, -1.11129891872406, -3.074467897415161, -1.439544916152954, -3.993471622467041, -4.718466281890869, -2.8984227180480957, -2.040029525756836, -0.8846682906150818, 0.20695090293884277, 4.5806884765625, 2.436755418777466, -0.7586976289749146, 2.177755117416382, -2.0666918754577637, 4.3454742431640625, 1.6484309434890747, -4.06751823425293, 2.2661566734313965, 2.448276996612549, -8.056785583496094, -2.2442848682403564, -1.4214675426483154, -5.785536766052246, -3.2015323638916016, -2.8450746536254883, 1.521287441253662, 0.9013895392417908, 7.87852144241333, -1.182778239250183, 3.45969295501709, 0.23653942346572876, 0.8127773404121399, -3.4788894653320312, -5.732699394226074, -2.3014447689056396, 2.286505699157715, 4.748416900634766, 1.8119730949401855, 1.494842767715454, -2.516300678253174, 4.60288143157959, 4.031418800354004, -1.59249746799469, -0.13911201059818268, -2.9881558418273926, 4.401620864868164, 1.2190624475479126, -0.2851940095424652, 1.2207000255584717, 1.3218668699264526, -1.234490156173706, -3.7246954441070557, -3.85123610496521, -5.215099334716797, 1.3837590217590332, 6.655625343322754, -5.097208023071289, 0.41486236453056335, 1.2329518795013428, -2.9538769721984863, 0.6132774353027344, -3.2908105850219727, 3.6012063026428223, -1.7827471494674683, 0.30168116092681885, 2.052724838256836, -1.289458155632019, -5.373692512512207, -1.5937669277191162, 2.2889857292175293, -0.2535979747772217, 0.6893675327301025, -3.2412054538726807, 0.42966872453689575, 2.1845009326934814, 0.10934406518936157, 5.724043846130371, -0.623380720615387, -3.1240506172180176, -4.852977752685547, 3.4560177326202393, -0.1770547330379486, 3.034865379333496, 1.3730067014694214, 4.875252723693848, 3.7983222007751465, -1.1326637268066406, 0.8216546773910522, 3.2561159133911133, -3.7082254886627197, 4.352687835693359, 3.6359801292419434, 2.948777675628662, -0.6027964949607849, 0.9064230918884277, -3.351717710494995, 0.1738625168800354, -1.7621386051177979, 0.25969189405441284, -1.4838787317276, 0.8514105081558228, 3.537996530532837, -3.928809642791748, 2.776172637939453, -5.975437641143799, -6.483099937438965, -0.35810601711273193, 0.020188145339488983, 1.3284025192260742, -4.942155838012695, -0.17981593310832977, 0.37957409024238586, 3.7656149864196777, -0.31151875853538513, 1.5869752168655396, -2.932276725769043, -0.5626639127731323, -1.3366059064865112, 2.1099131107330322, -2.9664578437805176, -2.502967357635498, 1.7279645204544067, 5.36830997467041, 1.148110032081604, 0.910680890083313, -1.0384272336959839, 3.265275716781616, 1.6197508573532104, 5.199578285217285, 0.9921345114707947, -2.8460307121276855, -1.709824562072754, -2.9622137546539307, -3.75246262550354, 3.791152238845825, 2.422010660171509, -3.715933084487915, 5.3439621925354, 0.7336821556091309, -2.870676040649414, 1.1586657762527466, -0.2075827717781067, -2.3179643154144287, -5.828803062438965, 0.11015971004962921, 0.8759875297546387, -7.737232208251953, 1.664721965789795, 1.8528999090194702, -1.4093868732452393, 0.3363691568374634, -0.8419744968414307, -2.511307716369629, -1.080164909362793, 4.054811477661133, -4.351851463317871, 0.7115768194198608, 0.5523532629013062, -2.1228909492492676, -1.3210176229476929, -4.201240062713623, -2.5558865070343018, 0.04485549032688141, 0.9801477193832397, -2.6951303482055664, 9.169027328491211, -1.7110337018966675, 0.3819900155067444, -0.6917402148246765, -1.1993393898010254, 3.9032068252563477, -1.4364341497421265, 4.003166675567627, 3.4254488945007324, -3.136995553970337, 3.106480598449707, -1.951134204864502, 8.209376335144043, -1.1574910879135132, -2.8227787017822266, -5.73322868347168, -3.8791353702545166, -1.7419466972351074, -2.9940834045410156, -3.686060667037964, -2.086019277572632, -0.6196432113647461, 4.348042011260986, -5.109884738922119, -1.6549229621887207, -2.861680746078491, -4.545443534851074, -3.798409938812256, -2.7172045707702637, 2.151130199432373, -0.5817017555236816, -1.6139553785324097, 1.833875060081482, 0.8423084616661072, -0.6699865460395813, 1.1528575420379639, -2.485053062438965, 1.8821439743041992, 1.4007748365402222, -1.2369128465652466, -3.658428430557251, 4.560193061828613, 0.28260302543640137, 9.15379524230957, -1.9602272510528564, 2.5585291385650635, -1.3026574850082397, 0.7610041499137878, 2.500173568725586, -2.397923469543457, -0.3002713918685913, 0.48282477259635925, 0.3404403030872345, 1.7893531322479248, 3.0766894817352295, 2.500885009765625, 0.7890376448631287, -1.042118787765503, -3.3890507221221924, -2.2064952850341797, -1.585646390914917, 2.048142194747925, -1.0908305644989014, -3.7270162105560303, 1.8790414333343506, -2.409752130508423, -2.297485828399658, -0.6788142323493958, 1.4195334911346436, -7.239550590515137, -3.3643620014190674, 0.5267674326896667, -1.0924830436706543, -2.5936336517333984, -1.6243586540222168, 4.155762195587158, -1.3671847581863403, 2.1379950046539307, 2.3756072521209717, -1.3506630659103394, 1.3303172588348389, -1.3507128953933716, -1.8312525749206543, -3.3779830932617188, 2.624868869781494, -1.1944780349731445, -1.656427264213562, -3.3823347091674805, 5.426534652709961, 2.225416898727417, -0.39841341972351074, -2.084203004837036, -4.589399814605713, 0.8754410147666931, 0.865265429019928, -4.2252020835876465, 3.4846510887145996, 2.327869176864624, -2.886221170425415, -1.667089581489563, -1.2819619178771973, 0.48203688859939575, -0.4367082715034485, -4.9168171882629395, 4.422292232513428, 0.9489312767982483, 3.782320022583008, 3.7644670009613037, -0.6237024664878845, 1.0258864164352417, 2.209993839263916, 2.308840274810791, -0.49175629019737244, -1.8545564413070679, 0.10585971176624298, -2.404628276824951, -1.6678019762039185, -6.090538024902344, 0.25092801451683044, -3.080673933029175, -1.6675190925598145, -0.15810102224349976, 0.9415053725242615, 5.667515277862549, 0.6131891012191772, -4.239794731140137, 0.3922394812107086, -1.787177562713623, 1.198156476020813, -2.0565834045410156, -2.12522029876709, -1.5337913036346436, -1.882688045501709, -5.4743428230285645, -2.004815101623535, -0.04203328490257263, 3.601466178894043, -0.8795340657234192, -1.1152271032333374, 1.7395151853561401, 1.618207573890686, 4.7985758781433105, 1.7553346157073975, -0.058350857347249985, 4.014986991882324, 4.588445663452148, -1.4261351823806763, 0.47441965341567993, -5.041921615600586, -1.0923824310302734, -2.651254653930664, -0.6422173976898193, -1.4449036121368408, -0.4981496036052704, -0.3091035783290863, 1.4804043769836426, 0.6928154230117798, -1.6854130029678345, -3.8674890995025635, 3.7045552730560303, 4.2519073486328125, -1.3633348941802979, 1.6842429637908936, -4.578314781188965, -4.686645030975342, -0.8096535801887512, 4.415045261383057, -3.1396543979644775, -2.502372980117798, 4.822784423828125, -5.6803436279296875, 1.9386667013168335, 1.4634417295455933, -0.6188123226165771, 0.5658749341964722, 7.865224838256836, -3.3805296421051025, -0.9418205618858337, -0.7143702507019043, -2.0690433979034424, -4.025648593902588, 4.779901027679443, 0.660285234451294, -5.2739763259887695, -2.108412742614746, 1.2162021398544312, 1.2102603912353516, -4.320981502532959, -0.2742806077003479, 0.7826452255249023, 1.177715539932251, 1.1837987899780273, -5.469042778015137, 5.540948390960693, 0.6334007978439331, 5.417969226837158, 8.5892972946167, -0.48167210817337036, -3.7012879848480225, 2.5923914909362793, -0.5425006747245789, -1.35422945022583, -3.0998592376708984, 5.063353538513184, -1.3280234336853027, -4.19875431060791, -1.3427141904830933, -3.4717764854431152, 0.4737125039100647, -1.2580937147140503, -3.1722726821899414, 1.8052984476089478, -1.9303017854690552, 2.171283721923828, 3.7829504013061523, -1.0858583450317383, -1.9395431280136108, 0.597888171672821, 2.6743485927581787, -0.008379184640944004, 0.41986754536628723, -0.08067238330841064, 0.9336438775062561, 1.8862290382385254, -2.1181888580322266, -1.447601556777954, 0.13076677918434143, -1.9971964359283447, 0.4083763360977173, 1.6084473133087158, -0.06516557186841965, -1.810649037361145, -2.331606864929199, 5.253676414489746, 2.6355175971984863, -3.359764575958252, -2.523080348968506, 5.62153434753418, 4.53389310836792, 0.3845289647579193, 3.4735467433929443, -4.460943698883057, 0.003935865592211485, 2.2224276065826416, -5.579902172088623, -2.348982572555542, 2.348564624786377, 3.3948895931243896, -1.7799224853515625, 0.8694283366203308, 4.802137851715088, -1.1457992792129517, -4.000092029571533, -0.18422281742095947, -1.1794729232788086, 3.9031448364257812, 0.4803466796875, -2.4062838554382324, 7.201633930206299, -1.5003509521484375, 1.417337417602539, 2.0286946296691895, 2.35296630859375, 0.5187476873397827, 1.7600215673446655, -0.20370638370513916, -4.2575459480285645, -0.6594318747520447, -0.5952361822128296, -0.5331607460975647, 0.40827518701553345, -1.8741086721420288, 3.9697725772857666, 4.7001447677612305, -6.6010847091674805, 0.027978068217635155, -2.1317834854125977, -7.818731784820557, 1.23040771484375, -1.8563988208770752, -1.1272038221359253, -1.9547338485717773, 3.261016368865967, 3.7178943157196045, 0.8303861618041992, 2.079326629638672, -0.3372849225997925, 0.4204777181148529, -2.225287914276123, 0.10362840443849564, -2.1410114765167236, 1.1190787553787231, 2.4819936752319336, 0.933128297328949, 3.552387237548828, 8.070735931396484, -6.31968355178833, 0.04105205833911896, -1.7060415744781494, -1.370086431503296, -4.074525356292725, 0.15760572254657745, -0.7748181223869324, -1.8983120918273926, -0.7629678845405579, -1.7782312631607056, -7.028765678405762, 3.232226848602295, -0.9104179739952087, -2.848912000656128, 0.9192362427711487, -1.9873151779174805, -6.5219926834106445, 2.6607069969177246, -5.03012752532959, -0.6552212834358215, 0.3604687452316284, 4.049005031585693, 3.5701820850372314, 0.4972442090511322, -2.290174722671509, -1.2476915121078491, 3.2631051540374756, 5.359347820281982, 1.6313658952713013, -1.9345335960388184, 3.088116407394409, -1.1257202625274658, -1.2235480546951294, -0.8369059562683105, 1.7704468965530396, 0.8258246183395386, 2.9784042835235596, 3.8891799449920654, 1.5397367477416992, -6.996790409088135, -5.089792728424072, 0.4757649600505829, -0.19412249326705933, 6.477588176727295, -0.09064218401908875, -0.4578298032283783, 2.8316643238067627, 3.193300247192383, -0.7062304615974426, 1.419606328010559, -3.4006400108337402, -2.8294553756713867, 3.6992194652557373, -4.221406936645508, 2.7669379711151123, 2.033665180206299, 2.309265613555908, 2.738494634628296, 1.691589117050171, -6.424376487731934, -3.01313853263855, -1.803934931755066, 2.7444615364074707, -3.383415937423706, -2.11468505859375, -1.0884361267089844, 3.9492952823638916, -4.323158264160156, -4.061788558959961, 1.881483554840088, -6.376583576202393, 0.9959337711334229, -1.3621227741241455, -4.730854034423828, -1.2608141899108887, -3.006410837173462, 0.5033542513847351, 1.0914661884307861, 4.466884613037109, -0.03835570812225342, 0.3016605079174042, -1.7319324016571045, -0.07090535759925842, 3.1488072872161865, -5.77714729309082, 1.9773865938186646, -1.605530858039856, 0.22167068719863892, 0.9751344323158264, 2.5517826080322266, 0.7152097225189209, -1.1775305271148682, 0.9350830316543579, 0.035619430243968964, -0.6134995222091675, 0.6154430508613586, 2.620741367340088, 0.4149961471557617, 0.46640777587890625, -3.8625311851501465, -0.381033718585968, -5.574052333831787, 2.160032033920288, -2.817761182785034, 3.8861446380615234, 8.980311393737793, -1.1921976804733276, 3.9888904094696045, 1.21079683303833, -1.670961856842041, -2.5239338874816895, 1.6858251094818115, 4.184465408325195, 1.061116337776184, -4.535694122314453, -3.1267786026000977, -0.6839228868484497, 4.071498870849609, 0.8502459526062012, 0.9817858934402466, 3.6315112113952637, -0.6137863397598267, -0.2673480212688446, -0.5455983281135559, -5.333881855010986, 1.4000364542007446, 1.2325164079666138, 2.9263272285461426, -3.566709518432617, -2.447589635848999, 2.803408622741699, 0.9691848754882812, 4.125077247619629, -2.589096784591675, 0.6024090051651001, -2.0279922485351562, 0.9292600154876709, -1.455134391784668, 1.6079013347625732, 3.244091272354126, 2.0258114337921143, -5.424819469451904, 2.8212108612060547, 1.3426264524459839, -3.1290621757507324, -2.6572983264923096, 0.02492017298936844, 1.720588207244873, -1.3521027565002441, -4.86753511428833, 0.8447372317314148, 0.509425938129425, 7.323398590087891, -3.865678071975708, 7.57125186920166, 5.23410701751709, 0.9093684554100037, 4.97695255279541, 0.28968116641044617, 2.93351149559021, -2.2012975215911865, 0.2390938401222229, -3.043785810470581, -4.726412773132324, -0.8149816393852234, -1.9766815900802612, 1.7184369564056396, 3.8124117851257324, -0.5074589252471924, 0.17679673433303833, 0.5637582540512085, -5.033904552459717, 0.8072431683540344, 0.03405154123902321, -0.1389688104391098, -3.3011248111724854, -5.9456400871276855, -0.7507809996604919, 0.9484702348709106, 4.022738933563232, 3.3935279846191406, 4.365952968597412, 1.1098533868789673, 3.4685025215148926, -0.6180327534675598, 1.8853001594543457, 2.7954437732696533, 8.067974090576172, 0.7278911471366882, -3.817708969116211, 4.1331987380981445, -2.373897075653076, -0.18203158676624298, -0.8826441764831543, -2.7232697010040283, 4.004317760467529, 1.7881832122802734, -1.836076259613037, 0.20801609754562378, -1.2997068166732788, 1.8455867767333984, -1.1356712579727173, -2.298490285873413, -1.9329224824905396, -1.310560941696167, 0.8681015968322754, -2.7566254138946533, -0.2681090533733368, 1.3307430744171143, 7.0118255615234375, 2.5687363147735596, -8.951294898986816, -1.5332280397415161, -1.190775752067566, 0.9611560106277466, 4.708157539367676, 2.2262258529663086, -6.9087300300598145, 0.9209322929382324, 0.32418954372406006, 1.238945484161377, 4.076606750488281, -3.272554636001587, 3.3092105388641357, -4.961591720581055, 0.15246731042861938, 1.1417917013168335, 6.8535966873168945, -2.1069929599761963, -5.08180570602417, 2.0613763332366943, -1.9836927652359009, 4.691476821899414, -2.1241700649261475, -2.6318111419677734, 3.733494758605957, 0.2562926411628723, 3.002351760864258, -4.290645122528076, 4.953266143798828, -1.7667158842086792, 2.4583258628845215, -3.0148189067840576, -4.652067184448242, -0.770470917224884, 0.5233867764472961, 4.729944229125977, 1.0501434803009033, 2.1465470790863037, 1.9767522811889648, 9.330463409423828, -2.454429864883423, 3.7654714584350586, -0.4477672278881073, 0.782339870929718, -1.6402572393417358, 3.487222671508789, -3.7645843029022217, -3.96683406829834, -0.6234816908836365, -3.4741203784942627, 2.1098334789276123, 2.76568603515625, -2.258988857269287, 2.3515079021453857, -1.9861335754394531, -0.13269521296024323, 0.020284373313188553, 5.629360198974609, -0.0012829889310523868, -2.73588490486145, -2.752641201019287, 3.5999908447265625, -5.007038116455078, -1.4107310771942139, -1.140639305114746, -7.351539134979248, -1.4077805280685425, 3.0064704418182373, 0.6909671425819397, -2.7022948265075684, 2.070801258087158, 1.5496327877044678, 6.073498725891113, -2.227013111114502, -1.1661604642868042, 0.12324231117963791, -0.4160616993904114, 0.2591296434402466, -0.3223157823085785, -2.084655284881592, -4.813233375549316, -1.682859182357788, -4.17770528793335, -3.1560471057891846, -1.6445896625518799, 1.5083242654800415, 2.5432536602020264, -3.1333107948303223, 0.8921796083450317, 2.375849962234497, -0.17305248975753784, -4.965666770935059, -6.514631748199463, 2.330451011657715, 2.69248366355896, 0.7719883918762207, -3.222933530807495, 4.294182300567627, 3.5750129222869873, -0.20661920309066772, -0.7400868535041809, -5.945897579193115, 2.1799721717834473, -4.917086601257324, 1.9654096364974976, 2.1107797622680664, 0.20374616980552673, 4.6610846519470215, -1.0448169708251953, 1.691973328590393, 2.313974380493164, 0.6697219610214233, -1.3321059942245483, 3.615161418914795, 1.1926555633544922, -0.6533565521240234, -0.573095977306366, 0.3156185746192932, 6.767882823944092, 0.2438635677099228, -0.7923216819763184, -3.2691450119018555, -3.734018325805664, -5.767999649047852, 0.6279538869857788, 0.009149827063083649, -3.1626598834991455, -4.53884744644165, -1.058664083480835, -1.6536753177642822, -4.090410232543945, 0.8221055865287781, 0.34395334124565125, -0.2369898110628128, 2.2291276454925537, -2.044843912124634, 3.9922924041748047, -3.1323435306549072, 1.4544219970703125, 1.8164006471633911, -0.8715693950653076, 0.9630312919616699, -3.9445362091064453, -4.602783679962158, -0.6973874568939209, -2.2841579914093018, 1.6809358596801758, -0.0038648948539048433, 1.7565089464187622, 0.8622984886169434, 0.5169596672058105, 0.18448492884635925, -1.2136603593826294, -1.4896608591079712, 6.009799003601074, -5.379633903503418, -0.6570227146148682, 2.914336919784546, 1.1602429151535034, 4.128535747528076, 1.5490145683288574, -3.889423131942749, -1.0898046493530273, -4.316221237182617, 2.8523521423339844, 4.225481033325195, -1.431028962135315, 1.5199921131134033, -1.3444688320159912, -4.4307146072387695, -1.4576306343078613, -4.867763042449951, -1.8277268409729004, 0.5107952356338501, 2.0538742542266846, 3.760006904602051, -1.1281803846359253, 0.5768398642539978, 1.1318261623382568, 1.5337631702423096, -8.688819885253906, -2.4721481800079346, 2.249119520187378, -3.5008625984191895, 0.5984172821044922, 2.355365514755249, -5.838468551635742, -0.7227457165718079, 0.5586864352226257, -0.12244394421577454, -4.747946262359619, 0.880494236946106, -1.845676302909851, 2.302797794342041, 1.12283456325531, -1.8215560913085938, 0.3522127866744995, -2.251119375228882, -2.284525156021118, -4.927976608276367, -0.908755362033844, 2.2154221534729004, 0.1356106698513031, -0.2823610007762909, -2.117034673690796, -1.811172366142273, 1.9878581762313843, -4.732601642608643, 5.933904647827148, -0.4516506791114807, 6.253084182739258, 3.2545292377471924, -1.5295658111572266, 1.9222707748413086, 0.4826984107494354, 1.0937033891677856, 2.082771062850952, -1.9473210573196411, -2.1662492752075195, -1.5718169212341309, -4.422549724578857, -1.1135215759277344, 0.3578285276889801, 0.4156646430492401, 0.23959879577159882, -1.5310081243515015, 2.013676166534424, -6.510387420654297, -1.5536649227142334, -1.355294942855835, 2.313720226287842, 1.611450433731079, 2.013624906539917, -0.352443665266037, 2.444779396057129, -3.42069149017334, -0.20425640046596527, -6.763315677642822, 4.708151340484619, 4.338641166687012, -2.2365634441375732, -2.990231513977051, -6.7423319816589355, -5.045754432678223, -0.08611129969358444, -0.9242492914199829, -4.879084587097168, -1.9514832496643066, -2.446476697921753, 0.9579230546951294, 1.6809793710708618, -1.1052252054214478, -0.09955909103155136, -0.45940959453582764, -6.710148334503174, 0.09708040207624435, -0.6338722705841064, 1.8122367858886719, -5.946167945861816, 1.8574062585830688, -2.35125470161438, 0.3674313724040985, 3.2019548416137695, -4.85869836807251, -4.364987850189209, 2.5555472373962402, -3.850024938583374, -3.3046722412109375, 1.3817511796951294, -1.0345492362976074, 1.9117155075073242, 0.47801506519317627, -0.49807366728782654, 2.5355610847473145, 6.148282527923584, -0.7155073881149292, 4.0244855880737305, 2.986672878265381, -0.08321155607700348, 0.9586356282234192, -3.225477933883667, -2.2617452144622803, 0.4212508797645569, -0.5522211194038391, 3.494013786315918, -4.309418678283691, -1.2890970706939697, -2.322092056274414, -0.864056408405304, 3.086031198501587, -4.02656888961792, 2.1460530757904053, -2.1743838787078857, -1.3755056858062744, 2.5951709747314453, -1.2501580715179443, 1.6041874885559082, -0.05442555993795395, -0.9147547483444214, -1.475515604019165, -5.588152885437012, -2.6308460235595703, 0.1123497262597084, 0.22157029807567596, -4.8585405349731445, 2.6284828186035156, 1.3431475162506104, -4.282229900360107, -3.5033016204833984, 1.1521104574203491, -5.649173259735107, -2.595165967941284, -0.5235995054244995, -0.7411395907402039, -0.5282666683197021, 0.8193661570549011, -2.747408151626587, -1.8103197813034058, 5.817656517028809, -4.552791595458984, -1.6896177530288696, -0.005092339124530554, -2.2769763469696045, 3.3351123332977295, -0.11165297031402588, 1.9479540586471558, -8.17577075958252, 3.2435455322265625, 3.12860369682312, -2.5881800651550293, 3.606518268585205, 0.48511454463005066, 2.476837396621704, 5.1360249519348145, 0.0007486953982152045, -3.339278221130371, -0.32835835218429565, -4.122575283050537, -2.696743965148926, -2.5796093940734863, -2.4182162284851074, 3.222026824951172, 2.2350332736968994, -2.9646687507629395, 3.12884521484375, -5.3066511154174805, -2.231577157974243, -0.673572838306427, 3.3183114528656006, 1.9376646280288696, 1.1247828006744385, 3.6760408878326416, 1.9201737642288208, 3.8754820823669434, -2.58388090133667, -5.293467998504639, -1.4688876867294312, 2.2813055515289307, 0.5718285441398621, 2.4635488986968994, 1.928959608078003, 1.3462351560592651, 2.2567379474639893, -2.348613739013672, 3.083981990814209, -3.2177045345306396, -0.539617657661438, -4.444727420806885, 1.3231254816055298, -4.855103492736816, 4.026778221130371, -4.231392860412598, 0.5849085450172424, -0.6133036017417908, -2.8266024589538574, 3.5593135356903076, -1.4792559146881104, 4.211345195770264, -3.2728209495544434, -2.600816249847412, -6.1969170570373535, 1.4701470136642456, 2.563483238220215, 0.4853585958480835, -0.6746441125869751, -2.9545063972473145, 6.871382713317871, 2.044422149658203, -5.725368022918701, -1.1173595190048218, -8.021689414978027, 4.558790683746338, 5.756738185882568, 4.210336208343506, -0.34221863746643066, -1.9590202569961548, -4.396110534667969, 1.4211173057556152, -2.4178295135498047, -1.3065783977508545, -2.097975730895996, -1.1029388904571533, 2.524526596069336, 4.9050750732421875, 0.28315505385398865, -0.1104603260755539, -3.515273332595825, -3.863410472869873, -3.3639092445373535, 2.4970862865448, 0.5821423530578613, 2.752520799636841, 1.1166653633117676, -1.3668996095657349, -3.896711826324463, -1.2632769346237183, -4.45427131652832, 5.874757289886475, 0.6611791253089905, -1.2554969787597656, 5.859282493591309, -1.303892970085144, 1.5320868492126465, -0.6386890411376953, 0.9338172674179077, -1.2538808584213257, 1.3453274965286255, 4.161559104919434, -3.9507462978363037, -6.848471641540527, 3.9414446353912354, 0.3680947422981262, 2.5016419887542725, -1.0123412609100342, 4.531787872314453, 5.372607231140137, -3.6752755641937256, 1.895810842514038, -1.986317753791809, -5.019286155700684, -1.062190055847168, 1.567510962486267, -4.303840160369873, -4.057148456573486, 3.56514310836792, -0.07900859415531158, 1.0365321636199951, -3.6505818367004395, -2.1773459911346436, 0.16426502168178558, 0.3559200167655945, 0.2187889963388443, 4.590958595275879, 0.26248699426651, -4.893409252166748, -2.911264181137085, -0.4461742341518402, 3.5920498371124268, -2.372284412384033, -1.484410285949707, -0.3876616656780243, -2.4590797424316406, -2.823589563369751, -1.0476679801940918, 2.671563148498535, 1.118278980255127, -1.2254786491394043, 0.8618946671485901, -4.254428863525391, 2.6590371131896973, 1.8749722242355347, 0.5320236086845398, -5.321187973022461, 5.789933681488037, -9.248620986938477, -3.4075162410736084, -3.170224666595459, 1.727150559425354, -4.926877975463867, 0.3320464491844177, -0.845518171787262, -1.2656234502792358, 2.212765693664551, 3.938046932220459, 0.8240513205528259, -1.213135004043579, -7.292194843292236, 4.084295749664307, -1.2880216836929321, -5.9945597648620605, 0.11203508079051971, 4.026224136352539, -0.7961060404777527, -3.91192889213562, 6.483000755310059, 1.5070412158966064, 0.6966337561607361, 2.7392282485961914, -6.928221225738525, -0.5498554706573486, -0.8346441984176636, 6.550899028778076, -2.0382606983184814, -2.0004615783691406, -4.541882514953613, -2.8130509853363037, -0.9375729560852051, 0.4872084856033325, -0.9109217524528503, 5.785756587982178, 1.1920346021652222, 1.638197422027588, 3.8567042350769043, 1.6272648572921753, -1.882899522781372, -4.993134498596191, 0.7951596975326538, -2.8159432411193848, 3.7130067348480225, -2.6249876022338867, -7.064456939697266, -0.16393040120601654, 2.9719550609588623, -0.919975757598877, 0.26523756980895996, 0.17963890731334686, 4.9501166343688965, 0.6264783143997192, -0.4171167016029358, -2.1680238246917725, -5.292294979095459, 3.989919424057007, 3.199913740158081, -3.2001495361328125, 1.5555387735366821, 0.6954183578491211, -1.460094928741455, -1.2547062635421753, 1.9933834075927734, -3.9736595153808594, -1.8899754285812378, -0.5391288995742798, -0.32637128233909607, -0.94557124376297, 0.3593808710575104, 1.3123924732208252, -3.02605938911438, -0.04554444178938866, -1.9116474390029907, 4.288188934326172, -4.586360931396484, -1.0186214447021484, 1.826104998588562, -1.1694285869598389, -1.3147443532943726, -1.9387176036834717, 0.6575703620910645, -3.83713960647583, -0.5383716225624084, -0.5958214998245239, -4.739519119262695, 0.554170548915863, -1.6915932893753052, 0.6718664765357971, 2.356062889099121, 0.7877699732780457, -3.8478546142578125, 0.7317370772361755, -5.110795021057129, -2.5689785480499268, 4.847077369689941, -0.2129589021205902, 4.514883518218994, -0.7701534628868103, -0.18607091903686523, -3.6049513816833496, 2.659607410430908, -0.8068442344665527, 1.7045866250991821, 1.1345473527908325, 1.442082166671753, 0.8171484470367432, -0.2839967906475067, -0.1634797304868698, -2.258296251296997, -6.608983039855957, 1.345624566078186, 6.194052219390869, 3.1598124504089355, 6.985862731933594, 1.3731050491333008, -3.0122616291046143, -3.0772016048431396, 2.193504571914673, 0.9747993350028992, -0.5579651594161987, 2.204146385192871, -2.7019224166870117, 1.5508713722229004, -0.8867726922035217, 0.4506970942020416, 3.582876682281494, -0.4387204945087433, 4.165647029876709, 4.0260820388793945, -2.308725357055664, 7.0589447021484375, -3.7580296993255615, -0.29408031702041626, 2.2507314682006836, -6.912054061889648, 3.1562106609344482, 3.4792368412017822, -0.3172978460788727, 0.8541628122329712, 0.6834432482719421, -0.6043773889541626, 3.2884011268615723, -1.100061297416687, -0.1008848175406456, 1.144330382347107, 2.6178126335144043, 3.3857176303863525, -4.46151876449585, -6.146316051483154, -0.38616591691970825, -0.13330453634262085, -3.260490655899048, 0.9588573575019836, 1.9434815645217896, 1.1911982297897339, 5.04642915725708, -3.123173952102661, 0.09685375541448593, 5.894752502441406, 2.0915822982788086, -3.3749420642852783, 0.6384881734848022, 3.611199140548706, 6.35964298248291, 2.037673234939575, 3.5723934173583984, -3.3614494800567627, -0.7396745085716248, 3.6507444381713867, -2.907106876373291, -0.17623399198055267, -5.674292087554932, 4.512014865875244, -0.3925170600414276, 1.7288700342178345, -4.570682525634766, -3.993478775024414, 5.52793025970459, -1.0409308671951294, -1.6549371480941772, 1.5575406551361084, 1.687583327293396, -5.9709153175354, -2.9692232608795166, 0.15438060462474823, 0.015996575355529785, 1.0695068836212158, -1.306943655014038, -0.852937638759613, 0.6543033719062805, -2.097337484359741, -2.5417239665985107, -1.670576810836792, 0.016559654846787453, -0.4592317044734955, -1.460060477256775, -0.7697338461875916, -0.1580561101436615, 0.9651627540588379, 1.091186761856079, -3.92476749420166, 2.945519208908081, -4.8788228034973145, 1.6434825658798218, -2.1644270420074463, 1.7402093410491943, 3.1959455013275146, -5.735718727111816, 1.7992465496063232, 0.8004112243652344, -1.439430832862854, 1.132365345954895, -1.5360509157180786, 0.32349443435668945, -3.222278594970703, 2.3379387855529785, 1.7462095022201538, 2.2094008922576904, -2.024371862411499, 5.526397705078125, 5.941000938415527, -0.30739256739616394, -1.3050862550735474, -1.2784552574157715, 2.0354526042938232, 3.0680856704711914, -2.303621768951416, -6.179781913757324, 1.3303534984588623, -0.1695684939622879, 0.47269704937934875, 1.5167180299758911, -3.4121763706207275, 0.14365148544311523, -0.9554252624511719, 1.0809550285339355, 3.6306517124176025, -0.05983776971697807, -2.66078782081604, 3.696204900741577, 2.0684659481048584, -0.6960211992263794, -5.5386223793029785, -2.7398383617401123, 0.3381274938583374, 1.5557141304016113, -1.7011663913726807, 1.2087066173553467, -2.0314784049987793, -1.3953776359558105, 1.4885417222976685, -0.9416601657867432, 0.2960825562477112, 2.5149104595184326, 3.280599355697632, -1.9140921831130981, -0.9005360007286072, -4.139903545379639, -4.0819196701049805, 0.7537158727645874, -1.7205936908721924, -5.74554443359375, 1.442846655845642, -2.6883738040924072, -1.101446270942688, 1.0980216264724731, 1.037773609161377, -0.33431464433670044, -6.166360855102539, 1.5410665273666382, -1.9554847478866577, -4.431580543518066, -6.813728332519531, 3.8126277923583984, 0.26401978731155396, 0.17447529733181, 1.6726100444793701, -0.3969186842441559, -4.608438014984131, 1.886036992073059, 2.8550524711608887, 1.3988237380981445, 4.471090793609619, 5.579573631286621, -2.551588773727417, -1.4862154722213745, 2.517190456390381, 0.3828144967556, 0.4878629148006439, 2.664762020111084, 2.0817983150482178, -1.4709182977676392, 0.23345202207565308, -2.2797412872314453, -0.7258853912353516, -3.4782233238220215, -6.039824962615967, -1.6288883686065674, -1.9106693267822266, 2.5530993938446045, 4.597336769104004, -0.05083709955215454, 1.2927454710006714, -2.030932664871216, -2.9015183448791504, -0.972683846950531, -0.9722777605056763, 0.7758138179779053, 0.229583278298378, 4.870057582855225, -0.7540674209594727, -5.055380344390869, 0.26193809509277344, -2.077833414077759, 0.5166639685630798, 3.2807888984680176, -2.020467758178711, -0.3191903829574585, 1.461127519607544, -2.926332950592041, 0.5606049299240112, 1.5505985021591187, 2.354477882385254, 3.2102487087249756, 1.5846929550170898, 1.097612977027893, -4.246910572052002, -8.108402252197266, -2.9638476371765137, 4.197357654571533, 0.2840549349784851, -6.822349548339844, -1.3267035484313965, 1.0226573944091797, -5.608784198760986, 1.6610426902770996, -3.4927611351013184, 3.0694024562835693, 0.03997410461306572, 9.23375415802002, -4.0371551513671875, 0.804630696773529, -4.462316989898682, 3.406693696975708, 0.3399006128311157, 2.729769706726074, 0.2787514925003052, -3.5378434658050537, -1.9781132936477661, -3.7309160232543945, -3.1569912433624268, 0.4123169481754303, 3.1865856647491455, 2.5517022609710693, 1.5207005739212036, -1.7085628509521484, 1.5097551345825195, -2.5787618160247803, 0.5994012951850891, 2.4912023544311523, -0.5760814547538757, -1.4228622913360596, -1.3115472793579102, 1.819868803024292, -0.9442582130432129, -4.271467685699463, -1.2562671899795532, -3.2011823654174805, 4.471784591674805, 0.020191388204693794, -4.4208197593688965, 0.7778427600860596, 2.1352829933166504, 1.1181403398513794, -1.0934957265853882, 0.42951491475105286, 0.8181782364845276, 2.98413348197937, 1.0724700689315796, 6.76505184173584, -0.6570438146591187, 4.5560126304626465, 0.04013040289282799, -5.10961389541626, -3.5205814838409424, -5.448432445526123, -5.149123191833496, 3.580350399017334, -1.9793490171432495, 1.7886934280395508, -2.6744656562805176, -0.0482318177819252, -3.1584439277648926, 2.5754871368408203, -0.7695016860961914, -1.3612723350524902, -0.4982813000679016, 6.489622116088867, 1.3274468183517456, 2.4863462448120117, -3.7207911014556885, -1.9505454301834106, 0.871136486530304, -5.559148788452148, 3.979936361312866, 0.7982226014137268, -0.06353344768285751, -0.6031047105789185, -0.7118592262268066, -1.5198713541030884, 4.146790504455566, 2.0416204929351807, 3.701969623565674, -0.9186568260192871, -2.6391944885253906, 4.042929172515869, -3.6471190452575684, -1.8324406147003174, 2.1123273372650146, 3.2267746925354004, -3.9409971237182617, -0.8316042423248291, 0.06841126084327698, -0.4121237099170685, -2.580900192260742, 2.003387689590454, -1.5828237533569336, -0.7327831983566284, -6.733522891998291, -0.3433566987514496, -5.8115739822387695, 2.0929126739501953, 1.4554650783538818, -1.5348984003067017, 0.46656376123428345, 0.30782854557037354, -0.45697206258773804, 1.063382625579834, -0.3858766555786133, -1.4392585754394531, -2.387169599533081, 4.981762409210205, 0.5611379742622375, -0.07796984165906906, 2.1375656127929688, -1.1772745847702026, -1.1583746671676636, 1.9681028127670288, 3.049591302871704, 1.9844696521759033, -4.437720775604248, 0.15392298996448517, -0.08748696744441986, 1.335434913635254, -2.656087875366211, -3.1459670066833496, -0.367262065410614, -2.6678109169006348, -3.193593978881836, -0.22786425054073334, 2.7347211837768555, 0.38162025809288025, 3.299347162246704, -2.9870333671569824, 2.0800225734710693, -0.058678269386291504, -2.636697769165039, -1.271033525466919, -2.578533411026001, -0.6695157885551453, -2.2480571269989014, 3.711507558822632, -1.0938329696655273, -4.250452041625977, -0.6090938448905945, 1.1164569854736328, -0.9718532562255859, 2.5239834785461426, -2.730177879333496, -2.5644454956054688, 4.787517547607422, -0.016376705840229988, 1.5766067504882812, 1.4393680095672607, 1.6806128025054932, 0.09935053437948227, -0.5923906564712524, 3.1809287071228027, -1.4799911975860596, 0.3157503604888916, 2.002336263656616, -1.195220947265625, -4.560230255126953, 1.334494948387146, 3.1689963340759277, 0.7307469248771667, -4.973682880401611, 1.8716516494750977, 5.790348052978516, -4.582464694976807, -8.0184907913208, -0.7396422624588013, 0.21030737459659576, 2.2906906604766846, 2.400629758834839, 0.26498904824256897, 1.586356282234192, 0.9811355471611023, 0.35898157954216003, -0.1726677119731903, -0.4549647569656372, 2.064643383026123, 1.1890285015106201, -1.2587300539016724, 3.212193727493286, -2.137303590774536, -0.680638313293457, 0.3943638503551483, -5.584815502166748, -1.1093482971191406, 2.508361339569092, 2.387915849685669, -3.5022871494293213, -2.43257737159729, -3.1557459831237793, -0.6914388537406921, -0.188108429312706, -4.671669006347656, 0.931298553943634, 0.20963068306446075, -0.21425850689411163, -1.969542384147644, -0.5381805896759033, -5.209213733673096, 0.6778640151023865, 0.6187108159065247, -0.30910924077033997, -3.4535863399505615, 3.7279720306396484, 3.308281421661377, -5.4806671142578125, 1.2116212844848633, 0.4472961127758026, -7.510931015014648, 1.515523910522461, 1.9041723012924194, 2.109529972076416, 0.4734262526035309, -1.9038857221603394, -1.4945377111434937, 1.5003368854522705, 3.4290554523468018, -1.4246143102645874, -0.8082797527313232, 2.043978214263916, -6.284438133239746, 1.3146501779556274, -0.6658180952072144, -6.39948844909668, 2.2614917755126953, -2.6609513759613037, 2.32448673248291, 4.033188819885254, -3.62288498878479, -3.485584259033203, -0.018541503697633743, -5.094387054443359, 2.148472785949707, 0.5656221508979797, 2.8876869678497314, -5.325318336486816, 0.6154320240020752, 1.3819772005081177, -5.220345973968506, 2.6395368576049805, -3.9898977279663086, 0.5672391653060913, -3.804600238800049, 2.3443267345428467, 5.02732515335083, 0.22979043424129486, 0.15014833211898804, 3.862990379333496, -1.2376892566680908, 0.22181405127048492, 2.4773430824279785, 1.326902985572815, 4.700841903686523, 3.6146345138549805, 5.750772953033447, 0.09198001027107239, 3.436727285385132, -2.2153942584991455, 2.8399057388305664, 5.340497016906738, 7.097175121307373, -3.4311881065368652, -0.26222991943359375, 2.335439443588257, -2.31540846824646, 1.8672128915786743, 1.8462669849395752, -1.1540995836257935, -1.300574541091919, -2.428633213043213, 1.1694798469543457, 3.989241600036621, 1.1290980577468872, 8.408056259155273, -4.062076091766357, -1.4200154542922974, 2.252857208251953, -3.0688459873199463, 1.3022924661636353, 2.6714365482330322, 0.8062818050384521, -0.3777843713760376, 3.042206048965454, 1.5078694820404053, 2.409489154815674, 1.8863645792007446, -1.690187692642212, -2.9517242908477783, -0.8339726328849792, 0.8033933043479919, -0.28079068660736084, 3.0579721927642822, 1.0247992277145386, 3.1238481998443604, 0.6158146262168884, -1.8476128578186035, 2.7462573051452637, 1.728139877319336, -2.2171790599823, -0.9512470364570618, 8.469788551330566, -9.714373588562012, 3.56467342376709, 0.4034913182258606, 0.023201873525977135, -0.6595410704612732, -1.297766089439392, 7.182684421539307, -1.1100815534591675, -0.7081771492958069, 2.550074338912964, 1.3092149496078491, -0.35810235142707825, -0.6787461042404175, -3.7758073806762695, 3.353600025177002, -2.9788308143615723, -7.0321125984191895, 3.2398416996002197, 2.3543248176574707, 1.1365935802459717, -1.420562505722046, 3.64493465423584, -3.097224712371826, -6.386616230010986, 0.5811916589736938, 2.4887733459472656, 1.9376673698425293, 6.259979724884033, 2.563112258911133, -0.5892664790153503, -0.4880976676940918, 2.732470750808716, -0.10266592353582382, -0.3199404776096344, -5.168890476226807, -0.8486177325248718, -3.7863411903381348, 0.7690026164054871, 0.4691547453403473, 2.1893997192382812, 5.395162582397461, -0.46824127435684204, 4.367421627044678, -3.304630756378174, 0.5335687398910522, 2.910255193710327, 5.897923946380615, -3.224879026412964, -0.7019791007041931, 0.48520374298095703, -0.23828013241291046, -2.7880165576934814, -4.8191399574279785, -3.6022775173187256, -0.1827142834663391, -3.9515538215637207, 5.363218307495117, 3.7319345474243164, -5.639679431915283, -1.4176908731460571, 4.0197367668151855, 4.3914947509765625, 3.3972747325897217, -4.224917888641357, -1.0435086488723755, 3.8803458213806152, 0.3624097406864166, -1.3117865324020386, 1.5619802474975586, 3.000622510910034, 0.4679037630558014, 0.7865843176841736, 3.2659575939178467, -1.6351807117462158, -2.6851396560668945, 2.3409056663513184, -3.540123462677002, 1.5353069305419922, -2.1390936374664307, 2.5611398220062256, -3.5175704956054688, 3.4449687004089355, 0.8963664174079895, -0.8685204982757568, -2.5400867462158203, -0.42435771226882935, -2.731302499771118, -3.554736375808716, 3.257411241531372, 1.8219001293182373, 3.7253801822662354, -4.020915508270264, -5.805854320526123, -4.719717502593994, 1.979144811630249, 2.444535970687866, -5.533225059509277, 3.090646743774414, -1.2878731489181519, -3.5690786838531494, -0.2202903777360916, 1.7198147773742676, -0.6311560869216919, -0.5164346098899841, -2.5172877311706543, -4.2550482749938965, -1.1238819360733032, -4.874472618103027, 1.803985834121704, -2.561528205871582, 3.9329535961151123, -4.769198417663574, 4.216974258422852, -2.6548919677734375, -2.6812758445739746, 0.8769670724868774, 2.6990630626678467, -0.12914110720157623, -1.081268310546875, 5.984899044036865, 1.5202008485794067, -3.6177594661712646, -0.7976445555686951, 5.0776686668396, 0.98426353931427, 2.473860502243042, 6.1022868156433105, -1.792230486869812, 1.1185939311981201, -2.0103232860565186, -0.023899737745523453, 1.608729362487793, -4.0142083168029785, 0.4314986765384674, 0.420795202255249, 3.160923719406128, -1.7901575565338135, -3.8339226245880127, 1.1782619953155518, -3.3886923789978027, 0.5383697748184204, 1.6952601671218872, -2.649958848953247, -0.3818487823009491, -3.135997772216797, -8.555098533630371, -4.400638103485107, -6.308236122131348, 0.017868107184767723, -3.2512028217315674, -0.47052812576293945, -0.18746823072433472, 3.1549370288848877, -1.0876282453536987, -1.2525819540023804, 1.4747960567474365, 4.812502861022949, -0.27507898211479187, 0.9832757711410522, -0.6212930083274841, -2.149980306625366, -4.682488441467285, -0.2830694615840912, -0.8483763337135315, -0.7885064482688904, 5.926034450531006, -1.0508419275283813, 0.9559968709945679, -1.0857274532318115, 3.5138235092163086, -0.9063029885292053, 5.562604904174805, -0.8219594359397888, 2.305189609527588, 3.034118413925171, -0.7407669425010681, 2.5436673164367676, 0.1654175966978073, -2.3080642223358154, -2.0452048778533936, 0.9762517213821411, 5.346043109893799, -1.5009241104125977, 1.6146751642227173, -2.449082851409912, 0.7472965121269226, -3.947205066680908, -2.3945868015289307, -0.04557173699140549, -2.275705575942993, -0.8381461501121521, 3.0109856128692627, -0.3999033272266388, -2.1819565296173096, -3.734575033187866, -1.622502326965332, 3.4410400390625, -0.2634732127189636, -0.45846596360206604, 2.1507341861724854, 2.4992151260375977, 2.9998998641967773, -1.0963374376296997, -2.7715916633605957, -5.034428596496582, -2.080141544342041, -1.1372299194335938, 2.1714324951171875, 0.9401715397834778, -1.7235207557678223, 2.6345677375793457, -3.8013885021209717, -3.182974338531494, -3.5587966442108154, -3.305835247039795, -0.7909762263298035, -2.5697500705718994, -2.314131259918213, 3.989107370376587, -4.432264804840088, -3.1323230266571045, -0.7488860487937927, 1.2992744445800781, 1.4279907941818237, -0.8296382427215576, -1.1358137130737305, -0.8337256908416748, 2.1991090774536133, -2.800046920776367, -7.093016624450684, 0.5218931436538696, -3.5446784496307373, -0.21155036985874176, 4.1371026039123535, 0.2609124481678009, 5.478753089904785, -4.808029651641846, 0.6838249564170837, 1.1384258270263672, 2.459307909011841, -2.219524383544922, 0.45245683193206787, 0.33790159225463867, 0.24390871822834015, -4.144956588745117, -5.200496673583984, 1.6027711629867554, 1.023987054824829, 0.09734100103378296, 1.901153564453125, 3.2337141036987305, -3.494392156600952, 2.343839406967163, 4.476181507110596, 3.0765557289123535, 1.7885206937789917, 2.137620210647583, -4.300317287445068, -1.4521206617355347, 3.116631507873535, 4.493837833404541, 1.4362971782684326, -1.3058775663375854, -0.34795868396759033, 2.632511615753174, 6.139531135559082, 2.4593122005462646, -1.4700614213943481, 1.9751845598220825, 1.410279631614685, 7.3119306564331055, 1.9047892093658447, 0.22786474227905273, -0.10470838099718094, -3.564549446105957, -3.0354526042938232, -1.3050537109375, -2.3042571544647217, 0.704008936882019, 0.4315218925476074, 2.956993341445923, 4.68428897857666, 0.6007985472679138, -0.44858449697494507, -5.437084197998047, 1.2805832624435425, 2.1224067211151123, -2.171016216278076, -1.0355077981948853, -1.9591525793075562, 3.3167154788970947, 1.9998180866241455, 3.7890384197235107, -3.4803779125213623, 8.11732006072998, -2.4312551021575928, -2.9414172172546387, 0.6969060897827148, -1.5390708446502686, 5.573956489562988, -1.1191924810409546, -1.9413602352142334, 3.0399327278137207, 3.775787591934204, -2.4100167751312256, -1.3311035633087158, -0.9802160859107971, -0.38660696148872375, 5.2277679443359375, 3.865604877471924, 3.289006471633911, 3.910438299179077, 0.1362042874097824, 2.699636936187744, -0.14109019935131073, -2.182054281234741, 5.069293022155762, -5.204802513122559, 0.07391568273305893, -1.6704456806182861, -4.935267448425293, -1.0940449237823486, 0.250129371881485, 1.4236488342285156, -2.4396488666534424, 2.294126510620117, 0.9521722793579102, 2.4596314430236816, -5.755337238311768, -1.338484525680542, -1.8340306282043457, 1.73784339427948, -2.210754156112671, -1.5311464071273804, -3.4286839962005615, -3.5870046615600586, -2.8113999366760254, 2.8809802532196045, -3.2035248279571533, -7.325898170471191, 0.8451271057128906, -1.3904184103012085, 1.6225720643997192, 1.5662931203842163, -0.9803694486618042, -1.3045904636383057, -1.704116940498352, -0.07352297753095627, -2.3811471462249756, 0.5359331965446472, 4.09556770324707, -2.0485877990722656, -1.932539463043213, -1.9571237564086914, 1.7880818843841553, 1.9203741550445557, 4.254781723022461, 2.096254587173462, 0.3450840413570404, 1.330719232559204, -3.9495575428009033, 1.6602003574371338, 0.49500930309295654, 6.2020039558410645, 4.103099346160889, -2.4883804321289062, -1.7133225202560425, 2.61755108833313, -6.208480358123779, -2.764760971069336, 1.0953913927078247, 1.9179162979125977, 0.5440011024475098, 3.4781837463378906, -0.8210094571113586, 0.279572069644928, -0.5854409337043762, -3.2811975479125977, 0.13093146681785583, -1.6571710109710693, 0.23596873879432678, -7.83669900894165, 0.3234787583351135, -1.6284890174865723, -1.8446565866470337, 0.13339616358280182, -1.4142519235610962, 2.775045871734619, 3.7353556156158447, 2.6447834968566895, 2.889543056488037, 1.0900706052780151, 0.0966196283698082, -3.8370449542999268, -1.101759672164917, 3.6288273334503174, 2.2957658767700195, -1.4620870351791382, 0.32928338646888733, 3.171569347381592, -0.4764552116394043, 4.862072467803955, -2.260244131088257, 2.3714210987091064, 2.089614152908325, 0.5268873572349548, 0.9135170578956604, 1.6247166395187378, -6.782039165496826, 1.4210625886917114, 3.439974546432495, 1.2103841304779053, 3.4241483211517334, -3.521594762802124, -0.18612921237945557, 1.0067956447601318, -0.7681881785392761, 3.5752453804016113, 1.4730762243270874, -1.5131516456604004, 2.4589853286743164, 2.46584415435791, -1.2886784076690674, 1.8937383890151978, 1.1483335494995117, 1.0776902437210083, -3.731168508529663, 3.8333165645599365, 1.9537502527236938, -0.06278545409440994, 1.0571917295455933, 1.017501711845398, -0.898022472858429, -1.7361092567443848, 0.30522269010543823, -1.7711496353149414, -1.441360354423523, -3.4754607677459717, -0.8187946677207947, 3.021101236343384, 0.6475421786308289, 0.1490166187286377, -0.9401870369911194, -1.2692646980285645, -4.477633953094482, 4.961424350738525, -2.1730058193206787, -4.489816665649414, 1.2022348642349243, 0.7621328830718994, -1.552255630493164, -4.416609764099121, -4.135004043579102, 4.699923038482666, -0.35210078954696655, -1.9601247310638428, 5.574336528778076, -1.6340001821517944, -2.7726805210113525, 1.6311405897140503, 1.0994435548782349, 4.686730861663818, -4.670881748199463, 0.8637598752975464, -5.086560249328613, 1.3807073831558228, -1.964803695678711, -1.3873780965805054, -2.209589719772339, 2.5143373012542725, 4.866916656494141, -0.26022976636886597, -0.774097204208374, -1.5124642848968506, 1.5646677017211914, -3.0971567630767822, 10.628511428833008, -0.703719973564148, 5.133297920227051, 0.439388245344162, -4.124863624572754, -4.697999000549316, -1.2767969369888306, -2.1678435802459717, -0.7587360739707947, -3.4760406017303467, 7.025782108306885, 6.367125511169434, -5.45986795425415, 1.4164774417877197, 3.566835880279541, -1.044502854347229, -0.7195456027984619, 3.2366156578063965, 1.5800120830535889, -3.6951630115509033, 3.634660005569458, -1.519503116607666, 5.533998489379883, -2.940896511077881, -6.955301761627197, 0.23841504752635956, 3.588010787963867, 1.4851408004760742, 0.39955979585647583, -2.8511900901794434, 2.684530735015869, -5.666642665863037, 2.299943447113037, -3.7404651641845703, -1.9113436937332153, -4.087047576904297, -2.8753607273101807, 4.3632707595825195, -3.6855251789093018, 0.4377081096172333, -2.5631747245788574, 2.9558610916137695, -4.260172367095947, -0.14694085717201233, 2.5734870433807373, 6.678278923034668, -3.750260353088379, -2.8258745670318604, -0.3312947154045105, 4.302021503448486, -3.2786362171173096, 2.3007147312164307, -2.0885157585144043, -3.335273504257202, -5.019941329956055, 0.20171517133712769, 2.774507999420166, -0.5205592513084412, -6.645909309387207, -1.91689133644104, 2.719435691833496, 2.428239345550537, 0.6149396896362305, 4.262387275695801, -4.079615116119385, -3.9743902683258057, 7.024021148681641, 1.8109210729599, -0.844931960105896, -1.7136470079421997, -8.06868839263916, 2.319749593734741, -6.527052879333496, -6.223623275756836, -0.38703206181526184, 4.354065895080566, -1.4416766166687012, 0.9042884707450867, -1.5610946416854858, 0.9735832214355469, -5.314909934997559, 6.180887222290039, -3.1759023666381836, 1.4753011465072632, -6.475954055786133, 5.187461853027344, 1.0974115133285522, 5.891942977905273, -2.232146739959717, 2.9491562843322754, -0.7549459338188171, -0.349565327167511, 3.7605998516082764, -2.9679739475250244, 2.05635404586792, 5.748566627502441, 5.1580424308776855, 3.0656826496124268, 0.8306035995483398, -3.1319446563720703, 4.2231059074401855, -2.2419514656066895, 3.522555112838745, -1.164381504058838, -0.1133880540728569, -0.15278807282447815, 1.7136198282241821, -1.4829075336456299, 0.410104900598526, -1.730929970741272, 1.4816807508468628, 0.7316337823867798, 1.1745951175689697, -2.7189722061157227, 0.8590112924575806, -0.004719818476587534, 1.6483256816864014, 0.021452218294143677, -3.104978084564209, 2.080087184906006, -2.1935675144195557, 0.7267653346061707, -6.471150875091553, 1.3903961181640625, 1.8746618032455444, -2.955583095550537, 1.6337225437164307, -1.2860347032546997, 4.065739631652832, 1.8760504722595215, -3.9578073024749756, -0.7724258303642273, -5.539728164672852, -1.130820393562317, 1.5103737115859985, 2.3801958560943604, -0.4772704243659973, -0.8776454925537109, 0.3463064134120941, -1.9833028316497803, 0.8550115823745728, 3.278099298477173, 0.4600793421268463, 2.4986748695373535, -2.498166799545288, -1.61220383644104, 0.9237591028213501, 6.0632500648498535, 2.9916646480560303, 5.120476722717285, 0.0749824047088623, 3.710113763809204, -5.193798542022705, 1.5323837995529175, -6.408697128295898, 3.7032294273376465, 1.4260883331298828, 3.5185000896453857, 1.2019609212875366, -1.5335419178009033, -2.2750349044799805, 0.5710307359695435, 4.553815841674805, 1.9087750911712646, -1.6429181098937988, -1.7225199937820435, -2.142167329788208, -0.6385698914527893, -0.26887062191963196, -1.457998514175415, 0.034491028636693954, -4.781294822692871, 4.7618279457092285, 0.7654030323028564, 4.656805992126465, 0.9019999504089355, -0.5695955753326416, 0.29093170166015625, 0.7049018144607544, -1.9724757671356201, -1.1127530336380005, 5.843400001525879, 2.8949248790740967, -1.5253839492797852, 1.6381425857543945, -2.4834980964660645, 4.315779685974121, -3.001558303833008, 2.776479721069336, 0.1479741334915161, 0.1750066876411438, -2.389178991317749, -0.711804986000061, -2.239211082458496, -2.542987585067749, 1.4707025289535522, 7.6913042068481445, -0.9652597904205322, 3.102616548538208, 0.033409200608730316, -2.0912864208221436, 4.1357293128967285, 1.9968862533569336, 3.525059461593628, 0.4224717915058136, 0.5157041549682617, -2.4009270668029785, -0.7324150204658508, 5.168243408203125, -6.82020378112793, 1.257101058959961, 4.5316009521484375, 2.50654673576355, 2.7545180320739746, -0.710084080696106, 3.5128698348999023, 3.5969326496124268, -0.014635298401117325, 5.170797824859619, -1.36482572555542, -7.39370059967041, 0.07201346009969711, -0.23720070719718933, -2.466510534286499, -1.2846547365188599, -1.525147795677185, -5.4243364334106445, 3.82598876953125, 0.11866749823093414, 8.217567443847656, 0.05716167390346527, 1.299658179283142, -4.460636615753174, 3.172482490539551, -3.191594123840332, 1.726121425628662, 2.8914618492126465, 2.403947591781616, -2.4180173873901367, -2.327042818069458, 2.3811702728271484, 3.669142961502075, 4.461637020111084, 0.007922380231320858, -0.22599385678768158, 5.504887104034424, -1.6976813077926636, 2.3846380710601807, -0.8552364110946655, -4.210651874542236, -3.2863352298736572, 1.4473228454589844, 0.7457581758499146, 1.5657613277435303, -0.981707751750946, -1.4376732110977173, 0.9325453639030457, -0.4798908233642578, -4.771124362945557, -1.241847276687622, 5.180476188659668, -2.2371156215667725, 1.7223262786865234, -0.1842445433139801, 2.264042854309082, -2.568840503692627, -3.7983596324920654, 1.3963149785995483, -2.069793462753296, -0.06687060743570328, -0.9077144861221313, 2.940314292907715, 0.8090233206748962, 0.45930564403533936, -2.898509979248047, -4.690668106079102, -2.4906256198883057, 2.5406696796417236, 2.111076831817627, 1.7824839353561401, -1.1835179328918457, -3.0601978302001953, 3.4769132137298584, 0.5701153874397278, 1.6265878677368164, -2.8137950897216797, -2.042019844055176, 1.954034447669983, 2.2329750061035156, -2.9848008155822754, 0.8782369494438171, -3.5028226375579834, 3.6716527938842773, 2.0015366077423096, 6.834639072418213, 4.540146350860596, -1.9719127416610718, 0.598987340927124, -1.421029806137085, 0.4302178919315338, 2.353283643722534, -0.013662041164934635, 0.18587787449359894, -4.009505748748779, 1.738848328590393, -0.041118789464235306, 1.8756933212280273, 3.1221225261688232, 0.0986073911190033, 2.096482992172241, -1.8085507154464722, 2.83408784866333, -0.0656789168715477, -1.7732423543930054, -0.6906068921089172, -4.250062465667725, -1.9455387592315674, -1.2867683172225952, 0.07498612999916077, -0.26447170972824097, -1.9319119453430176, -0.5596362352371216, -6.402173042297363, 0.26222819089889526, 1.6557024717330933, 2.3439781665802, 3.205488920211792, -0.6325691938400269, -0.27007991075515747, -1.4507224559783936, -1.466164469718933, 0.34366604685783386, 3.9846420288085938, -5.078802585601807, -1.514234185218811, 0.7897266745567322, -3.0161397457122803, 1.1947004795074463, -4.138542175292969, -1.0846408605575562, -4.981603622436523, 0.7623886466026306, -2.4027061462402344, 1.8641422986984253, -0.9439283609390259, -2.9332895278930664, -1.3291189670562744, 0.2920195460319519, -0.12318065017461777, -0.8549421429634094, -1.1372206211090088, -0.06652219593524933, 2.4151229858398438, -1.8029260635375977, 1.2036960124969482, -4.047155380249023, 2.6450109481811523, 0.9778488874435425, -1.1450427770614624, -1.0787246227264404, -5.143329620361328, 0.17485010623931885, -0.8272972106933594, -4.247277736663818, -1.4113123416900635, 1.614958643913269, 0.4264122247695923, 0.7140676379203796, -1.7428276538848877, 2.4260146617889404, -0.16166818141937256, 2.434464693069458, 6.495809078216553, -6.029266834259033, -1.5735650062561035, -0.057708363980054855, -2.2603518962860107, 2.99478816986084, -1.1050416231155396, 4.902857303619385, 2.189502239227295, 1.201919436454773, -0.11052810400724411, 2.4626388549804688, 1.418102502822876, 4.328526973724365, 1.217021107673645, 3.04036021232605, 1.1934142112731934, 0.9801517724990845, -4.6330108642578125, 0.022435583174228668, 1.163217544555664, -3.457580327987671, -4.113520622253418, -0.9010557532310486, 0.8769121766090393, -5.044719219207764, 0.9311633706092834, 2.8500325679779053, -0.7355446815490723, 0.7175765037536621, 1.3852826356887817, -1.958385944366455, 1.8474658727645874, 1.3427835702896118, -3.6729588508605957, 1.5417401790618896, 2.6034789085388184, -3.196409225463867, 2.583106517791748, -0.7841159701347351, -3.9798803329467773, -3.077796697616577, -2.6252856254577637, 1.6054524183273315, -1.4280660152435303, -1.164752721786499, -3.756276845932007, 0.06502396613359451, -3.6993207931518555, -1.995802640914917, 1.8188477754592896, 1.9744807481765747, -1.5798949003219604, -0.1534978300333023, 1.5795992612838745, -0.032048631459474564, 3.580409526824951, 3.98465633392334, -0.2584863603115082, -3.659092426300049, -0.4542708098888397, 4.010651111602783, 3.356558084487915, 3.5271852016448975, 3.3304061889648438, -1.5208195447921753, 0.8297184705734253, 3.241225481033325, 0.8042009472846985, 2.6349709033966064, -2.9814066886901855, -3.6089236736297607, 6.0170135498046875, 2.3154900074005127, 1.8485376834869385, 4.399291038513184, 6.673903942108154, 8.627710342407227, -7.132455348968506, 1.2010270357131958, 3.1318459510803223, -5.676135540008545, -5.2932658195495605, -0.48184746503829956, 5.697741985321045, 2.8654589653015137, 6.503233432769775, 0.9120089411735535, 3.267172336578369, 4.614763259887695, 1.4297603368759155, -0.0047379410825669765, 0.4186069667339325, -11.016403198242188, 1.0070970058441162, -5.548985481262207, -2.423247814178467, 2.147021770477295, 1.2199071645736694, -5.9790496826171875, 1.5986114740371704, 2.271977424621582, 0.6100900769233704, 2.0753462314605713], "dtype": "float32"}], "e": {"__nd__": [0.40532076358795166, -0.09180106222629547, -0.1336050182580948, -0.0, 5.39912223815918, 3.431696891784668, -0.16070707142353058, 2.298251152038574, 0.6752819418907166, -0.08041965216398239, 2.9282097816467285, -0.1638728827238083, -0.15990790724754333, -0.020386027172207832, 1.2470617294311523, -0.1139829158782959, 1.5523775815963745, -0.062458354979753494, 0.24662576615810394, -0.009474772959947586, 2.51021409034729, 0.40255212783813477, 0.8322259187698364, 1.0969756841659546, -0.003279698546975851, 2.3278310298919678, 6.170108318328857, -4.026420583613799e-07, -6.18880093838925e-08, -4.727853138319915e-06, 2.510195016860962, 0.2511071264743805, 3.2333734035491943, 2.1347930431365967, 0.46503815054893494, 0.6840828061103821, -0.15549057722091675, 2.5938913822174072, -0.0009436466498300433, -0.13036829233169556, 0.558897852897644, 5.404262542724609, -0.02469310536980629, -0.0016413995763286948, -0.07707349210977554, 2.9029669761657715, -0.1695425659418106, 3.972961187362671, -2.3055115594416975e-09, 3.384613037109375, 3.1020121574401855, -1.9519775378284976e-05, 0.31214678287506104, 3.6469333171844482, 0.1593121439218521, 2.9954564571380615, 7.124660968780518, 0.6527474522590637, -0.1684384047985077, -0.02361040748655796, 1.893818974494934, -0.16382750868797302, -0.15869352221488953, -0.11876292526721954, 1.8997312784194946, -0.0018759447848424315, -3.0550479550583987e-06, -0.0, 3.595593214035034, 0.13008256256580353, 4.530354022979736, -0.013146471232175827, -0.028638729825615883, 1.324853777885437, -0.09400636702775955, -0.0002075953671010211, -0.010051063261926174, 5.300337791442871, 0.9099263548851013, 1.11677086353302, -0.1688178926706314, -0.039705682545900345, 2.6654443740844727, -0.11827471852302551, -0.025600675493478775, -0.13830767571926117, -0.008480674587190151, 0.40802428126335144, 3.3867340087890625, -0.01477354671806097, 4.28561544418335, -0.04515651613473892, 0.31171777844429016, -0.014739831909537315, -0.16837701201438904, 0.0791478231549263, -0.12556011974811554, -0.036682162433862686, -0.04256545379757881, -0.016521906480193138, -1.4799315977143124e-06, -0.16976113617420197, 1.0655871629714966, 2.7169220447540283, 1.8906670808792114, 7.372009754180908, 0.7938344478607178, -0.1172824501991272, 5.614903926849365, -0.0022838106378912926, 2.900547981262207, -0.005513468757271767, 0.9091402292251587, -1.9835152964553515e-10, 2.6888797283172607, -0.15073804557323456, -0.0049207983538508415, 5.035257339477539, 2.271519660949707, 0.07622209936380386, -0.028029566630721092, -0.058311738073825836, -0.15317578613758087, 2.142307758331299, 2.374681234359741, -0.04512140527367592, -0.08201485872268677, -0.08833175152540207, -5.576745388680138e-05, -0.06747337430715561, -0.10834215581417084, 2.033440351486206, 3.9610540866851807, -0.018150273710489273, 1.5714915990829468, -0.1218431144952774, 6.2255635261535645, -0.06459197402000427, 1.4067667722702026, -0.0065254648216068745, -0.017843401059508324, 0.44012922048568726, -0.14379224181175232, 0.9064329266548157, -1.721605713100871e-06, 1.9393980503082275, -0.008926119655370712, 5.167270183563232, -0.16748207807540894, 3.2904443740844727, -2.8187439966131933e-05, 1.1591160297393799, -0.00981074757874012, -0.10547171533107758, 0.03906477242708206, -0.16255249083042145, 0.6230672597885132, 2.215341091156006, 1.9092642068862915, -0.1040579304099083, -4.5329688873607665e-05, -0.018643144518136978, -0.16359172761440277, -2.509173858272762e-12, 2.22941255569458, -0.16979369521141052, -0.17004026472568512, 2.83927059173584, 1.958649754524231, 0.4947764575481415, -0.003591301152482629, 0.7673656940460205, 0.8830943703651428, -0.010075108148157597, 3.1275594234466553, 1.4262455701828003, -1.91958685058502e-10, 0.2351476550102234, -0.00018077556160278618, 3.4080698490142822, 2.402759313583374, -0.001260938122868538, -0.010066957212984562, 0.1894536167383194, -0.16975536942481995, 4.575605869293213, 2.194061279296875, -0.16821618378162384, 1.65998113155365, -3.4419939121121956e-11, 0.7655875086784363, 2.587543249130249, -0.1587619185447693, -0.035071857273578644, 1.6213161945343018, 5.8758440017700195, 0.4489915072917938, -0.0010442929342389107, 4.337401390075684, -1.9884365265454562e-09, 1.4104125499725342, -0.07066696137189865, 2.784945011138916, -0.013334063813090324, 3.9714629650115967, 0.7350929379463196, -6.539982422282264e-08, -1.7161143339716745e-10, -0.16275781393051147, 3.8825132846832275, 0.247104674577713, 0.1167476698756218, -0.16959825158119202, 2.1842589378356934, 0.9859612584114075, 0.6725953817367554, -0.004202808253467083, 3.5630815029144287, 4.089195251464844, -0.00011012202594429255, 6.90448522567749, 2.135035753250122, -0.1050407737493515, -1.0447523957068938e-09, 1.922553539276123, 1.593425989151001, -0.06614980101585388, -1.6000750946432163e-08, 3.372429370880127, -0.06854565441608429, -0.11132711917161942, 8.536934852600098, 6.38391637802124, 2.855626344680786, -0.11763420701026917, 1.898393988609314, -1.443775959517838e-14, 0.1998617947101593, -0.025478914380073547, -0.0001366042997688055, -1.1692696944010095e-06, 1.4845190048217773, 0.162242591381073, 0.3277820646762848, 2.5130457878112793, -0.013687428086996078, -0.00026409554993733764, -2.2052560666452337e-08, -0.15834049880504608, 2.255047559738159, -0.1465865820646286, 1.871070146560669, -0.0005192554090172052, -0.11234236508607864, -0.02444789744913578, -0.00022392674873117357, 5.578420639038086, 2.543360948562622, -0.025141363963484764, -0.024434546008706093, -0.07410676032304764, -1.3453563042276073e-05, 1.1823537349700928, -0.00012144044012529776, -0.14823684096336365, -0.002858156803995371, -0.10819198191165924, -7.24654455552809e-05, -1.4070220686335233e-06, -0.0049905721098184586, -0.042042236775159836, -0.16657814383506775, 0.12043984979391098, 4.580685138702393, 2.419114351272583, -0.17003241181373596, 2.1459641456604004, -0.039900027215480804, 4.345462322235107, 1.5664544105529785, -5.069490725873038e-05, 2.23991060256958, 2.4311254024505615, -0.0, -0.02754535898566246, -0.11052320897579193, -5.643715494940693e-10, -0.0018596384907141328, -0.005858821328729391, 1.423570156097412, 0.7356948256492615, 7.87852144241333, -0.14030788838863373, 3.4589719772338867, 0.14038348197937012, 0.6434910893440247, -0.0006692875758744776, -8.853655031337837e-10, -0.024247536435723305, 2.2614266872406006, 4.748415946960449, 1.748505711555481, 1.393755316734314, -0.014475688338279724, 4.602878570556641, 4.031358242034912, -0.08878828585147858, -0.06186056509613991, -0.003776844358071685, 4.401611804962158, 1.0830286741256714, -0.11058539152145386, 1.0848623514175415, 1.1985605955123901, -0.13417552411556244, -0.0002445660356897861, -0.00014008967264089733, -5.100980615679873e-08, 1.2683745622634888, 6.655625343322754, -1.1771195573828663e-07, 0.2741650640964508, 1.0985900163650513, -0.004206636920571327, 0.44775116443634033, -0.0013555233599618077, 3.6007957458496094, -0.06659936159849167, 0.18660305440425873, 2.011712074279785, -0.12739484012126923, -1.575581443091778e-08, -0.08863149583339691, 2.2640461921691895, -0.10141590982675552, 0.5202158093452454, -0.0016182409599423409, 0.2862705588340759, 2.15316104888916, 0.05943230167031288, 5.724043846130371, -0.16617923974990845, -0.0024235998280346394, -6.037566322447674e-07, 3.4552862644195557, -0.07608652114868164, 3.0316131114959717, 1.2562389373779297, 4.8752522468566895, 3.798144817352295, -0.1459447294473648, 0.652601420879364, 3.2545809745788574, -0.00026244125911034644, 4.3526763916015625, 3.635624408721924, 2.9445035457611084, -0.1647910326719284, 0.7410117983818054, -0.001085018622688949, 0.09892977774143219, -0.06885667890310287, 0.15644976496696472, -0.10249100625514984, 0.683351457118988, 3.537466526031494, -9.821471758186817e-05, 2.7690064907073975, -1.0554437879228828e-10, -7.492792193988795e-13, -0.12897038459777832, 0.01025665458291769, 1.2059268951416016, -3.3736577620402386e-07, -0.07707817107439041, 0.24590818583965302, 3.7654101848602295, -0.11766450852155685, 1.4975037574768066, -0.004498452879488468, -0.16141624748706818, -0.1214306503534317, 2.073323965072632, -0.00404426408931613, -0.01497198548167944, 1.6552791595458984, 5.36830997467041, 1.0038658380508423, 0.7455154657363892, -0.15544793009757996, 3.2637901306152344, 1.5343071222305298, 5.199578285217285, 0.8326814770698547, -0.005842204205691814, -0.07475928962230682, -0.004098427947610617, -0.0002169205981772393, 3.790969133377075, 2.4037270545959473, -0.00025393321993760765, 5.3439621925354, 0.5637181401252747, -0.005427404306828976, 1.0156033039093018, -0.08672397583723068, -0.023352567106485367, -3.883011967520389e-10, 0.05991126224398613, 0.7089866995811462, -0.0, 1.5846889019012451, 1.7936773300170898, -0.11208147555589676, 0.2124817669391632, -0.16840921342372894, -0.014659970998764038, -0.15144020318984985, 4.054757595062256, -1.1712260857166257e-05, 0.541904628276825, 0.3919511139392853, -0.035633962601423264, -0.1234140619635582, -2.593142016849015e-05, -0.013079491443932056, 0.02323015220463276, 0.8197413682937622, -0.009008392691612244, 9.169027328491211, -0.07462017983198166, 0.24781566858291626, -0.16921845078468323, -0.13837498426437378, 3.9030961990356445, -0.10859300941228867, 4.0030975341796875, 3.424626350402832, -0.0023201024159789085, 3.103910207748413, -0.04973819479346275, 8.209376335144043, -0.14319473505020142, -0.0062578278593719006, -8.814114438315812e-10, -0.000123440710012801, -0.07110625505447388, -0.0037064894568175077, -0.0002883714623749256, -0.03839487209916115, -0.16594401001930237, 4.348030090332031, -1.0775018921549417e-07, -0.0811997577548027, -0.005575810093432665, -3.954934527428122e-06, -0.00017736195877660066, -0.008471508510410786, 2.117513656616211, -0.1631293147802353, -0.0861511081457138, 1.7727001905441284, 0.6739110946655273, -0.1685047447681427, 1.0091428756713867, -0.01566009782254696, 1.8258503675460815, 1.2875827550888062, -0.13388171792030334, -0.00032394591835327446, 4.560189247131348, 0.17274177074432373, 9.15379524230957, -0.0489119254052639, 2.545538902282715, -0.1257362812757492, 0.5909790396690369, 2.485095977783203, -0.01937267556786537, -0.11470170319080353, 0.33090806007385254, 0.21557550132274628, 1.7234688997268677, 3.073852062225342, 2.4858343601226807, 0.6192784905433655, -0.1551077961921692, -0.0009439917048439384, -0.02990194782614708, -0.08963605761528015, 2.0067598819732666, -0.15036210417747498, -0.00024213829601649195, 1.8224413394927979, -0.018831729888916016, -0.024465832859277725, -0.16882209479808807, 1.3087607622146606, -0.0, -0.0010352876270189881, 0.36914965510368347, -0.1501932293176651, -0.011851477436721325, -0.08488297462463379, 4.155729293823242, -0.11751583218574524, 2.1034505367279053, 2.3551812171936035, -0.11963441967964172, 1.208085298538208, -0.11962804198265076, -0.06144688278436661, -0.000983992125838995, 2.6139612197875977, -0.13894562423229218, -0.08102019131183624, -0.0009680880466476083, 5.426534652709961, 2.1967127323150635, -0.13752487301826477, -0.03853461518883705, -3.059184336962062e-06, 0.7084144353866577, 0.6977773904800415, -2.291637792950496e-05, 3.483996629714966, 2.3050410747528076, -0.005178835708647966, -0.07975227385759354, -0.1283321976661682, 0.33023327589035034, -0.14463141560554504, -3.9872313095656864e-07, 4.422284126281738, 0.7862182855606079, 3.7821295261383057, 3.76426100730896, -0.16619913280010223, 0.8693051338195801, 2.1803159713745117, 2.284996509552002, -0.15317164361476898, -0.05905432254076004, 0.05739213898777962, -0.019064586609601974, -0.07966786623001099, -3.6515034051998185e-11, 0.15032097697257996, -0.0028003042098134756, -0.07970137894153595, -0.06912016868591309, 0.7782829999923706, 5.667515277862549, 0.44766879081726074, -2.1243569790385664e-05, 0.2559525668621063, -0.06611929833889008, 1.0596424341201782, -0.04070319980382919, -0.03546438738703728, -0.09613247215747833, -0.05623992905020714, -7.251883094028244e-09, -0.044988978654146194, -0.020312000066041946, 3.6010560989379883, -0.1668311506509781, -0.14782088994979858, 1.6681355237960815, 1.5325756072998047, 4.798574924468994, 1.6857240200042725, -0.027817875146865845, 4.01492166519165, 4.588442802429199, -0.10992114245891571, 0.32372960448265076, -1.7238386362805613e-07, -0.15020352602005005, -0.010159257799386978, -0.16725315153598785, -0.10750135034322739, -0.15403951704502106, -0.11703602969646454, 1.3774679899215698, 0.5235676169395447, -0.077593594789505, -0.0001301567244809121, 3.7042887210845947, 4.251887321472168, -0.11801011860370636, 1.6065123081207275, -3.2650602861394873e-06, -1.709369826130569e-06, -0.16936103999614716, 4.415037155151367, -0.0022993285674601793, -0.014994421973824501, 4.822783470153809, -1.37380806730647e-09, 1.8877812623977661, 1.3583269119262695, -0.16589070856571198, 0.40415534377098083, 7.865224838256836, -0.0009746571304276586, -0.16320107877254486, -0.16972137987613678, -0.0397147573530674, -6.211922300281003e-05, 4.779900074005127, 0.4921736419200897, -3.3202525173692266e-08, -0.03670085221529007, 1.0798263549804688, 1.0731767416000366, -1.3832791410095524e-05, -0.1075017973780632, 0.6127969622612, 1.0368231534957886, 1.0436091423034668, -7.558842440857916e-09, 5.540948390960693, 0.4666272699832916, 5.417969226837158, 8.5892972946167, -0.1517511010169983, -0.00027031986974179745, 2.580501079559326, -0.15937553346157074, -0.11917771399021149, -0.002627840731292963, 5.063353538513184, -0.12252387404441833, -2.6264455300406553e-05, -0.12065093964338303, -0.0006880780565552413, 0.3231276869773865, -0.13129176199436188, -0.002057391917333007, 1.741123080253601, -0.051664527505636215, 2.1390559673309326, 3.7827606201171875, -0.1508672833442688, -0.05080428346991539, 0.4334619343280792, 2.664809226989746, -0.0041615827940404415, 0.2782409191131592, -0.03774268925189972, 0.769899308681488, 1.8303375244140625, -0.03597802296280861, -0.1071537509560585, 0.07218579202890396, -0.045644115656614304, 0.26890820264816284, 1.5216219425201416, -0.030889861285686493, -0.06360743939876556, -0.022632628679275513, 5.253676414489746, 2.6249172687530518, -0.0010531301377341151, -0.014228437095880508, 5.62153434753418, 4.533888816833496, 0.24982459843158722, 3.4728634357452393, -6.410016794689e-06, 0.001974112819880247, 2.1935367584228516, -3.1317388682339242e-09, -0.02174043096601963, 2.326802968978882, 3.3939661979675293, -0.0669064000248909, 0.702124834060669, 4.8021368980407715, -0.14450079202651978, -7.02151155564934e-05, -0.0786486491560936, -0.14068980515003204, 3.903034210205078, 0.3287869989871979, -0.01898910291492939, 7.201633930206299, -0.10038360208272934, 1.3062814474105835, 1.9857184886932373, 2.3314266204833984, 0.3620826303958893, 1.690930724143982, -0.0854129046201706, -1.9362010789336637e-05, -0.16807475686073303, -0.16422365605831146, -0.15834973752498627, 0.2688264548778534, -0.05708959326148033, 3.969691514968872, 4.700143337249756, -2.143637779219651e-13, 0.014301273971796036, -0.03498966246843338, -0.0, 1.0957382917404175, -0.05886751413345337, -0.14653731882572174, -0.04941004142165184, 3.259507894515991, 3.7176425457000732, 0.6615909337997437, 2.040415048599243, -0.12410827726125717, 0.27873894572257996, -0.028712159022688866, 0.056090690195560455, -0.0343298576772213, 0.9716681838035583, 2.4662137031555176, 0.7693501114845276, 3.551886796951294, 8.070735931396484, -3.9754120134682935e-12, 0.02119816467165947, -0.0751953050494194, -0.11714306473731995, -4.898459883406758e-05, 0.08867134153842926, -0.16993463039398193, -0.05471227318048477, -0.17001713812351227, -0.06709060072898865, -1.560699498172227e-15, 3.2305564880371094, -0.16518078744411469, -0.005792365875095129, 0.7545816898345947, -0.04650313779711723, -4.98171599458419e-13, 2.6508054733276367, -1.8683627445170714e-07, -0.16788767278194427, 0.23096588253974915, 4.048949718475342, 3.569716215133667, 0.34332606196403503, -0.024872874841094017, -0.132568359375, 3.2616078853607178, 5.359347820281982, 1.547333002090454, -0.05126947537064552, 3.0853841304779053, -0.14669755101203918, -0.1354958415031433, -0.16858407855033875, 1.7025049924850464, 0.6568910479545593, 2.9745092391967773, 3.88906192779541, 1.4443556070327759, -2.330399396133394e-15, -1.2394025361572858e-07, 0.3248755633831024, -0.08212203532457352, 6.477588176727295, -0.042047884315252304, -0.14813750982284546, 2.825568199157715, 3.1913866996765137, -0.16956806182861328, 1.3088430166244507, -0.0009036646806634963, -0.006136007606983185, 3.698946714401245, -2.337108708161395e-05, 2.7595791816711426, 1.9911003112792969, 2.285444974899292, 2.730515480041504, 1.6147171258926392, -1.3765707324031595e-12, -0.003487908747047186, -0.06432051956653595, 2.7366161346435547, -0.0009641721262596548, -0.03623592481017113, -0.1506059318780899, 3.9492061138153076, -1.3672241948370356e-05, -5.213426993577741e-05, 1.825124740600586, -2.242055499590556e-12, 0.8367904424667358, -0.11816567182540894, -1.3036072914474062e-06, -0.1309565007686615, -0.003563777543604374, 0.34862685203552246, 0.9411689639091492, 4.466878414154053, -0.01859109103679657, 0.18658791482448578, -0.07223549485206604, -0.033448655158281326, 3.146578311920166, -6.064818647288917e-10, 1.9300097227096558, -0.08718311786651611, 0.13027839362621307, 0.8143401741981506, 2.5385637283325195, 0.5454742908477783, -0.14091362059116364, 0.7714327573776245, 0.018315762281417847, -0.16554120182991028, 0.4497722387313843, 2.609712600708008, 0.2742737829685211, 0.31692853569984436, -0.00013311608927324414, -0.13397356867790222, -3.283213034777077e-09, 2.1270339488983154, -0.006350711919367313, 3.8860251903533936, 8.980311393737793, -0.13921238481998444, 3.988816261291504, 1.0737769603729248, -0.07929392158985138, -0.01419755071401596, 1.6082797050476074, 4.184437274932861, 0.9078038930892944, -4.184561475994997e-06, -0.00240146042779088, -0.16898837685585022, 4.071449279785156, 0.6821419596672058, 0.8215076923370361, 3.6311488151550293, -0.16556045413017273, -0.10549729317426682, -0.15970440208911896, -2.127581488764463e-08, 1.286749243736267, 1.0981018543243408, 2.921745538711548, -0.00047236494719982147, -0.01718038320541382, 2.796785831451416, 0.807938814163208, 4.125039100646973, -0.011993920430541039, 0.4376462996006012, -0.04303446784615517, 0.7652320861816406, -0.10618370771408081, 1.521009087562561, 3.2424895763397217, 1.9825955629348755, -1.0655083038102475e-08, 2.8149242401123047, 1.2219643592834473, -0.0023830635473132133, -0.009993896819651127, 0.012707809917628765, 1.64706289768219, -0.11945009976625443, -5.496735866472591e-07, 0.6764272451400757, 0.35391703248023987, 7.323398590087891, -0.0001312306703766808, 7.57125186920166, 5.23410701751709, 0.7441266179084778, 4.976952075958252, 0.17785333096981049, 2.929030179977417, -0.03023732453584671, 0.1421363353729248, -0.003159662475809455, -1.3398280316323508e-06, -0.16923119127750397, -0.04743931442499161, 1.6446658372879028, 3.8122451305389404, -0.1552581936120987, 0.10080316662788391, 0.4022379517555237, -1.8208673679964704e-07, 0.6378269791603088, 0.017488257959485054, -0.06180474907159805, -0.00130589515902102, -1.3814478727613988e-10, -0.17004014551639557, 0.7857251763343811, 4.02267599105835, 3.3925998210906982, 4.365942001342773, 0.9614641070365906, 3.4678056240081787, -0.16584035754203796, 1.8293174505233765, 2.788665771484375, 8.067974090576172, 0.5579822063446045, -0.00016280599811580032, 4.133162021636963, -0.0205086600035429, -0.07786963880062103, -0.16667892038822174, -0.008328738622367382, 4.004249095916748, 1.7221726179122925, -0.060947153717279434, 0.12114626914262772, -0.12610788643360138, 1.7856180667877197, -0.14561638236045837, -0.02441030740737915, -0.05141965299844742, -0.1247386559844017, 0.7007386684417725, -0.007578861899673939, -0.10571908205747604, 1.208565354347229, 7.0118255615234375, 2.5560858249664307, -0.0, -0.09620372205972672, -0.1393784135580063, 0.7993150949478149, 4.708156108856201, 2.1975722312927246, -6.903207982322931e-15, 0.756381630897522, 0.20329676568508148, 1.1053106784820557, 4.0765581130981445, -0.001447463990189135, 3.3079423904418945, -2.965028329526831e-07, 0.08547160029411316, 0.9968476891517639, 6.8535966873168945, -0.03680666908621788, -1.3100003570798435e-07, 2.021055221557617, -0.046820685267448425, 4.6914753913879395, -0.03554077446460724, -0.010706566274166107, 3.7332592010498047, 0.15406577289104462, 2.998741626739502, -1.626156154088676e-05, 4.95326566696167, -0.06835190206766129, 2.4415926933288574, -0.0034691779874265194, -2.107033196807606e-06, -0.16997170448303223, 0.3661659061908722, 4.729942798614502, 0.8957849740982056, 2.1126086711883545, 1.9293192625045776, 9.330463409423828, -0.01689443364739418, 3.7652664184570312, -0.1465032696723938, 0.61248779296875, -0.08295892924070358, 3.48657488822937, -0.0002057702949969098, -8.220808376790956e-05, -0.16618548333644867, -0.000681834586430341, 2.0732383728027344, 2.75830078125, -0.026666732504963875, 2.329894781112671, -0.04660656303167343, -0.05934368446469307, 0.010306322015821934, 5.629360198974609, -0.0006408377666957676, -0.008038184605538845, -0.007665353827178478, 3.5995781421661377, -2.185387160125174e-07, -0.11190810054540634, -0.14507102966308594, -0.0, -0.11228865385055542, 3.0029072761535645, 0.5217701196670532, -0.008831134997308254, 2.031224489212036, 1.4554986953735352, 6.073498725891113, -0.028604712337255478, -0.14221428334712982, 0.06766517460346222, -0.14092151820659637, 0.15605485439300537, -0.12042265385389328, -0.03849978372454643, -7.78295316194999e-07, -0.07789288461208344, -2.924819455074612e-05, -0.0021748016588389874, -0.08243764936923981, 1.408957839012146, 2.529741048812866, -0.002349162707105279, 0.7259871959686279, 2.355435609817505, -0.07463870942592621, -2.8855419031970087e-07, -5.384741047151365e-13, 2.307758092880249, 2.6834089756011963, 0.6020287275314331, -0.0017257595900446177, 4.294166564941406, 3.5745561122894287, -0.0863991528749466, -0.17000755667686462, -1.378250014116844e-10, 2.148329973220825, -3.980182441409852e-07, 1.9169646501541138, 2.0742547512054443, 0.11831975728273392, 4.661082744598389, -0.15485736727714539, 1.6151461601257324, 2.29040789604187, 0.5012273192405701, -0.1220041960477829, 3.614773750305176, 1.0534967184066772, -0.16780197620391846, -0.16238033771514893, 0.19689686596393585, 6.767882823944092, 0.14542247354984283, -0.1697070151567459, -0.0014652330428361893, -0.0002349456917727366, -6.558548149016019e-10, 0.46149688959121704, 0.004608312156051397, -0.0021262618247419596, -4.1089542719419114e-06, -0.15354835987091064, -0.08134880661964417, -4.5303146180231124e-05, 0.6530649065971375, 0.21825455129146576, -0.09629762172698975, 2.2006542682647705, -0.04164975881576538, 3.9922194480895996, -0.0023568435572087765, 1.3481465578079224, 1.7534005641937256, -0.1672063022851944, 0.8013277649879456, -9.12757168407552e-05, -2.8269041649764404e-06, -0.1693664938211441, -0.025211673229932785, 1.602817177772522, -0.0019264882430434227, 1.6870287656784058, 0.6946825981140137, 0.36051228642463684, 0.10574334114789963, -0.1366790533065796, -0.10175041109323502, 6.009799003601074, -1.5060365399222064e-08, -0.16796880960464478, 2.90958309173584, 1.0173583030700684, 4.128498077392578, 1.4548027515411377, -0.00011777400504797697, -0.1504667103290558, -1.4190044566930737e-05, 2.846618890762329, 4.225458145141602, -0.1092899814248085, 1.4221103191375732, -0.1204267218708992, -7.592817382828798e-06, -0.10586242377758026, -5.488645342666132e-07, -0.06181361898779869, 0.3551132082939148, 2.012953996658325, 3.7597970962524414, -0.14643166959285736, 0.4141285717487335, 0.9857902526855469, 1.4376271963119507, -0.0, -0.016171211376786232, 2.22186541557312, -0.000614111078903079, 0.43395107984542847, 2.3339459896087646, -3.5695252309508874e-10, -0.16984733939170837, 0.39765414595603943, -0.05525580793619156, -1.172668476101535e-06, 0.7137097120285034, -0.05995955690741539, 2.2786245346069336, 0.9758263230323792, -0.06245845556259155, 0.2245875895023346, -0.027134325355291367, -0.025190899148583412, -3.704968776219175e-07, -0.16527733206748962, 2.1860897541046143, 0.07511945068836212, -0.10979335755109787, -0.03606283664703369, -0.06355203688144684, 1.941402554512024, -1.2896102816739585e-06, 5.933904647827148, -0.1471417248249054, 6.253084182739258, 3.252985715866089, -0.0966673269867897, 1.8698512315750122, 0.33079981803894043, 0.9436351656913757, 2.044126033782959, -0.05008732154965401, -0.032570838928222656, -0.09135536104440689, -7.945739525894169e-06, -0.1480018049478531, 0.22892093658447266, 0.27481725811958313, 0.1424834132194519, -0.09648467600345612, 1.969441294670105, -5.630597984390695e-13, -0.09362723678350449, -0.11904119700193405, 2.290140151977539, 1.5249929428100586, 1.9693856239318848, -0.12767834961414337, 2.427480459213257, -0.0008375071338377893, -0.0855996310710907, -3.604218703004425e-14, 4.7081499099731445, 4.338628768920898, -0.02801528386771679, -0.0037520783953368664, -4.5661503143010554e-14, -1.6792112944585824e-07, -0.040101099759340286, -0.1643449366092682, -5.100702651361644e-07, -0.049706313759088516, -0.01722726784646511, 0.7958475351333618, 1.6028658151626587, -0.14887496829032898, -0.04583178088068962, -0.14838816225528717, -6.518541267744662e-14, 0.05229414626955986, -0.16680020093917847, 1.7487972974777222, -1.3749029692533554e-10, 1.7986407279968262, -0.021625792607665062, 0.23638202250003815, 3.2000980377197266, -5.819275088470022e-07, -1.090553905669367e-05, 2.5424561500549316, -0.0001408569805789739, -0.0012892017839476466, 1.2661082744598389, -0.15580207109451294, 1.8582934141159058, 0.3267948627471924, -0.15402935445308685, 2.5217790603637695, 6.148282527923584, -0.16974036395549774, 4.024423122406006, 2.9828782081604004, -0.038846638053655624, 0.7966116070747375, -0.001710420590825379, -0.026504386216402054, 0.27937036752700806, -0.1603887528181076, 3.4933829307556152, -1.4715541510668118e-05, -0.12744006514549255, -0.023132922127842903, -0.16754050552845, 3.083280086517334, -6.184443918755278e-05, 2.1120798587799072, -0.03201797977089882, -0.11644641309976578, 2.583367347717285, -0.13226643204689026, 1.5168393850326538, -0.02603164315223694, -0.1649249643087387, -0.10356365144252777, -2.929487985525725e-09, -0.01073435042053461, 0.061199869960546494, 0.130210742354393, -5.825196467412752e-07, 2.617680072784424, 1.222551941871643, -1.700262146187015e-05, -0.000608245434705168, 1.0083123445510864, -1.7788038819688268e-09, -0.011803679168224335, -0.15724605321884155, -0.17001298069953918, -0.1577916145324707, 0.6502499580383301, -0.007780207321047783, -0.06364230066537857, 5.817656517028809, -3.789757329286658e-06, -0.07710195332765579, -0.0025358241982758045, -0.025620611384510994, 3.333958864212036, -0.05086348205804825, 1.897924780845642, -0.0, 3.24194073677063, 3.1262168884277344, -0.01202287059277296, 3.606116533279419, 0.33287137746810913, 2.460853338241577, 5.1360249519348145, 0.0003745713329408318, -0.0011360036442056298, -0.12192969024181366, -3.8620069972239435e-05, -0.008968215435743332, -0.012296292930841446, -0.018451955169439316, 3.2202956676483154, 2.2069242000579834, -0.004067021422088146, 3.126460313796997, -2.6073044168128945e-08, -0.028321893885731697, -0.16863827407360077, 3.317084789276123, 1.8866862058639526, 0.9779841899871826, 3.6757400035858154, 1.8675559759140015, 3.875356674194336, -0.012159398756921291, -2.8752499048323443e-08, -0.10441483557224274, 2.255932092666626, 0.40956205129623413, 2.4470298290252686, 1.8771693706512451, 1.2260342836380005, 2.22993803024292, -0.02175908535718918, 3.081212043762207, -0.0017576630925759673, -0.159064382314682, -7.021143574093003e-06, 1.1999790668487549, -5.955559458925563e-07, 4.026716232299805, -2.219227098976262e-05, 0.4215107560157776, -0.16552801430225372, -0.006187813822180033, 3.5588269233703613, -0.10308372229337692, 4.211320400238037, -0.0014460842357948422, -0.01162878330796957, -1.3272818608078829e-11, 1.3658939599990845, 2.550658702850342, 0.33308082818984985, -0.1686769425868988, -0.00419838260859251, 6.871382713317871, 2.0027379989624023, -9.419293123258399e-10, -0.14759401977062225, -0.0, 4.558786869049072, 5.756738185882568, 4.210311412811279, -0.12528806924819946, -0.04902109503746033, -9.19690774026094e-06, 1.3105489015579224, -0.018469173461198807, -0.12524175643920898, -0.03748374432325363, -0.14911356568336487, 2.510350465774536, 4.905074596405029, 0.17313911020755768, -0.0503724031150341, -0.0005801775841973722, -0.00013258679246064276, -0.0010370326926931739, 2.48189115524292, 0.418975830078125, 2.7448527812957764, 0.9689974188804626, -0.11755245178937912, -0.00011390475992811844, -0.1306525319814682, -6.655167908320436e-06, 5.874757289886475, 0.4930293560028076, -0.13161124289035797, 5.859282493591309, -0.1255805343389511, 1.4357386827468872, -0.1670660376548767, 0.770084023475647, -0.13180981576442719, 1.2250105142593384, 4.161527156829834, -8.866206917446107e-05, -1.368599583163356e-14, 3.941352128982544, 0.23689980804920197, 2.4866199493408203, -0.15776591002941132, 4.531783580780029, 5.372607231140137, -0.00030181152396835387, 1.8408557176589966, -0.04659043252468109, -2.0113353116357757e-07, -0.15320880711078644, 1.4756182432174683, -1.5159978829615284e-05, -5.332750515663065e-05, 3.5646677017211914, -0.03701655939221382, 0.8809107542037964, -0.00033474882366135716, -0.03181836009025574, 0.09284871816635132, 0.22744576632976532, 0.12833932042121887, 4.59095573425293, 0.1584164947271347, -4.647081937037001e-07, -0.004798758774995804, -0.1462385505437851, 3.591623544692993, -0.020586717873811722, -0.10242289304733276, -0.1353522539138794, -0.016702206805348396, -0.006242924835532904, -0.15459108352661133, 2.6619508266448975, 0.970782995223999, -0.13526371121406555, 0.6942615509033203, -1.968074866454117e-05, 2.64909029006958, 1.817968487739563, 0.3738022744655609, -2.339609572743484e-08, 5.789933681488037, -0.0, -0.0008804702665656805, -0.002071898663416505, 1.6543726921081543, -3.731890956260031e-07, 0.20920976996421814, -0.1682814359664917, -0.13036248087882996, 2.18326473236084, 3.937952756881714, 0.6550660133361816, -0.13674165308475494, -0.0, 4.084249019622803, -0.12757472693920135, -8.86934969912545e-11, 0.06101449951529503, 4.026162147521973, -0.16964156925678253, -0.00010619997192407027, 6.483000755310059, 1.4075112342834473, 0.527286171913147, 2.7312655448913574, -5.384309348339467e-15, -0.16014723479747772, -0.16865907609462738, 6.550899028778076, -0.04218706116080284, -0.045362576842308044, -4.037397047795821e-06, -0.006438961252570152, -0.1634858101606369, 0.3346696197986603, -0.16515137255191803, 5.785756587982178, 1.0528031587600708, 1.5549901723861694, 3.856567621231079, 1.5427348613739014, -0.056219082325696945, -2.4003358589652635e-07, 0.6255012154579163, -0.006384651176631451, 3.712749719619751, -0.010904273949563503, -1.1764684126387313e-15, -0.07129241526126862, 2.967980146408081, -0.1646093726158142, 0.16035757958889008, 0.10262413322925568, 4.950116157531738, 0.46010974049568176, -0.14111794531345367, -0.03244965150952339, -2.9003293988694168e-08, 3.9898457527160645, 3.1980435848236084, -0.0018685944378376007, 1.4621468782424927, 0.5261017680168152, -0.10554534196853638, -0.13170842826366425, 1.9474090337753296, -7.960291986819357e-05, -0.055524226278066635, -0.1590111404657364, -0.12143699079751968, -0.16294538974761963, 0.23012270033359528, 1.1878854036331177, -0.003346098819747567, -0.021944982931017876, -0.053428638726472855, 4.288172245025635, -3.114372930212994e-06, -0.15722188353538513, 1.7641222476959229, -0.14184212684631348, -0.12420933693647385, -0.050880756229162216, 0.48957720398902893, -0.0001492616574978456, -0.15892839431762695, -0.16426868736743927, -1.2355886838122387e-06, 0.393585205078125, -0.07687144726514816, 0.5032909512519836, 2.3346781730651855, 0.6179918050765991, -0.0001422415516572073, 0.5617899298667908, -1.0706671815796653e-07, -0.012642426416277885, 4.847076892852783, -0.08852330595254898, 4.514878749847412, -0.16997410356998444, -0.07930280268192291, -0.00040429746150039136, 2.6496760845184326, -0.1694251000881195, 1.629223346710205, 0.9888080954551697, 1.3342171907424927, 0.6479731798171997, -0.11025140434503555, -0.07112541794776917, -0.026707641780376434, -1.9701040386739532e-13, 1.2253456115722656, 6.194052219390869, 3.157665491104126, 6.985862731933594, 1.2563499212265015, -0.003497718134894967, -0.0028325465973466635, 2.162759304046631, 0.8139793872833252, -0.16096170246601105, 2.174093246459961, -0.008840277791023254, 1.4568930864334106, -0.166471928358078, 0.3037112355232239, 3.5824341773986816, -0.14497794210910797, 4.165616035461426, 4.026020050048828, -0.02384994551539421, 7.0589447021484375, -0.00021173302957322448, -0.11303157359361649, 2.223573923110962, -6.522833437798284e-15, 3.1540369987487793, 3.4785685539245605, -0.11915124207735062, 0.686211884021759, 0.5144699215888977, -0.16490572690963745, 3.287033796310425, -0.149412602186203, -0.04638898745179176, 0.9996668696403503, 2.6066973209381104, 3.3847618103027344, -6.3892898651829455e-06, -2.1560420115918077e-11, -0.13504375517368317, -0.05958406627178192, -0.0015112020773813128, 0.7968493700027466, 1.8930411338806152, 1.0518691539764404, 5.04642915725708, -0.0024307523854076862, 0.052163343876600266, 5.894752502441406, 2.0536131858825684, -0.0009952438995242119, 0.47143298387527466, 3.610805034637451, 6.35964298248291, 1.9954379796981812, 3.571931838989258, -0.001046559540554881, -0.17000529170036316, 3.6504099369049072, -0.0048601566813886166, -0.07579062879085541, -1.444742547818123e-09, 4.512010097503662, -0.13634313642978668, 1.656287431716919, -3.4143183711421443e-06, -7.246297900564969e-05, 5.52793025970459, -0.15521757304668427, -0.0811980590224266, 1.4643999338150024, 1.6102436780929565, -1.0996131782903262e-10, -0.004009302239865065, 0.08666062355041504, 0.00810036901384592, 0.917010486125946, -0.12519565224647522, -0.16799940168857574, 0.4864576756954193, -0.03753199428319931, -0.013565952889621258, -0.07933944463729858, 0.008389221504330635, -0.148360013961792, -0.10554977506399155, -0.16997721791267395, -0.06910333037376404, 0.8036165833473206, 0.9408609867095947, -0.00010007489618146792, 2.941201686859131, -5.109371272737917e-07, 1.5609118938446045, -0.032695624977350235, 1.6689077615737915, 3.19404935836792, -8.630362535733127e-10, 1.7344255447387695, 0.6308510303497314, -0.10820668190717697, 0.986388087272644, -0.0958467572927475, 0.2027757614850998, -0.001729727489873767, 2.3156344890594482, 1.6755812168121338, 2.179685115814209, -0.04333595186471939, 5.526397705078125, 5.941000938415527, -0.11658823490142822, -0.12543004751205444, -0.12876947224140167, 1.9930349588394165, 3.06516695022583, -0.024128127843141556, -1.5654597410041937e-11, 1.208126187324524, -0.07336827367544174, 0.3222638964653015, 1.4184201955795288, -0.000865054433234036, 0.08002985268831253, -0.16225481033325195, 0.9295939803123474, 3.6302881240844727, -0.02849130518734455, -0.009899433702230453, 3.6959285736083984, 2.0287058353424072, -0.16933205723762512, -4.362886496522833e-09, -0.007948885671794415, 0.21381650865077972, 1.4623441696166992, -0.07575895637273788, 1.0714384317398071, -0.042745497077703476, -0.11388776451349258, 1.3866480588912964, -0.16321192681789398, 0.18250778317451477, 2.5003836154937744, 3.279193162918091, -0.053195346146821976, -0.1657419502735138, -3.541045225574635e-05, -4.7237776016118005e-05, 0.5836754441261292, -0.07352466881275177, -7.940715862631009e-10, 1.3350802659988403, -0.00917824637144804, -0.14926885068416595, 0.9483979344367981, 0.8822657465934753, -0.12338966131210327, -1.780517053240338e-11, 1.4458532333374023, -0.04934176430106163, -7.556262062280439e-06, -2.0424847658453635e-14, 3.8124613761901855, 0.15949749946594238, 0.09932046383619308, 1.5935108661651611, -0.13722755014896393, -2.733854216785403e-06, 1.8301266431808472, 2.849365234375, 1.2853801250457764, 4.4710845947265625, 5.579573631286621, -0.013225599192082882, -0.10219160467386246, 2.5027472972869873, 0.2484675496816635, 0.3352321982383728, 2.6549692153930664, 2.0430781841278076, -0.10415396094322205, 0.1382714807987213, -0.02546258270740509, -0.16988633573055267, -0.0006710276356898248, -5.853612039530276e-11, -0.08433303982019424, -0.053522154688835144, 2.5399253368377686, 4.597333908081055, -0.024387964978814125, 1.1657627820968628, -0.042790647596120834, -0.00494374567642808, -0.1609814614057541, -0.16101233661174774, 0.6058887839317322, 0.1356351226568222, 4.870057106018066, -0.17004019021987915, -1.571935968058824e-07, 0.15802980959415436, -0.03902747482061386, 0.3602527678012848, 3.279383659362793, -0.043662648648023605, -0.11963288486003876, 1.355715036392212, -0.004581725224852562, 0.39938628673553467, 1.4565858840942383, 2.3330140113830566, 3.208444595336914, 1.4949387311935425, 0.947947084903717, -2.0469755327212624e-05, -0.0, -0.004077502992004156, 4.197331428527832, 0.17378729581832886, -1.8557107822505907e-14, -0.12269174307584763, 0.8657898902893066, -2.477293037372874e-09, 1.5805723667144775, -0.0006339605315588415, 3.0664963722229004, 0.02062436379492283, 9.23375415802002, -5.876328214071691e-05, 0.6351572871208191, -6.360623501677765e-06, 3.405810594558716, 0.21516470611095428, 2.7215917110443115, 0.16997581720352173, -0.0005304074729792774, -0.04731255769729614, -0.00023810805578250438, -0.002167812781408429, 0.2720986604690552, 3.1846272945404053, 2.538480520248413, 1.4229087829589844, -0.07490457594394684, 1.4105710983276367, -0.012323603965342045, 0.4348612129688263, 2.475781202316284, -0.1626448780298233, -0.11034329980611801, -0.1246139332652092, 1.7572333812713623, -0.16303536295890808, -1.7996277165366337e-05, -0.13151654601097107, -0.0018619017209857702, 4.471778392791748, 0.010258329100906849, -8.022463589441031e-06, 0.6079385280609131, 2.1005446910858154, 0.9706295728683472, -0.15008950233459473, 0.28614404797554016, 0.6490301489830017, 2.9803082942962646, 0.9202651381492615, 6.76505184173584, -0.1679697483778, 4.556008815765381, 0.0207075048238039, -1.0795434945976012e-07, -0.0005681092734448612, -8.875529644569724e-09, -8.17643197592588e-08, 3.5799033641815186, -0.04720333218574524, 1.7227379083633423, -0.009536225348711014, -0.023188207298517227, -0.0021570974495261908, 2.5630576610565186, -0.16997890174388885, -0.11827478557825089, -0.15405713021755219, 6.489622116088867, 1.2048496007919312, 2.4707367420196533, -0.0002487003512214869, -0.04979199916124344, 0.7039104104042053, -3.701678963352606e-09, 3.9798591136932373, 0.6286200881004333, -0.030157482251524925, -0.16481350362300873, -0.16967734694480896, -0.09789709746837616, 4.146756172180176, 1.9997081756591797, 3.70169997215271, -0.16468988358974457, -0.010495956987142563, 4.042871952056885, -0.0003396192332729697, -0.061323586851358414, 2.0759172439575195, 3.225072145462036, -9.27967339521274e-05, -0.16875694692134857, 0.036071259528398514, -0.1401817351579666, -0.012254794128239155, 1.958276391029358, -0.08998611569404602, -0.16995643079280853, -5.008727062120501e-14, -0.1255577653646469, -4.509015660847382e-10, 2.0550448894500732, 1.3493239879608154, -0.09599246084690094, 0.31706055998802185, 0.1911260038614273, -0.14800076186656952, 0.9102892279624939, -0.13498391211032867, -0.1082288920879364, -0.01987491175532341, 4.981761932373047, 0.3998679518699646, -0.03656210005283356, 2.1029903888702393, -0.14094308018684387, -0.14309526979923248, 1.919899344444275, 3.0464906692504883, 1.9377171993255615, -7.301746791199548e-06, 0.08637597411870956, -0.04069390147924423, 1.2138549089431763, -0.010026837699115276, -0.0022506567183882, -0.13101214170455933, -0.00971154123544693, -0.0019115686882287264, -0.09339691698551178, 2.726656436920166, 0.2475234717130661, 3.298032760620117, -0.0037902956828475, 2.0411648750305176, -0.027966313064098358, -0.010566792450845242, -0.12969224154949188, -0.012330972589552402, -0.16848674416542053, -0.027317922562360764, 3.7112488746643066, -0.1500549167394638, -2.009449599427171e-05, -0.16523988544940948, 0.9687668681144714, -0.16104456782341003, 2.5097877979278564, -0.008168566040694714, -0.012792399153113365, 4.7875165939331055, -0.008081362582743168, 1.485848069190979, 1.3311532735824585, 1.6024562120437622, 0.0536065399646759, -0.16400198638439178, 3.178931713104248, -0.10298941284418106, 0.1969948559999466, 1.9571348428726196, -0.1388585865497589, -3.629170350905042e-06, 1.2127951383590698, 3.1669156551361084, 0.5608089566230774, -2.735060604663886e-07, 1.8143173456192017, 5.790348052978516, -3.1865040455159033e-06, -0.0, -0.17000511288642883, 0.12266874313354492, 2.2658467292785645, 2.3813819885253906, 0.16018196940422058, 1.4968081712722778, 0.8208063840866089, 0.22981342673301697, -0.07449886202812195, -0.14767886698246002, 2.0245814323425293, 1.0494463443756104, -0.13121339678764343, 3.210401773452759, -0.03459389880299568, -0.16888292133808136, 0.2576480507850647, -3.0097346837010264e-09, -0.14844246208667755, 2.493591785430908, 2.3680760860443115, -0.0006106790388002992, -0.01782134175300598, -0.0021770349703729153, -0.16921012103557587, -0.08002094179391861, -1.8720099888014374e-06, 0.7674016952514648, 0.12221869081258774, -0.08895496279001236, -0.04807467758655548, -0.1589074581861496, -5.3223917717559743e-08, 0.5090742707252502, 0.4528266489505768, -0.11703750491142273, -0.0007382990443147719, 3.7277309894561768, 3.307008981704712, -6.9012693337811015e-09, 1.0746994018554688, 0.30087095499038696, -0.0, 1.417074203491211, 1.8500263690948486, 2.0729122161865234, 0.32288411259651184, -0.05417349189519882, -0.10112646222114563, 1.399951457977295, 3.4282443523406982, -0.11011730879545212, -0.16939274966716766, 2.002258062362671, -5.643101003843798e-12, 1.1904289722442627, -0.16834154725074768, -1.7758596242839486e-12, 2.2349724769592285, -0.009895024821162224, 2.301480531692505, 4.03312873840332, -0.00037554019945673645, -0.0006520234164781868, -0.00913360808044672, -1.200453283445313e-07, 2.1146697998046875, 0.40392622351646423, 2.88253116607666, -2.2684897516001e-08, 0.4497619569301605, 1.2663633823394775, -4.9110582267530845e-08, 2.6290504932403564, -7.370769162662327e-05, 0.40539246797561646, -0.00017256847058888525, 2.322350025177002, 5.02732515335083, 0.13577601313591003, 0.08403428643941879, 3.8628575801849365, -0.13378745317459106, 0.13037502765655518, 2.461379289627075, 1.204236626625061, 4.700840473175049, 3.614245891571045, 5.750772953033447, 0.04936040937900543, 3.4359395503997803, -0.029334107413887978, 2.833956480026245, 5.340497016906738, 7.097175121307373, -0.0008046288858167827, -0.10399454832077026, 2.3130059242248535, -0.023489363491535187, 1.8094348907470703, 1.7863678932189941, -0.1435755491256714, -0.12599866092205048, -0.01799282245337963, 1.0276435613632202, 3.9891676902770996, 0.982765793800354, 8.408056259155273, -5.206114656175487e-05, -0.1107105165719986, 2.2258267402648926, -0.002911456860601902, 1.176510214805603, 2.661820888519287, 0.636844277381897, -0.13328665494918823, 3.039030075073242, 1.408445119857788, 2.3906455039978027, 1.8304864168167114, -0.07703541964292526, -0.0042349728755652905, -0.16868099570274353, 0.6338937282562256, -0.10935179144144058, 3.054955244064331, 0.868121325969696, 3.1214230060577393, 0.450119286775589, -0.05976146087050438, 2.7384517192840576, 1.6554744243621826, -0.0292211826890707, -0.1625509262084961, 8.469788551330566, -0.0, 3.564197063446045, 0.2649676203727722, 0.01181567832827568, -0.16807949542999268, -0.12635205686092377, 7.182684421539307, -0.14836524426937103, -0.1696075201034546, 2.536797046661377, 1.1843061447143555, -0.12896955013275146, -0.16881978511810303, -0.0001959150395123288, 3.352522611618042, -0.003889848478138447, -1.5614426637043243e-15, 3.238215684890747, 2.332853078842163, 0.9910781979560852, -0.11063995212316513, 3.644592046737671, -0.002650962211191654, -2.0250663125553903e-12, 0.4181055724620819, 2.4732582569122314, 1.8866891860961914, 6.259979724884033, 2.5502755641937256, -0.1637534648180008, -0.15266360342502594, 2.7243547439575195, -0.047135405242443085, -0.11982306838035583, -7.10587002572538e-08, -0.16816599667072296, -0.00018705871480051428, 0.5990201830863953, 0.3192557692527771, 2.158384323120117, 5.395162582397461, -0.1497599333524704, 4.367410659790039, -0.0012893958482891321, 0.3751731216907501, 2.9054415225982666, 5.897923946380615, -0.0017140202689915895, -0.1694757342338562, 0.33294790983200073, -0.0967027023434639, -0.006925514899194241, -7.496290663766558e-07, -0.00040875148260965943, -0.07811274379491806, -8.832731691654772e-05, 5.363218307495117, 3.7316975593566895, -1.9235204540279938e-09, -0.11101037263870239, 4.019672870635986, 4.391485214233398, 3.396359443664551, -2.295014201081358e-05, -0.15497899055480957, 3.880223035812378, 0.23247236013412476, -0.1245836690068245, 1.4693957567214966, 2.996992349624634, 0.3181953430175781, 0.6167889833450317, 3.2644755840301514, -0.0835714116692543, -0.009260482154786587, 2.318753957748413, -0.0005255998112261295, 1.439366102218628, -0.034466248005628586, 2.5482370853424072, -0.000574926903937012, 3.4442055225372314, 0.7303969264030457, -0.16734424233436584, -0.013623116537928581, -0.1424461156129837, -0.008142734877765179, -0.000495706801302731, 3.2558834552764893, 1.7594777345657349, 3.7251362800598145, -6.355030927807093e-05, -4.737613079619507e-10, -1.39623477934947e-06, 1.9319233894348145, 2.4272267818450928, -4.55479209904297e-09, 3.087937355041504, -0.12759332358837128, -0.0004678653785958886, -0.09094157814979553, 1.6462011337280273, -0.16664497554302216, -0.15638308227062225, -0.014439478516578674, -1.9617024008766748e-05, -0.1468956023454666, -5.255484438748681e-07, 1.739670753479004, -0.012889674864709377, 3.9328572750091553, -1.0271820656271302e-06, 4.216950416564941, -0.010059470310807228, -0.009359525516629219, 0.710012674331665, 2.690152406692505, -0.05793575942516327, -0.1513296365737915, 5.984899044036865, 1.4223456382751465, -0.0003835642710328102, -0.16961334645748138, 5.0776686668396, 0.8241804242134094, 2.4577579498291016, 6.1022868156433105, -0.06557406485080719, 0.9711315631866455, -0.044519226998090744, -0.011722016148269176, 1.5219385623931885, -6.563042552443221e-05, 0.28777679800987244, 0.27899816632270813, 3.158784866333008, -0.06579744815826416, -0.0001514303730800748, 1.0374325513839722, -0.0009452637750655413, 0.37944158911705017, 1.6188157796859741, -0.010195002891123295, -0.13414473831653595, -0.0023279404267668724, -0.0, -8.970327144197654e-06, -4.456008045189419e-12, 0.009061416611075401, -0.0015619468176737428, -0.1501070261001587, -0.07979564368724823, 3.152754068374634, -0.15068797767162323, -0.13196925818920135, 1.3711400032043457, 4.812501907348633, -0.10773035883903503, 0.8231146931648254, -0.16604875028133392, -0.03369713947176933, -1.7531194771436276e-06, -0.10999197512865067, -0.1681751161813736, -0.16976720094680786, 5.926034450531006, -0.15429265797138214, 0.7937829494476318, -0.15088053047657013, 3.513240098953247, -0.16541816294193268, 5.562604904174805, -0.16904474794864655, 2.2811472415924072, 3.030858278274536, -0.17001111805438995, 2.5301690101623535, 0.09357522428035736, -0.02388584427535534, -0.04162043333053589, 0.8155433535575867, 5.346043109893799, -0.10031041502952576, 1.5286120176315308, -0.017117636278271675, 0.5772615075111389, -9.01439125300385e-05, -0.019527437165379524, -0.0219576396048069, -0.025693489238619804, -0.16854214668273926, 3.0074734687805176, -0.13781975209712982, -0.03150945156812668, -0.0002343822707189247, -0.08510874211788177, 3.44026517868042, -0.1043613851070404, -0.1482386291027069, 2.1170897483825684, 2.4841010570526123, 2.9962613582611084, -0.1497974544763565, -0.007261214312165976, -1.81436917046085e-07, -0.038848377764225006, -0.14544567465782166, 2.1392147541046143, 0.7768592834472656, -0.07319080829620361, 2.6239402294158936, -0.00017504052084404975, -0.0019830255769193172, -0.0004876773455180228, -0.0012837701942771673, -0.16972890496253967, -0.01261704321950674, -0.02355795167386532, 3.9890334606170654, -7.5274929258739576e-06, -0.002357006538659334, -0.17003799974918365, 1.173112154006958, 1.31830894947052, -0.1688184142112732, -0.1456007957458496, -0.16868901252746582, 2.168729782104492, -0.0066879806108772755, -7.874830370514156e-16, 0.3648498058319092, -0.0005161123699508607, -0.08805402368307114, 4.137066841125488, 0.15730787813663483, 5.478753089904785, -8.044083870117902e-07, 0.5148396492004395, 0.9931113719940186, 2.442615032196045, -0.029073279350996017, 0.3051837980747223, 0.21364490687847137, 0.14545366168022156, -3.4522192436270416e-05, -5.6672821102665694e-08, 1.5152490139007568, 0.8672370910644531, 0.05244459584355354, 1.8467164039611816, 3.2320525646209717, -0.0006299186497926712, 2.3218376636505127, 4.476175785064697, 3.0737171173095703, 1.7225465774536133, 2.103048801422119, -1.5447063560714014e-05, -0.10657171159982681, 3.1141467094421387, 4.493832588195801, 1.3276865482330322, -0.12533020973205566, -0.1266389787197113, 2.6218252182006836, 6.139531135559082, 2.442619562149048, -0.104264035820961, 1.9276124238967896, 1.2983132600784302, 7.3119306564331055, 1.8507027626037598, 0.13446766138076782, -0.04798826947808266, -0.0004765013582073152, -0.0032461562659591436, -0.12543414533138275, -0.02409336157143116, 0.5344880819320679, 0.28779590129852295, 2.9528274536132812, 4.684287071228027, 0.43615442514419556, -0.14663845300674438, -9.69185265375927e-09, 1.1520791053771973, 2.0867373943328857, -0.03224603086709976, -0.15571483969688416, -0.04900911822915077, 3.315481662750244, 1.9544000625610352, 3.788853645324707, -0.0006654142052866518, 8.11732006072998, -0.017878687009215355, -0.004372919909656048, 0.5275516510009766, -0.09546524286270142, 5.573956489562988, -0.147398442029953, -0.050636205822229385, 3.036733388900757, 3.7755916118621826, -0.018819764256477356, -0.12213186174631119, -0.16040104627609253, -0.13513487577438354, 5.2277679443359375, 3.865473508834839, 3.287642002105713, 3.9103312492370605, 0.07548023015260696, 2.6907403469085693, -0.06262997537851334, -0.03150292485952377, 5.069293022155762, -5.494333521482986e-08, 0.03913548216223717, -0.07935494929552078, -3.530928438522096e-07, -0.1500331610441208, 0.14976531267166138, 1.3134069442749023, -0.01751711592078209, 2.2694742679595947, 0.7896865010261536, 2.4429519176483154, -7.306468208234662e-10, -0.12119098752737045, -0.06115879490971565, 1.6662755012512207, -0.029629193246364594, -0.09646716713905334, -0.0008123659063130617, -0.0004350529925432056, -0.006470138672739267, 2.875718832015991, -0.001846800558269024, -0.0, 0.6768313646316528, -0.11452679336071014, 1.5374717712402344, 1.4742481708526611, -0.16038906574249268, -0.12549258768558502, -0.07541758567094803, -0.03460690751671791, -0.02016056515276432, 0.3772735893726349, 4.095523357391357, -0.04134627804160118, -0.05145539715886116, -0.04919293895363808, 1.7220603227615356, 1.867775321006775, 4.254762172698975, 2.058640718460083, 0.21911868453025818, 1.2085384130477905, -8.915699436329305e-05, 1.5796297788619995, 0.3413929045200348, 6.2020039558410645, 4.103056907653809, -0.015530419535934925, -0.07435718923807144, 2.6064279079437256, -1.18687082731328e-11, -0.007404780015349388, 0.9454965591430664, 1.8650846481323242, 0.3844655752182007, 3.4775125980377197, -0.169071227312088, 0.170564204454422, -0.16344180703163147, -0.001403271802701056, 0.0722852349281311, -0.08093146979808807, 0.13999255001544952, -0.0, 0.20276401937007904, -0.08438146859407425, -0.06006401777267456, 0.07377597689628601, -0.11145396530628204, 2.7678563594818115, 3.735121965408325, 2.6344447135925293, 2.884416103363037, 0.9396309852600098, 0.052028264850378036, -0.00014932506019249558, -0.1492362767457962, 3.6284608840942383, 2.271204710006714, -0.10528909415006638, 0.20712529122829437, 3.1695070266723633, -0.15099121630191803, 4.862071990966797, -0.026592707261443138, 2.350792407989502, 2.051494836807251, 0.3692556321620941, 0.7485184669494629, 1.53987717628479, -2.898886913935833e-14, 1.310487151145935, 3.4391965866088867, 1.073315143585205, 3.4233217239379883, -0.0005658313748426735, -0.07932338863611221, 0.8485569953918457, -0.1699880212545395, 3.574789047241211, 1.3691993951797485, -0.09875158220529556, 2.44227933883667, 2.449418544769287, -0.1274925023317337, 1.8385815620422363, 1.0041142702102661, 0.9260029196739197, -0.00023784923541825265, 3.833164691925049, 1.9042507410049438, -0.029821129515767097, 0.903502345085144, 0.8601821660995483, -0.16587978601455688, -0.07176342606544495, 0.18920539319515228, -0.0678647980093956, -0.10795797407627106, -0.0006782870623283088, -0.1691315919160843, 3.0177013874053955, 0.4800187647342682, 0.08333436399698257, -0.1633111983537674, -0.1299116015434265, -5.833386694575893e-06, 4.961423873901367, -0.03211115673184395, -5.443627742351964e-06, 1.0642009973526, 0.592112123966217, -0.09380429983139038, -8.212086868297774e-06, -3.629192360676825e-05, 4.699921607971191, -0.12759938836097717, -0.048921193927526474, 5.574336528778076, -0.08371409773826599, -0.0072385454550385475, 1.5470805168151855, 0.9499669671058655, 4.6867289543151855, -1.8809515722750803e-06, 0.6962065696716309, -1.2675259597472177e-07, 1.2649301290512085, -0.04849942773580551, -0.11491841822862625, -0.02970358543097973, 2.4997894763946533, 4.866916179656982, -0.10340198874473572, -0.16994132101535797, -0.09883908182382584, 1.4724195003509521, -0.0026515608187764883, 10.628511428833008, -0.16951456665992737, 5.133297920227051, 0.29429587721824646, -3.818128243437968e-05, -1.5950612350934534e-06, -0.128975972533226, -0.032461944967508316, -0.1700323075056076, -0.0006767575396224856, 7.025782108306885, 6.367125511169434, -8.119963368358185e-09, 1.305310606956482, 3.56636381149292, -0.15488660335540771, -0.16980299353599548, 3.234971046447754, 1.489676833152771, -0.00027745371335186064, 3.6343023777008057, -0.09794388711452484, 5.533998489379883, -0.004379990044981241, -3.860968026547654e-15, 0.14167006313800812, 3.5875775814056396, 1.3828115463256836, 0.2618078887462616, -0.005753216799348593, 2.6752545833587646, -1.5394735486395916e-09, 2.275613307952881, -0.00022849597735330462, -0.05345766991376877, -4.606040602084249e-05, -0.005351451691240072, 4.363259792327881, -0.00028902574558742344, 0.29290416836738586, -0.0128346998244524, 2.9516804218292236, -1.9097187760053203e-05, -0.06488766521215439, 2.560992479324341, 6.678278923034668, -0.00021900450519751757, -0.006201090291142464, -0.12265259772539139, 4.302006244659424, -0.001416242215782404, 2.2764270305633545, -0.038203369826078415, -0.0011528641916811466, -2.002407200052403e-07, 0.11698020994663239, 2.7673072814941406, -0.15688364207744598, -1.3207410696786798e-13, -0.05292900651693344, 2.711016893386841, 2.410229444503784, 0.4493022561073303, 4.262368202209473, -4.777591675519943e-05, -7.932855805847794e-05, 7.024021148681641, 1.7473424673080444, -0.16830289363861084, -0.07431994378566742, -0.0, 2.296492338180542, -4.7210846343404e-13, -1.0246260437329902e-11, -0.13522258400917053, 4.3540544509887695, -0.10791721194982529, 0.738756000995636, -0.09269534796476364, 0.8126702904701233, -2.4518177710319833e-08, 6.180887222290039, -0.002031899755820632, 1.371709942817688, -8.070497937474885e-13, 5.187461853027344, 0.9477248191833496, 5.891942977905273, -0.02828674577176571, 2.944887399673462, -0.17003943026065826, -0.1270129382610321, 3.760390520095825, -0.004025064408779144, 2.015632390975952, 5.748566627502441, 5.1580424308776855, 3.0627408027648926, 0.6618151664733887, -0.0023600172717124224, 4.223082542419434, -0.027686746791005135, 3.521991491317749, -0.14241628348827362, -0.0515759103000164, -0.06711733341217041, 1.6392967700958252, -0.10261549055576324, 0.27030643820762634, -0.07234901934862137, 1.3789080381393433, 0.5616875290870667, 1.0333441495895386, -0.008429691195487976, 0.6912572383880615, -0.0023510220926254988, 1.5663365125656128, 0.010909687727689743, -0.0025834168773144484, 2.0412344932556152, -0.030741170048713684, 0.556868851184845, -8.484800861040254e-13, 1.2758665084838867, 1.817627191543579, -0.004184294957667589, 1.5499749183654785, -0.1278233528137207, 4.065688610076904, 1.819153904914856, -8.577404514653608e-05, -0.1699559986591339, -4.32454116960912e-09, -0.1461453139781952, 1.4112684726715088, 2.359989881515503, -0.15111108124256134, -0.16692203283309937, 0.22005389630794525, -0.04685495048761368, 0.687094509601593, 3.2766802310943604, 0.311585396528244, 2.4835402965545654, -0.015153966844081879, -0.08636530488729477, 0.7593835592269897, 6.0632500648498535, 2.987929582595825, 5.120476722717285, 0.03973209112882614, 3.709853410720825, -5.946708370174747e-08, 1.4360733032226562, -1.6165468486387113e-12, 3.7029614448547363, 1.3161611557006836, 3.5179271697998047, 1.0638947486877441, -0.09616401791572571, -0.0257320124655962, 0.40883636474609375, 4.553812026977539, 1.8550715446472168, -0.08263861387968063, -0.07330486923456192, -0.03424784541130066, -0.16705960035324097, -0.10594059526920319, -0.10581508278846741, 0.017720013856887817, -9.521839956505573e-07, 4.761826992034912, 0.5953980684280396, 4.656804084777832, 0.7363392114639282, -0.16206374764442444, 0.17876024544239044, 0.5353617072105408, -0.04781295359134674, -0.1480831652879715, 5.843400001525879, 2.889880895614624, -0.09719736129045486, 1.5549287796020508, -0.015720993280410767, 4.315765380859375, -0.00361937889829278, 2.769319772720337, 0.0826905146241188, 0.0996594950556755, -0.01978030614554882, -0.16967636346817017, -0.027853483334183693, -0.013521965593099594, 1.366520881652832, 7.6913042068481445, -0.1615390032529831, 3.10001277923584, 0.017149806022644043, -0.03799161687493324, 4.135693073272705, 1.951215386390686, 3.524501323699951, 0.28036823868751526, 0.3594107925891876, -0.019234182313084602, -0.16995322704315186, 5.168243408203125, -1.8929868387867685e-14, 1.1256871223449707, 4.531596660614014, 2.4917092323303223, 2.7468934059143066, -0.16964447498321533, 3.51228404045105, 3.596514940261841, -0.0072322022169828415, 5.170797824859619, -0.11781875044107437, -0.0, 0.0380738228559494, -0.09636391699314117, -0.016398612409830093, -0.12799587845802307, -0.0972273126244545, -1.0694840568703512e-08, 3.825831890106201, 0.06493842601776123, 8.217567443847656, 0.029883651062846184, 1.17354416847229, -6.421111265808577e-06, 3.170426607131958, -0.0019248499302193522, 1.653226613998413, 2.886364698410034, 2.3848519325256348, -0.01846080645918846, -0.02287156507372856, 2.361010789871216, 3.6688332557678223, 4.461630821228027, 0.0039862291887402534, -0.09279460459947586, 5.504887104034424, -0.07616309821605682, 2.3646435737609863, -0.16790807247161865, -2.470569233992137e-05, -0.0013775710249319673, 1.3401331901550293, 0.575727105140686, 1.4736498594284058, -0.1602843552827835, -0.10843326151371002, 0.7687292695045471, -0.15149356424808502, -1.0148780802410329e-06, -0.13328169286251068, 5.180476188659668, -0.027981482446193695, 1.6489993333816528, -0.07865636050701141, 2.237673282623291, -0.012646972201764584, -0.00017740140901878476, 1.2825480699539185, -0.039655789732933044, -0.031652696430683136, -0.16533733904361725, 2.9359264373779297, 0.6396476626396179, 0.310933917760849, -0.004989247303456068, -1.6680091903253924e-06, -0.015443402342498302, 2.527066946029663, 2.0745739936828613, 1.7158559560775757, -0.1402222365140915, -0.0029951301403343678, 3.47623872756958, 0.40800419449806213, 1.541975736618042, -0.006424951832741499, -0.04187968000769615, 1.9045608043670654, 2.2047393321990967, -0.0038171722553670406, 0.7113432288169861, -0.0006093933479860425, 3.671346426010132, 1.9562665224075317, 6.834639072418213, 4.540142059326172, -0.04786311462521553, 0.4344782829284668, -0.11057966947555542, 0.286722332239151, 2.3317599296569824, -0.006756559945642948, 0.10664328187704086, -6.712611502734944e-05, 1.667393684387207, -0.019885072484612465, 1.8187612295150757, 3.119683265686035, 0.05317647010087967, 2.0588862895965576, -0.06382981687784195, 2.8280351161956787, -0.031119775027036667, -0.06763552129268646, -0.16918690502643585, -2.0135459635639563e-05, -0.05025104060769081, -0.127731591463089, 0.03973417356610298, -0.10465516895055771, -0.05151398852467537, -0.1611248105764389, -1.7279158789559879e-12, 0.1582341492176056, 1.5745958089828491, 2.321983575820923, 3.2036547660827637, -0.166726216673851, -0.10629145056009293, -0.10675176978111267, -0.10476481914520264, 0.21803513169288635, 3.9845664501190186, -1.3375245089264354e-07, -0.09861382097005844, 0.6199780702590942, -0.0034545157104730606, 1.0557808876037598, -3.565336373867467e-05, -0.150990292429924, -2.5937239911399956e-07, 0.5923689603805542, -0.01915251463651657, 1.8060561418533325, -0.16305787861347198, -0.004484393168240786, -0.12238448858261108, 0.17955009639263153, -0.05555233359336853, -0.16791985929012299, -0.14544670283794403, -0.03149700537323952, 2.3965330123901367, -0.06442804634571075, 1.065834641456604, -5.598350253421813e-05, 2.634678602218628, 0.8172638416290283, -0.14458462595939636, -0.151584193110466, -8.518329508433453e-08, 0.09955957531929016, -0.16888976097106934, -2.0430548829608597e-05, -0.1118331328034401, 1.5289301872253418, 0.28359559178352356, 0.5443513989448547, -0.07100734114646912, 2.407907247543335, -0.07045256346464157, 2.416724920272827, 6.495809078216553, -6.452451073446497e-11, -0.09113746136426926, -0.02752634324133396, -0.02658635936677456, 2.9910900592803955, -0.14889416098594666, 4.902856826782227, 2.158493757247925, 1.0638483762741089, -0.05040034279227257, 2.446082592010498, 1.3071452379226685, 4.3285136222839355, 1.0807430744171143, 3.0371651649475098, 1.0543440580368042, 0.8197457194328308, -2.36199639402912e-06, 0.011418584734201431, 1.0206693410873413, -0.0007269993075169623, -4.040251224068925e-05, -0.16571319103240967, 0.7099551558494568, -1.691154949412521e-07, 0.7672577500343323, 2.84425950050354, -0.1699785441160202, 0.5478031039237976, 1.270094394683838, -0.049078527837991714, 1.787689447402954, 1.2221415042877197, -0.0003047716454602778, 1.4466118812561035, 2.5919318199157715, -0.0018930085934698582, 2.570922374725342, -0.16982921957969666, -7.729497883701697e-05, -0.0028269982431083918, -0.01089557632803917, 1.5182596445083618, -0.10967210680246353, -0.1423741728067398, -0.00021335406927391887, 0.03419756516814232, -0.0002725927042774856, -0.04576464742422104, 1.7561050653457642, 1.9268461465835571, -0.09034982323646545, -0.06738616526126862, 1.4892127513885498, -0.015614626929163933, 3.579962730407715, 3.9845807552337646, -0.10288305580615997, -0.00032304643536917865, -0.14756697416305542, 4.010584354400635, 3.355492353439331, 3.5266318321228027, 3.3292324542999268, -0.09777665883302689, 0.6609025597572327, 3.239607334136963, 0.6347183585166931, 2.624354839324951, -0.0038583464920520782, -0.0003977614687755704, 6.0170135498046875, 2.2920050621032715, 1.7888706922531128, 4.399281978607178, 6.673903942108154, 8.627710342407227, -3.9593080752678455e-16, 1.0628507137298584, 3.1294851303100586, -1.4227681255363223e-09, -2.879557747803574e-08, -0.15177635848522186, 5.697741985321045, 2.8599460124969482, 6.503233432769775, 0.7469213008880615, 3.2656967639923096, 4.614760398864746, 1.3203067779541016, -0.0023600151762366295, 0.27721282839775085, -0.0, 0.848883867263794, -4.016003085638431e-09, -0.018229052424430847, 2.11311674118042, 1.0839744806289673, -1.0214011580966798e-10, 1.5105774402618408, 2.2460691928863525, 0.4447811543941498, 2.0361251831054688], "dtype": "float32"}}, {"a": [{"__nd__": [[1.1435546875, -3.408203125, -0.419677734375, -0.59765625, -2.44140625, -0.1058349609375, -1.64453125, -0.179443359375, -1.7451171875, 2.08203125, 4.421875, -1.1640625, 0.0625, 0.0872802734375, 0.87109375, -0.36669921875, -4.484375, 0.6826171875, -0.285400390625, -2.1796875, -0.432373046875, -1.0498046875, 0.437744140625, -0.67626953125, -0.60107421875, 0.1728515625, 0.449462890625, -0.01126861572265625, 2.3046875, 0.44287109375, -1.5859375, 3.41796875, 0.19580078125, 3.09375, -2.17578125, 1.0849609375, 3.40625, 2.50390625, 0.1407470703125, -1.2177734375, -1.25390625, -0.415283203125, -1.9912109375, -1.541015625, -2.0390625, -2.05078125, 0.43017578125, -3.587890625, 0.281494140625, 1.33203125, -1.9365234375, -1.873046875, -0.0023956298828125, -0.1329345703125, 1.8564453125, -4.0390625, -0.1859130859375, 2.765625, 0.123779296875, 3.5, 4.328125, -1.1396484375, 2.962890625, 1.0009765625, -0.06512451171875, -0.59912109375, -0.478515625, -1.8623046875, -0.375, 1.84765625, 2.4921875, -0.79736328125, 1.03125, 0.92724609375, -4.76171875, 1.3994140625, -2.501953125, -2.330078125, 1.6533203125, 3.330078125, -2.2578125, -1.8076171875, -1.939453125, -1.09765625, -0.80859375, -1.7060546875, -3.310546875, 3.50390625, -1.05078125, -1.576171875, 1.7529296875, -0.78515625, 2.123046875, 1.521484375, 1.9267578125, -0.8515625, 1.224609375, -3.16015625, -2.802734375, 0.08673095703125, -0.662109375, 2.6484375, -0.5908203125, -3.58984375, 0.080810546875, -0.63232421875, 0.339599609375, -2.375, -1.5947265625, 0.08612060546875, 1.447265625, 0.7880859375, 1.4736328125, -2.9453125, -0.460205078125, 0.40869140625, 1.90625, 0.505859375, -0.5048828125, -0.260498046875, -0.197998046875, -2.876953125, 0.042724609375, 1.6005859375, 1.412109375, -3.740234375, -0.01153564453125, -3.01953125, 2.181640625, 1.072265625, 1.4111328125, -1.3408203125, -3.880859375, -2.095703125, -0.71240234375, -3.0078125, -2.021484375, 3.50390625, -2.64453125, 1.091796875, -3.734375, 0.49951171875, 1.671875, 0.89013671875, 2.408203125, 3.591796875, -2.76171875, 0.53125, 1.0458984375, 0.85888671875, 3.1953125, 0.57373046875, 2.431640625, 2.34375, 0.330078125, 1.134765625, -1.48828125, 0.6748046875, 0.429931640625, -0.244873046875, -0.8681640625, 2.0, -0.1551513671875, 4.625, 2.689453125, 1.595703125, -1.08203125, 1.89453125, 0.367919921875, 0.26416015625, -2.099609375, 1.255859375, 0.59423828125, -0.826171875, -1.2412109375, -0.4404296875, -2.53515625, -0.99169921875, 2.869140625, -1.18359375, 4.734375, -2.560546875, 2.294921875, 2.28515625, 2.439453125, 1.2607421875, -3.091796875, 2.40625, 2.251953125, 0.197021484375, -4.6015625, 2.2578125, -1.740234375, -0.441162109375, 2.587890625, 2.76171875, 0.875, -1.0068359375, 3.515625, 2.412109375, -0.2047119140625, -1.8251953125, 0.66943359375, -2.615234375, 3.630859375, 1.1025390625, -0.62939453125, 0.8994140625, -3.265625, -0.061187744140625, -1.2109375, -3.1015625, -1.20703125, 0.7568359375, -2.810546875, -0.340576171875, -2.869140625, 0.448974609375, -3.486328125, 1.47265625, 1.3701171875, 1.705078125, 3.66015625, 0.377685546875, 0.26806640625, 1.240234375, -0.9091796875, -3.78515625, -2.4375, -2.431640625, 0.4052734375, -1.2666015625, -0.77880859375, 1.142578125, 0.38916015625, 4.0234375, -0.08984375, -1.8798828125, 2.5546875, -0.82080078125, -0.1314697265625, 2.0078125, 0.37646484375, 1.41796875, -1.22265625, -0.56640625, 1.31640625, -1.03515625, -0.7470703125, -0.0283660888671875, 3.25390625, 3.0234375, 0.53857421875, 0.2193603515625, -1.3115234375, 2.74609375, 0.92919921875, 2.001953125, -2.705078125, 1.736328125, 0.436767578125, 1.6513671875, 2.21875, 0.1690673828125, -0.5830078125, 1.5419921875, -2.26171875, 3.025390625, 0.85986328125, -2.056640625, 2.76171875, 2.1171875, 3.265625, 2.20703125, 1.1953125, 0.13330078125, -1.90625, 1.7314453125, 1.4130859375, -0.6171875, 1.419921875, -1.5546875, 2.357421875, 0.54150390625, 0.72705078125, -1.224609375, 2.130859375, 0.55859375, -0.919921875, 1.0029296875, -0.19140625, 2.310546875, -2.82421875, -2.083984375, 4.265625, -0.78125, 1.19921875, -0.60205078125, 0.63037109375, 5.44921875, -3.76171875, 1.2880859375, 0.306884765625, 0.260009765625, -1.3896484375, -1.7177734375, 2.91796875, 1.201171875, -2.21875, 0.18896484375, -2.49609375, 5.93359375, 0.9326171875, -3.439453125, 2.02734375, 1.4248046875, 0.79345703125, 0.525390625, -0.94189453125, -0.219970703125, -3.892578125, 2.775390625, 4.1796875, 1.16015625, 1.3896484375, -2.58203125, -0.705078125, -2.693359375, -2.4765625, 1.2275390625, 2.98828125, 0.00983428955078125, 5.1171875], [1.794921875, -4.390625, -0.67529296875, 2.751953125, 0.44970703125, 2.216796875, -3.25, 2.69140625, -2.25, 0.81201171875, 4.37890625, 0.272705078125, -0.3447265625, 1.6533203125, -1.55078125, -1.7919921875, 2.078125, -2.880859375, -2.3828125, 3.64453125, -2.40234375, 0.9990234375, 2.330078125, -0.01204681396484375, -2.029296875, 1.755859375, -1.775390625, 4.34765625, 4.55078125, -1.9658203125, 1.2646484375, -1.3193359375, 0.85546875, -0.416748046875, -1.87890625, 1.125, -3.23046875, 0.7099609375, -0.2294921875, -1.4912109375, 1.8935546875, 0.1600341796875, -0.6220703125, -0.25927734375, -0.379150390625, 1.943359375, -0.5146484375, -5.75390625, -6.17578125, -2.0546875, -2.466796875, -0.4912109375, -2.66796875, -1.5634765625, 2.708984375, -2.626953125, 1.7607421875, -0.1875, 2.67578125, 1.8662109375, 0.1917724609375, 2.24609375, 0.53271484375, -2.16796875, 1.0908203125, 1.111328125, -0.125732421875, 0.6328125, -0.7529296875, 0.77734375, -0.42431640625, -2.96875, -0.405517578125, 3.33203125, 1.7939453125, 3.15234375, 0.1400146484375, -0.67919921875, -2.0, 0.44140625, 2.1875, 1.77734375, 1.625, -0.88427734375, 0.35302734375, -1.41796875, 1.794921875, 1.337890625, -1.14453125, 0.386474609375, -4.4140625, 1.5419921875, -1.515625, -0.61376953125, 1.3359375, 1.478515625, -1.2275390625, 3.3125, 4.171875, 3.02734375, 2.75, 0.233642578125, 2.912109375, -1.642578125, -2.111328125, 0.443359375, 0.6318359375, -1.615234375, 3.689453125, -2.818359375, -4.03515625, -0.6962890625, 3.3984375, 1.6103515625, 0.8037109375, 0.3818359375, 1.232421875, 0.7451171875, 0.30078125, -0.1492919921875, -3.123046875, 2.0390625, -0.76953125, 0.84716796875, 1.076171875, 2.166015625, -1.4970703125, 1.73828125, 1.79296875, -0.1334228515625, -0.0185394287109375, 0.381591796875, -1.9716796875, -0.833984375, -2.19921875, 2.458984375, -1.4912109375, -2.076171875, 0.381591796875, 1.1484375, -0.67138671875, -1.8408203125, 0.90625, 1.76953125, -3.21875, 1.8779296875, -3.49609375, 1.37890625, -0.82421875, -0.44970703125, 1.4970703125, 2.947265625, 4.23828125, 2.80078125, 1.4345703125, -2.62109375, -0.7548828125, 0.32177734375, -1.1435546875, 1.2255859375, -0.4326171875, 3.642578125, -0.3134765625, -2.484375, -0.9736328125, -1.3876953125, 1.4697265625, 2.853515625, 2.095703125, 2.103515625, -1.3193359375, -2.15625, -1.5478515625, 0.1282958984375, -0.62548828125, 0.6396484375, 2.12109375, 0.2388916015625, 3.490234375, -1.4677734375, 1.35546875, -0.031341552734375, 2.10546875, 0.0231781005859375, -0.97412109375, -0.1612548828125, -1.673828125, -2.111328125, 1.6884765625, -1.125, 2.509765625, 5.109375, -0.6748046875, 1.822265625, -0.0859375, -3.71875, -0.12030029296875, 3.025390625, -2.369140625, -0.270263671875, -0.296142578125, -2.84765625, 0.68310546875, -0.2462158203125, 1.2060546875, 0.1470947265625, -2.794921875, 0.304443359375, 1.291015625, -0.46728515625, 1.15625, 1.123046875, -4.2890625, 1.271484375, -4.046875, -2.58203125, -0.2744140625, -1.9755859375, -1.6396484375, 2.1953125, -2.181640625, 2.3359375, 3.619140625, 2.767578125, -2.259765625, -3.126953125, 0.067626953125, 4.19140625, -0.48876953125, -1.1904296875, -0.310302734375, 1.443359375, 2.32421875, -0.0723876953125, -0.329833984375, 0.953125, 0.0017213821411132812, -0.890625, -3.607421875, 0.1732177734375, -1.27734375, -0.53857421875, -0.65966796875, -1.8408203125, 1.044921875, 2.07421875, 0.9921875, -1.375, 0.276611328125, 1.333984375, -0.272216796875, 1.33203125, -2.8984375, -0.50048828125, 2.8125, 1.93359375, -0.314453125, 0.18701171875, 4.2578125, 2.13671875, -1.4541015625, 0.576171875, -0.9560546875, -1.2216796875, -2.45703125, 1.3955078125, 4.41796875, 1.6943359375, -2.638671875, 1.0400390625, -2.94140625, 0.433837890625, 2.095703125, 2.951171875, 0.2384033203125, 0.6416015625, 0.023162841796875, -0.342041015625, 3.126953125, -1.904296875, -3.185546875, 3.5078125, 4.53515625, -3.1875, 1.8779296875, 1.6171875, -0.8369140625, -1.9072265625, 0.320068359375, -0.66259765625, 1.642578125, -1.2607421875, 2.365234375, -1.2568359375, 3.265625, 1.783203125, 2.283203125, -0.765625, -0.452392578125, -0.1773681640625, 2.376953125, -1.31640625, -1.5615234375, -0.444580078125, 0.71923828125, -0.1531982421875, -0.121826171875, -2.62109375, -1.2314453125, -2.138671875, 2.099609375, -0.07940673828125, 1.3603515625, -2.626953125, -0.0723876953125, 0.74951171875, -2.109375, -1.291015625, -1.033203125, 0.294677734375, 0.5625, -2.83984375, 0.65966796875, 0.415771484375, 0.58056640625, -1.7451171875, -0.14697265625, 2.552734375, 2.291015625, -1.232421875, 3.197265625, -0.84765625, -1.80078125], [2.23828125, 0.64453125, 4.57421875, -0.470703125, 0.2607421875, 1.462890625, 0.52197265625, -0.8095703125, -0.0002982616424560547, -0.048370361328125, -0.04412841796875, -0.06439208984375, -3.080078125, 1.779296875, -1.2431640625, -0.4951171875, -1.583984375, -0.417724609375, 1.951171875, 2.529296875, 0.64599609375, 0.32470703125, -1.0419921875, -0.56005859375, 0.46826171875, -0.311279296875, -1.9755859375, 0.126220703125, -2.08984375, 4.14453125, 0.61376953125, -1.1318359375, 0.9326171875, -1.4931640625, -0.5048828125, 3.103515625, 2.578125, -2.3671875, -2.232421875, -1.025390625, 4.5078125, 0.125, 0.51708984375, -1.017578125, 2.65625, 1.9296875, -3.45703125, -1.6943359375, -3.6640625, -2.322265625, -1.630859375, -2.673828125, -1.2763671875, 1.3818359375, 2.25390625, 0.224853515625, 1.4541015625, -2.71484375, 2.947265625, 4.58984375, -0.291259765625, 1.142578125, 0.7119140625, -4.34375, 2.037109375, 0.955078125, -3.244140625, -0.8427734375, 2.31640625, 1.1474609375, -2.759765625, -1.150390625, -1.3837890625, 0.30078125, -1.41796875, 0.090576171875, -0.354248046875, -2.974609375, 0.8076171875, 0.5078125, -1.3818359375, 1.474609375, 4.578125, -3.18359375, 0.42822265625, 1.666015625, 0.05419921875, -1.173828125, 4.39453125, -1.0185546875, -0.273681640625, -1.322265625, 2.423828125, -4.30078125, -1.806640625, -0.927734375, 0.619140625, 0.6806640625, 1.974609375, -0.258544921875, -1.484375, -0.1378173828125, -1.7548828125, -1.083984375, -0.9697265625, 0.6025390625, 1.658203125, -2.216796875, -1.2021484375, -0.059173583984375, 0.018157958984375, -0.51904296875, 1.1455078125, -0.445068359375, -3.501953125, 2.623046875, -2.51171875, 3.26171875, 0.515625, 0.640625, -2.943359375, 0.53466796875, -0.41943359375, -1.685546875, 1.08984375, 1.490234375, 2.3671875, -2.44921875, -1.5751953125, 2.765625, 1.044921875, 1.615234375, -1.041015625, 0.708984375, 1.6328125, -2.421875, -1.3388671875, 0.51220703125, 2.755859375, -0.82275390625, 1.3515625, -1.6328125, -2.328125, 1.2783203125, -1.990234375, 0.1163330078125, -1.0673828125, 1.0595703125, -2.458984375, -2.734375, -2.767578125, -0.36328125, 2.560546875, 1.6552734375, 0.41015625, -0.251953125, 2.421875, -0.177978515625, 2.076171875, 2.6953125, 1.35546875, 0.79736328125, 0.78076171875, -0.1500244140625, 0.56494140625, -1.724609375, -1.654296875, 3.091796875, -0.2364501953125, -3.333984375, 2.669921875, 2.671875, 0.8857421875, -2.431640625, 3.818359375, 2.1328125, 2.427734375, -0.487060546875, 0.8212890625, -0.54248046875, -0.46728515625, -2.30078125, -1.5458984375, -1.0068359375, 0.06964111328125, 1.9345703125, 1.0849609375, -3.556640625, 2.029296875, -2.98046875, 0.9423828125, -0.40234375, 0.1129150390625, -0.96142578125, -2.099609375, 3.173828125, -4.83984375, -1.775390625, 2.57421875, -0.0570068359375, -0.0010328292846679688, -1.7216796875, -2.267578125, 1.1328125, -0.374755859375, 0.6279296875, 1.4736328125, -3.43359375, -2.09375, -1.3701171875, 0.397705078125, 1.328125, -2.09765625, -0.48095703125, -0.268310546875, -0.271484375, 3.037109375, -1.5205078125, 0.2666015625, -4.64453125, -0.28076171875, 0.1778564453125, 1.3408203125, -1.0703125, -1.1630859375, -2.44921875, -1.0498046875, 4.66015625, -3.451171875, -3.435546875, 0.13720703125, -1.927734375, 2.4140625, -3.04296875, 3.220703125, -0.9326171875, -3.341796875, -4.4609375, -1.8447265625, -1.33984375, 0.6689453125, -2.615234375, 6.015625, 1.6220703125, -0.414794921875, -1.1494140625, 5.32421875, -1.5615234375, 1.470703125, 2.9140625, -1.8837890625, 1.134765625, -1.015625, 2.75, 2.8828125, 1.07421875, 0.312255859375, -2.44140625, 4.8828125, 3.017578125, 1.439453125, 2.994140625, 0.041473388671875, -2.23046875, 1.888671875, -1.921875, -1.0537109375, 1.51171875, -3.134765625, 0.06475830078125, -0.58251953125, -2.31640625, -2.41796875, -0.430419921875, -0.58447265625, 0.32568359375, -2.98828125, 0.17333984375, -2.5234375, -1.6162109375, 1.9404296875, -1.3583984375, -0.9873046875, -1.626953125, -0.016632080078125, 1.673828125, -0.697265625, 1.33203125, -2.609375, 0.9248046875, -0.218505859375, 0.147216796875, -1.849609375, -1.232421875, 0.6171875, -2.36328125, 1.541015625, -0.8779296875, 0.90625, -2.2578125, -3.248046875, 1.28125, 1.373046875, -0.08551025390625, 1.3095703125, -0.0254364013671875, 1.3203125, 0.29296875, -1.9345703125, 2.2109375, -1.923828125, 4.51953125, -0.293212890625, -1.853515625, -0.425048828125, 0.133544921875, 0.281982421875, 1.12890625, -1.1806640625, -4.1015625, -1.6884765625, 0.89111328125, 2.259765625, -3.955078125, 0.234130859375, 2.263671875, -2.779296875, 3.451171875, -1.5048828125, -0.1961669921875, -0.95263671875, 3.611328125, 1.83984375], [0.31298828125, -0.6728515625, -3.3359375, -0.54736328125, -1.1767578125, -0.2130126953125, -3.80078125, 2.576171875, 1.8642578125, 3.248046875, -1.138671875, -0.035064697265625, -2.421875, 1.81640625, -0.406005859375, 3.400390625, 1.5146484375, 1.58984375, 1.146484375, -2.91796875, 1.5078125, -3.181640625, 0.65185546875, 1.234375, -2.275390625, -1.220703125, -0.303955078125, 0.5859375, 0.63525390625, 0.11297607421875, -1.712890625, 0.603515625, 5.0390625, -1.1171875, 3.23828125, -2.361328125, -1.25390625, 2.69921875, 1.7353515625, 3.26171875, 1.2353515625, -1.201171875, 0.1732177734375, -0.873046875, 2.12109375, 1.2529296875, 2.525390625, -1.365234375, -0.48974609375, -0.84521484375, -0.720703125, -0.303955078125, 2.580078125, 0.304443359375, 2.3828125, 0.430908203125, -3.564453125, -2.7734375, -0.34619140625, 2.341796875, 0.0283966064453125, -2.951171875, 0.25927734375, -0.491943359375, 1.1845703125, 1.67578125, 0.57958984375, -1.2841796875, -2.0703125, -0.7060546875, -0.1392822265625, 0.73583984375, -0.9072265625, -1.2666015625, 0.99462890625, 1.6494140625, 0.56982421875, -2.05078125, -0.61181640625, 4.34765625, -2.140625, -0.71923828125, 1.8408203125, -1.91796875, -0.62060546875, 2.58203125, 1.9658203125, -3.591796875, 1.216796875, -0.63232421875, -2.76953125, 2.138671875, -1.0380859375, -0.0743408203125, -1.703125, 1.6005859375, 1.8291015625, 1.611328125, 4.83203125, 1.287109375, -0.56884765625, 1.1591796875, 0.7958984375, 1.583984375, -4.58203125, 1.888671875, 4.234375, 1.9755859375, -0.9931640625, 4.578125, 1.85546875, 4.4609375, 2.359375, -0.191162109375, -2.837890625, 0.67041015625, 1.548828125, -0.294677734375, -2.13671875, 2.177734375, -0.982421875, 3.326171875, -0.583984375, 1.275390625, -0.120849609375, 1.328125, 1.607421875, -3.7109375, -0.8583984375, -0.63232421875, -0.69384765625, 0.33447265625, 2.75, -1.55859375, 1.5546875, -0.33447265625, -0.94580078125, -2.5390625, -1.11328125, -1.767578125, 1.31640625, 3.453125, 1.4853515625, 1.8720703125, -1.2314453125, 1.7841796875, -1.2890625, -1.453125, 0.90087890625, -0.90380859375, 1.623046875, 0.6650390625, -0.164306640625, -0.78466796875, 0.7548828125, 0.892578125, 0.5341796875, 2.615234375, -0.767578125, -2.3671875, -2.24609375, -1.564453125, -1.447265625, 0.097900390625, 3.64453125, -1.638671875, -1.3046875, -0.68505859375, -1.58984375, -1.1484375, 1.7666015625, 2.90625, -1.037109375, 0.2269287109375, 2.18359375, 1.0, -0.0025463104248046875, 1.0693359375, 0.1572265625, -0.974609375, 1.365234375, -0.8955078125, -0.62939453125, 0.9443359375, -1.9296875, 1.73828125, -1.087890625, -1.5888671875, 1.4072265625, 1.626953125, 2.20703125, 2.953125, -1.6298828125, -0.1849365234375, -1.802734375, -1.158203125, 1.9892578125, 0.8994140625, 0.806640625, 1.095703125, -0.5322265625, 2.748046875, -1.6767578125, 1.576171875, -0.88671875, -1.5263671875, 4.1953125, -0.47607421875, 1.59375, -1.2958984375, -1.2021484375, 2.572265625, 1.0380859375, 0.36474609375, 1.1357421875, 1.4306640625, 1.8837890625, 1.1669921875, 1.5595703125, 0.826171875, 0.48583984375, 2.388671875, -4.05859375, 1.8154296875, 0.25, 0.5546875, -0.306640625, -1.1875, 1.3623046875, -2.15625, -1.76953125, -1.79296875, 3.3359375, 0.1165771484375, 0.067626953125, -2.6484375, 0.2392578125, 1.73046875, -2.052734375, 0.1929931640625, 0.45263671875, 1.2431640625, 1.947265625, 3.884765625, 0.0258331298828125, 0.081298828125, -1.220703125, 1.892578125, -0.052490234375, 0.92578125, 1.37109375, -0.27685546875, -0.34130859375, 0.32080078125, 0.286865234375, 0.12042236328125, -0.28515625, -0.07958984375, 2.369140625, -1.1650390625, 0.1865234375, 0.035797119140625, 2.091796875, -0.98583984375, -0.435302734375, -0.95751953125, 1.5849609375, 1.267578125, 0.873046875, 0.312255859375, 4.390625, 0.6337890625, -3.685546875, -2.787109375, -2.6640625, -0.810546875, -2.9140625, 0.0302886962890625, -2.763671875, -0.10101318359375, -1.224609375, 0.97802734375, 0.2437744140625, -1.8740234375, 3.484375, -2.19140625, -1.0380859375, -2.71875, 1.884765625, 1.8193359375, -0.1944580078125, -4.77734375, 1.494140625, 3.455078125, -1.72265625, 0.900390625, 1.7314453125, 0.06494140625, -2.30078125, -0.406494140625, -1.2744140625, 1.0517578125, -3.435546875, 0.99365234375, 0.11712646484375, -3.0703125, -1.990234375, 1.0341796875, 2.4765625, 1.654296875, 5.66015625, 0.339599609375, -2.369140625, 0.0269775390625, -1.4521484375, -4.734375, 2.029296875, -1.9619140625, -1.8544921875, 3.505859375, -1.302734375, -1.7041015625, 1.619140625, 3.583984375, -3.720703125, -2.1875, -1.26953125, 1.5029296875, 0.45361328125, 2.1953125, -2.9140625, -0.2724609375, 2.83984375], [-3.654296875, 2.0859375, -1.6865234375, 0.51025390625, -2.123046875, 1.2021484375, 1.4990234375, -0.341552734375, 3.21484375, -2.017578125, 1.900390625, -2.294921875, 1.861328125, 1.625, -1.4560546875, 3.08984375, -0.11260986328125, -2.73828125, -0.6591796875, 1.7744140625, 0.0736083984375, 1.8037109375, 2.80078125, 1.6328125, 1.0732421875, 0.422119140625, 0.943359375, 2.8828125, -1.166015625, -0.36572265625, 1.5888671875, -1.9296875, 1.994140625, 0.07452392578125, 0.99755859375, -1.181640625, -0.390869140625, 2.408203125, -2.12109375, 2.0390625, 2.244140625, 0.2142333984375, -3.822265625, -2.158203125, 0.802734375, 2.125, -2.734375, 0.75390625, 0.50048828125, -2.955078125, 0.378173828125, -1.4697265625, -2.947265625, 3.56640625, -1.4765625, -2.810546875, -2.787109375, 3.84765625, 1.9736328125, 1.1083984375, -1.6298828125, 0.779296875, 0.166015625, 1.4775390625, -2.869140625, 1.45703125, 1.96875, -1.7607421875, -1.845703125, -0.222412109375, -2.654296875, -0.5625, -0.5751953125, 2.734375, -1.9873046875, -0.63134765625, 2.625, -3.408203125, 2.052734375, -0.9169921875, 1.1611328125, -0.020843505859375, 1.267578125, -6.640625, -1.421875, -4.34375, -0.5947265625, -1.3125, 0.1280517578125, 3.337890625, 1.0595703125, 0.81640625, 1.2861328125, -0.46337890625, 3.03515625, 1.3544921875, 0.11956787109375, 2.4765625, -0.6953125, -0.98046875, -1.708984375, -2.78125, -2.384765625, 2.37109375, -0.8359375, -0.0550537109375, 1.4638671875, -0.293701171875, -1.9453125, -1.677734375, -0.896484375, 2.037109375, -1.0595703125, -1.1494140625, 1.033203125, 0.036285400390625, 0.10693359375, 0.43798828125, 3.376953125, -0.68701171875, -0.9404296875, 3.572265625, -1.8212890625, -3.46875, 1.697265625, -0.97802734375, -0.057708740234375, 0.153076171875, -2.318359375, 0.9775390625, -2.431640625, 1.650390625, 2.248046875, -1.4052734375, 2.154296875, 0.333251953125, 3.64453125, 0.0300140380859375, -3.609375, -3.658203125, 1.7529296875, -0.94384765625, 0.7451171875, -2.25, -1.181640625, -4.46875, 0.7392578125, 1.5380859375, -1.5810546875, 0.958984375, -1.4970703125, 0.67578125, -0.966796875, -1.8154296875, 0.859375, -3.8125, 3.56640625, -3.248046875, -4.046875, 0.457763671875, -2.173828125, 1.1220703125, -2.892578125, 2.873046875, 2.453125, -2.19921875, 3.87890625, -0.96142578125, -0.078369140625, -1.3935546875, 0.462890625, 0.69482421875, -0.336181640625, 0.52197265625, 0.1815185546875, 0.87841796875, -2.703125, -0.9375, -2.40234375, -0.188232421875, -0.215087890625, -0.47119140625, -1.451171875, 3.568359375, -2.015625, 0.9599609375, -0.98291015625, 2.865234375, 2.330078125, 0.775390625, 1.2783203125, -1.76953125, -2.12890625, 1.46875, -0.041015625, -3.169921875, -0.41748046875, -2.685546875, -2.953125, -0.2022705078125, 0.03887939453125, -0.72216796875, -1.0849609375, -2.1875, -0.50048828125, -0.35009765625, -1.04296875, 4.609375, -0.67041015625, 0.1358642578125, -1.646484375, -0.447509765625, 0.0113067626953125, 1.037109375, -6.1171875, 2.013671875, 0.99462890625, 0.406982421875, -2.140625, -3.623046875, -2.33984375, 1.2939453125, 1.22265625, -0.08428955078125, 1.3642578125, -1.9189453125, -2.24609375, 2.44140625, 2.15234375, 1.50390625, 2.14453125, -0.1175537109375, -4.62109375, -0.56591796875, 2.158203125, -0.53125, 1.3564453125, 1.19921875, -1.8271484375, 0.9990234375, 0.14501953125, -2.904296875, 2.1015625, -2.158203125, 0.8974609375, 0.97705078125, -1.5419921875, -1.86328125, 0.85009765625, 0.443603515625, 2.388671875, 0.328369140625, 1.0029296875, 0.328857421875, 1.8583984375, -1.9990234375, -3.7578125, 0.96728515625, -0.417236328125, -1.2802734375, 3.306640625, -2.8359375, -0.58447265625, 1.7841796875, -0.413330078125, -0.86181640625, 0.607421875, -0.7666015625, -3.4140625, -2.646484375, 1.32421875, 1.296875, -2.369140625, 0.49853515625, 1.2421875, 1.3037109375, 2.84765625, 1.9228515625, -0.14306640625, -0.49951171875, 1.3330078125, -2.283203125, -3.240234375, 1.587890625, 2.2265625, 1.18359375, 0.61279296875, -3.1328125, -2.8046875, -1.2333984375, 3.28125, -0.5791015625, 0.34033203125, -1.5693359375, 2.640625, -3.716796875, 3.259765625, 0.02215576171875, -5.21484375, 0.17822265625, 1.1650390625, 1.1572265625, 0.947265625, -0.57666015625, -0.5390625, 0.2481689453125, -1.1796875, 3.427734375, -2.328125, 0.302001953125, -0.5439453125, -2.85546875, 2.833984375, 0.5517578125, -1.333984375, -1.6884765625, -0.50634765625, 3.142578125, 0.461181640625, -1.787109375, 0.29833984375, -1.36328125, 1.078125, 0.91357421875, 1.5322265625, 0.69287109375, 0.267578125, -2.490234375, 0.54248046875, 2.2734375, -2.78125, 1.9013671875, -1.119140625], [-1.173828125, -1.1318359375, 1.4833984375, 0.16162109375, 0.458984375, -1.892578125, 3.216796875, -1.794921875, 2.68359375, 2.568359375, 1.5390625, -2.369140625, 0.219482421875, -1.3798828125, 3.19140625, 0.6572265625, 0.98095703125, 0.5703125, 0.65576171875, 2.193359375, 4.37109375, 1.6640625, 2.62109375, 1.5078125, 1.025390625, -2.126953125, -0.59814453125, -1.021484375, 1.349609375, -5.6171875, -4.12890625, -0.7998046875, 0.98291015625, -0.50830078125, 0.01396942138671875, 1.630859375, 3.04296875, 3.53515625, 1.3857421875, 2.3984375, -0.4599609375, -1.458984375, 1.5, 3.39453125, 1.392578125, -3.423828125, 2.435546875, 4.12109375, 1.8212890625, -0.1402587890625, -1.279296875, -1.8115234375, 3.29296875, -0.14453125, -0.054718017578125, 0.0633544921875, 0.1507568359375, 1.1416015625, -0.35400390625, 1.0986328125, -0.6240234375, -1.08203125, -3.02734375, 0.66552734375, 2.9765625, 1.63671875, 2.583984375, -0.3134765625, -1.0673828125, -1.087890625, -1.76171875, 1.998046875, 2.875, -0.958984375, 1.267578125, -2.34375, 1.0263671875, 1.94921875, -1.2294921875, 2.8359375, -0.466552734375, 0.90869140625, 3.640625, -2.6640625, 1.3134765625, -0.1322021484375, 3.2890625, 0.30322265625, 0.02783203125, 1.3369140625, 3.8203125, -0.80615234375, -1.5078125, -1.193359375, 2.216796875, -3.205078125, -1.9658203125, -0.9736328125, -1.529296875, -0.09796142578125, -1.1728515625, -3.744140625, 4.703125, 2.263671875, -0.1339111328125, -0.89599609375, -1.16796875, -3.4453125, -2.025390625, 0.55810546875, 0.73095703125, 2.98828125, 0.173095703125, -1.041015625, 2.015625, 1.3671875, -2.34765625, -1.22265625, 1.9326171875, -1.330078125, 0.355712890625, -1.787109375, 2.25, 4.2734375, -3.888671875, -1.892578125, 1.5751953125, 0.75244140625, 0.042724609375, -2.41015625, -3.328125, -0.62060546875, 2.13671875, 2.171875, 4.953125, -3.25, 0.63427734375, 0.77197265625, 4.08984375, -1.111328125, 3.013671875, -2.896484375, 2.404296875, -1.6142578125, -1.84765625, 0.94921875, -0.71337890625, 0.70751953125, -0.08221435546875, -1.4697265625, 1.7646484375, -0.384033203125, -1.263671875, -1.3115234375, -3.716796875, -2.5546875, 0.74072265625, 0.5166015625, -0.185546875, 2.736328125, 0.08837890625, 0.11627197265625, 1.1767578125, 0.79443359375, 2.45703125, -0.95068359375, 0.313720703125, 0.1435546875, -0.55908203125, -0.0228424072265625, 2.009765625, 3.177734375, 0.2386474609375, 0.2108154296875, 0.70654296875, 2.45703125, 1.4306640625, 3.51171875, 3.97265625, 2.53515625, -0.900390625, -0.4755859375, -0.52099609375, -0.6796875, 2.244140625, -1.01953125, 1.2841796875, -0.5517578125, -1.4619140625, 2.71484375, -2.5703125, 2.982421875, -2.505859375, 1.2763671875, 0.81103515625, 0.80322265625, 0.09588623046875, -1.0703125, -0.250244140625, 2.9921875, 2.201171875, 4.7578125, 2.576171875, 1.5361328125, 0.7568359375, 0.0311279296875, 3.076171875, -0.32666015625, -0.23486328125, -0.9404296875, 1.408203125, 6.75, -0.31591796875, -3.091796875, -1.1689453125, 1.98046875, -0.49462890625, -0.74365234375, 1.6162109375, 0.040985107421875, -1.5419921875, 1.0810546875, -1.8037109375, -0.050048828125, 3.88671875, -4.30859375, 3.359375, -3.548828125, 0.62890625, -4.0703125, -0.43212890625, -1.3447265625, -1.126953125, -1.5771484375, 3.134765625, 2.09375, -2.97265625, -3.404296875, -0.2093505859375, -2.708984375, 4.01171875, 2.93359375, 1.3447265625, -3.375, -1.3115234375, 2.080078125, -1.7763671875, 1.556640625, 0.19970703125, 1.98828125, -0.0099029541015625, -0.03192138671875, -2.529296875, -0.1796875, -1.65625, 1.513671875, -2.771484375, 1.8037109375, 2.099609375, 1.375, -2.96484375, -2.47265625, 0.431396484375, 0.25537109375, -0.7236328125, -0.38525390625, -2.103515625, -2.001953125, -2.125, -1.1259765625, -0.2427978515625, -0.677734375, -1.845703125, 2.12109375, 0.40283203125, -1.99609375, 5.3671875, -5.109375, 1.2431640625, 0.67431640625, -1.1533203125, -2.068359375, -0.1654052734375, 2.125, 1.802734375, 0.9794921875, 0.42578125, 0.10186767578125, 1.876953125, 0.26953125, 0.489990234375, 2.755859375, -1.03125, 0.69091796875, -0.11199951171875, -1.6552734375, 2.91015625, -1.880859375, -0.708984375, -1.662109375, 0.908203125, 1.1943359375, 3.306640625, 3.326171875, 3.279296875, 1.103515625, 2.08203125, -0.442626953125, -1.9150390625, 2.5, -1.5224609375, 1.4140625, 3.072265625, 0.056304931640625, 2.916015625, 0.275634765625, -0.00719451904296875, 1.4853515625, 3.818359375, -2.296875, 0.71044921875, -1.259765625, 0.71826171875, 1.1884765625, -2.66015625, -1.953125, 1.9560546875, 1.0166015625, -1.111328125, 0.264892578125, 2.009765625, 1.935546875, -3.998046875], [-0.93798828125, 3.453125, -0.6220703125, -0.74462890625, 0.397705078125, 0.794921875, 0.58935546875, 0.8603515625, -1.4375, 0.94921875, 1.6904296875, 0.309814453125, -0.1865234375, -1.2666015625, 0.455322265625, -2.314453125, -0.822265625, -0.29150390625, -4.50390625, 0.483154296875, -3.0234375, -1.76171875, -1.783203125, -0.826171875, -0.53955078125, 1.0068359375, -2.275390625, -0.962890625, 0.623046875, -1.29296875, -1.5927734375, 0.29296875, 1.15234375, 0.8798828125, -2.75, 0.0833740234375, -0.253662109375, 3.1875, -0.2462158203125, -0.70166015625, 2.580078125, -1.5810546875, 2.3359375, 2.3359375, -0.3076171875, 0.424560546875, 1.1474609375, 4.27734375, 0.427490234375, -0.76953125, 1.37890625, 0.295654296875, -1.45703125, 0.1427001953125, 0.8740234375, 2.462890625, 1.2333984375, 1.181640625, -0.97265625, 2.111328125, -1.25, -1.3349609375, -0.8447265625, -1.509765625, 0.279296875, 1.4052734375, 0.2064208984375, -0.6923828125, 0.55322265625, -0.93212890625, 2.00390625, 2.962890625, -0.3984375, -0.51611328125, 2.0546875, 4.5703125, -0.306396484375, 2.7890625, 1.1728515625, -1.578125, 0.1436767578125, 0.8662109375, -0.658203125, 3.919921875, -4.5, -0.93310546875, -0.98828125, -1.44921875, -1.306640625, 0.115478515625, 1.0146484375, 0.189453125, 1.3046875, -0.291259765625, -1.994140625, -1.333984375, 2.43359375, -1.1181640625, -0.828125, 0.765625, 1.1435546875, -0.103515625, 1.6064453125, 2.8125, 3.220703125, 0.419189453125, -2.6171875, -2.048828125, -2.044921875, 0.619140625, 2.4140625, 0.94921875, 1.796875, 5.96875, -2.32421875, 2.625, -0.0016078948974609375, -1.1640625, -0.7265625, -0.5673828125, 3.876953125, 0.86962890625, 2.81640625, -2.5234375, 3.068359375, 0.98388671875, 1.94921875, -1.7294921875, 1.236328125, 0.364990234375, -0.30908203125, 0.1680908203125, 0.485595703125, 0.79248046875, -1.4931640625, 1.638671875, -2.373046875, -1.87109375, -3.2578125, -5.359375, 3.48828125, 3.498046875, -0.94384765625, -0.8408203125, -1.81640625, 0.07525634765625, -0.07208251953125, -2.208984375, -0.15380859375, -2.7734375, -0.49072265625, 2.25390625, 1.359375, 0.442626953125, -0.55810546875, 3.865234375, -0.45751953125, -4.30859375, -0.65966796875, -4.5703125, -0.10498046875, -0.4541015625, 2.884765625, 2.732421875, -0.611328125, -0.396240234375, 1.44921875, -2.068359375, -1.560546875, 2.890625, -2.0078125, 0.4404296875, 4.88671875, -0.73095703125, 3.869140625, -2.5703125, -1.6591796875, 2.810546875, 1.5166015625, 0.1612548828125, -0.06878662109375, 2.240234375, 3.8203125, -2.203125, -0.64794921875, 2.01953125, -0.462890625, 2.033203125, -1.724609375, 2.931640625, 1.6611328125, 0.7138671875, 0.90478515625, 3.408203125, -2.109375, 0.82763671875, -1.4716796875, -1.5185546875, 3.03125, 1.1083984375, 0.6494140625, 1.630859375, -0.81689453125, 2.646484375, -0.6318359375, 1.4794921875, 0.391845703125, 2.0625, -1.57421875, 1.7568359375, 1.8310546875, 0.45947265625, -0.7890625, -1.908203125, -0.68017578125, 0.86328125, -0.78857421875, -3.13671875, 1.4052734375, 1.1435546875, -0.64794921875, -2.45703125, -1.1171875, -0.91748046875, -1.2236328125, 0.391845703125, -3.89453125, 0.08880615234375, -0.6708984375, 1.69921875, -1.78515625, -0.2705078125, -0.71435546875, -4.03515625, 4.484375, -0.61767578125, 0.9052734375, 0.1578369140625, -0.361083984375, -0.1669921875, -3.904296875, 0.73046875, -1.0849609375, -0.495361328125, 2.244140625, 0.08099365234375, 3.9453125, 2.525390625, 1.314453125, 0.873046875, -1.875, 4.55078125, 3.576171875, -0.1337890625, 0.32373046875, -2.8203125, 2.59375, 0.759765625, 0.76708984375, -0.40966796875, -0.08538818359375, -0.423095703125, -0.701171875, -0.75, -0.9375, 3.771484375, 4.84375, -3.826171875, -0.038330078125, -1.216796875, -1.2265625, 1.470703125, -6.58203125, 1.599609375, -3.888671875, 0.319091796875, -0.94775390625, -1.4130859375, -0.182861328125, -2.236328125, 3.552734375, -3.50390625, -3.333984375, -0.422607421875, -0.07757568359375, 2.5546875, -0.3994140625, -2.44921875, -1.2451171875, -1.6689453125, -0.8408203125, 0.91357421875, -0.7353515625, -1.4111328125, 1.181640625, -1.994140625, -0.2169189453125, 0.5771484375, -1.4091796875, 2.056640625, -0.533203125, 3.501953125, -0.57080078125, -1.08203125, -2.42578125, 0.1024169921875, 1.216796875, -3.1328125, -3.853515625, -2.685546875, -0.1395263671875, -0.168212890625, -2.021484375, -1.021484375, -0.2491455078125, 0.69921875, -2.544921875, -3.78515625, -1.0703125, 3.357421875, -1.158203125, -0.9501953125, 0.15869140625, -3.53125, 1.2451171875, -1.794921875, -3.802734375, -0.55810546875, -0.525390625, 1.9052734375, 0.95458984375, -0.99853515625, 1.6943359375]], "dtype": "float16"}], "e": {"__nd__": [[0.9990234375, -0.0008783340454101562, -0.1416015625, -0.1644287109375, -0.0174407958984375, -0.0484619140625, -0.08245849609375, -0.07696533203125, -0.07073974609375, 2.04296875, 4.421875, -0.1424560546875, 0.032806396484375, 0.046661376953125, 0.7041015625, -0.130859375, -5.602836608886719e-06, 0.513671875, -0.11065673828125, -0.031646728515625, -0.1439208984375, -0.1544189453125, 0.29296875, -0.168701171875, -0.1646728515625, 0.0982666015625, 0.302734375, -0.005584716796875, 2.28125, 0.297119140625, -0.089599609375, 3.41796875, 0.11309814453125, 3.091796875, -0.03192138671875, 0.93408203125, 3.40625, 2.48828125, 0.0782470703125, -0.13623046875, -0.1318359375, -0.1407470703125, -0.046173095703125, -0.09521484375, -0.0421142578125, -0.041168212890625, 0.28662109375, -0.0004334449768066406, 0.1719970703125, 1.2099609375, -0.05108642578125, -0.05718994140625, -0.0011959075927734375, -0.0594482421875, 1.7978515625, -5.823373794555664e-05, -0.0792236328125, 2.7578125, 0.0679931640625, 3.5, 4.328125, -0.1451416015625, 2.958984375, 0.84228515625, -0.0308685302734375, -0.16455078125, -0.1512451171875, -0.058258056640625, -0.1326904296875, 1.7880859375, 2.4765625, -0.169677734375, 0.875, 0.76318359375, -1.0728836059570312e-06, 1.2861328125, -0.01500701904296875, -0.022705078125, 1.572265625, 3.328125, -0.0267333984375, -0.06390380859375, -0.050811767578125, -0.149658203125, -0.16943359375, -0.0751953125, -0.0012617111206054688, 3.50390625, -0.154296875, -0.0908203125, 1.6826171875, -0.1697998046875, 2.087890625, 1.423828125, 1.875, -0.1680908203125, 1.0888671875, -0.00214385986328125, -0.0066375732421875, 0.046356201171875, -0.168212890625, 2.638671875, -0.163818359375, -0.00043010711669921875, 0.042999267578125, -0.166748046875, 0.2149658203125, -0.0204620361328125, -0.0885009765625, 0.0460205078125, 1.33984375, 0.6181640625, 1.3701171875, -0.004322052001953125, -0.1485595703125, 0.26904296875, 1.8525390625, 0.350830078125, -0.1549072265625, -0.10345458984375, -0.08343505859375, -0.0053253173828125, 0.0220947265625, 1.5126953125, 1.30078125, -0.00022876262664794922, -0.00571441650390625, -0.00341796875, 2.150390625, 0.919921875, 1.298828125, -0.12091064453125, -0.00012242794036865234, -0.03765869140625, -0.169677734375, -0.00354766845703125, -0.0435791015625, 3.50390625, -0.010345458984375, 0.94140625, -0.0002346038818359375, 0.34521484375, 1.5927734375, 0.7236328125, 2.388671875, 3.591796875, -0.00746917724609375, 0.373046875, 0.89111328125, 0.69091796875, 3.193359375, 0.411376953125, 2.4140625, 2.322265625, 0.207763671875, 0.9892578125, -0.1019287109375, 0.50634765625, 0.286376953125, -0.0987548828125, -0.1673583984375, 1.955078125, -0.0679931640625, 4.625, 2.6796875, 1.5068359375, -0.1512451171875, 1.83984375, 0.23681640625, 0.1595458984375, -0.037353515625, 1.1240234375, 0.43017578125, -0.1689453125, -0.13330078125, -0.145263671875, -0.0137939453125, -0.1595458984375, 2.86328125, -0.1402587890625, 4.734375, -0.0129241943359375, 2.26953125, 2.259765625, 2.421875, 1.1298828125, -0.0026988983154296875, 2.38671875, 2.224609375, 0.1138916015625, -2.86102294921875e-06, 2.23046875, -0.0712890625, -0.1453857421875, 2.576171875, 2.75390625, 0.7080078125, -0.158203125, 3.515625, 2.392578125, -0.08575439453125, -0.06207275390625, 0.5009765625, -0.01119232177734375, 3.630859375, 0.95361328125, -0.16650390625, 0.7333984375, -0.001483917236328125, -0.0290985107421875, -0.136962890625, -0.002613067626953125, -0.137451171875, 0.5869140625, -0.0064849853515625, -0.1248779296875, -0.005451202392578125, 0.30224609375, -0.0006499290466308594, 1.369140625, 1.2529296875, 1.6298828125, 3.66015625, 0.244384765625, 0.162353515625, 1.1064453125, -0.165283203125, -0.00018799304962158203, -0.017608642578125, -0.0178680419921875, 0.266357421875, -0.1302490234375, -0.169921875, 0.99755859375, 0.25341796875, 4.0234375, -0.041717529296875, -0.0565185546875, 2.541015625, -0.1690673828125, -0.058868408203125, 1.962890625, 0.243408203125, 1.306640625, -0.1356201171875, -0.1617431640625, 1.1923828125, -0.15576171875, -0.1700439453125, -0.01386260986328125, 3.251953125, 3.01953125, 0.379638671875, 0.1287841796875, -0.1246337890625, 2.73828125, 0.76513671875, 1.95703125, -0.00876617431640625, 1.6650390625, 0.292236328125, 1.5693359375, 2.189453125, 0.09588623046875, -0.1632080078125, 1.447265625, -0.0265045166015625, 3.021484375, 0.6923828125, -0.04071044921875, 2.75390625, 2.08203125, 3.263671875, 2.177734375, 1.056640625, 0.07373046875, -0.053955078125, 1.6591796875, 1.3017578125, -0.165771484375, 1.3095703125, -0.093505859375, 2.3359375, 0.38232421875, 0.55712890625, -0.1353759765625, 2.095703125, 0.3974609375, -0.1646728515625, 0.84423828125, -0.0811767578125, 2.287109375, -0.00623321533203125, -0.038543701171875, 4.265625, -0.169921875, 1.060546875, -0.164794921875, 0.4638671875, 5.44921875, -0.00020837783813476562, 1.16015625, 0.1904296875, 0.1566162109375, -0.1146240234375, -0.0738525390625, 2.9140625, 1.0634765625, -0.0291290283203125, 0.108642578125, -0.01523590087890625, 5.93359375, 0.76904296875, -0.0007796287536621094, 1.984375, 1.314453125, 0.62353515625, 0.367919921875, -0.1632080078125, -0.0908203125, -0.00011610984802246094, 2.767578125, 4.1796875, 1.017578125, 1.275390625, -0.0122222900390625, -0.1695556640625, -0.00905609130859375, -0.0159912109375, 1.0927734375, 2.984375, 0.004955291748046875, 5.1171875], [1.7294921875, -9.47713851928711e-06, -0.168701171875, 2.744140625, 0.302978515625, 2.1875, -0.0015687942504882812, 2.681640625, -0.0272064208984375, 0.642578125, 4.37890625, 0.1656494140625, -0.1258544921875, 1.572265625, -0.093994140625, -0.06561279296875, 2.0390625, -0.0052642822265625, -0.02008056640625, 3.64453125, -0.0191650390625, 0.84033203125, 2.306640625, -0.0059661865234375, -0.042938232421875, 1.6865234375, -0.0673828125, 4.34765625, 4.55078125, -0.04840087890625, 1.1337890625, -0.1236572265625, 0.6875, -0.1409912109375, -0.056610107421875, 0.97802734375, -0.0016803741455078125, 0.54052734375, -0.09393310546875, -0.1015625, 1.8388671875, 0.0902099609375, -0.1661376953125, -0.10308837890625, -0.133544921875, 1.892578125, -0.1561279296875, -0.0, -0.0, -0.040863037109375, -0.016387939453125, -0.153076171875, -0.00970458984375, -0.0924072265625, 2.701171875, -0.0108489990234375, 1.69140625, -0.079833984375, 2.666015625, 1.80859375, 0.1104736328125, 2.21875, 0.37451171875, -0.032440185546875, 0.9404296875, 0.962890625, -0.05657958984375, 0.466064453125, -0.1700439453125, 0.607421875, -0.1424560546875, -0.004016876220703125, -0.138916015625, 3.330078125, 1.728515625, 3.150390625, 0.07781982421875, -0.1688232421875, -0.04541015625, 0.2958984375, 2.15625, 1.7099609375, 1.5400390625, -0.1666259765625, 0.2252197265625, -0.1109619140625, 1.7294921875, 1.216796875, -0.1446533203125, 0.25146484375, -8.344650268554688e-06, 1.447265625, -0.09844970703125, -0.16552734375, 1.21484375, 1.375, -0.135009765625, 3.310546875, 4.171875, 3.0234375, 2.7421875, 0.138427734375, 2.908203125, -0.08270263671875, -0.0364990234375, 0.297607421875, 0.465087890625, -0.08599853515625, 3.689453125, -0.00634002685546875, -5.930662155151367e-05, -0.1693115234375, 3.3984375, 1.5234375, 0.63427734375, 0.2476806640625, 1.09765625, 0.5751953125, 0.1859130859375, -0.0657958984375, -0.0024318695068359375, 1.9970703125, -0.169921875, 0.6787109375, 0.92431640625, 2.1328125, -0.100830078125, 1.6669921875, 1.7275390625, -0.05963134765625, -0.00913238525390625, 0.24755859375, -0.047882080078125, -0.168701171875, -0.030364990234375, 2.44140625, -0.1015625, -0.039154052734375, 0.24755859375, 1.00390625, -0.1685791015625, -0.060455322265625, 0.74072265625, 1.701171875, -0.001750946044921875, 1.8212890625, -0.0006256103515625, 1.2626953125, -0.1689453125, -0.1468505859375, 1.396484375, 2.943359375, 4.23828125, 2.794921875, 1.326171875, -0.011016845703125, -0.1700439453125, 0.2015380859375, -0.144775390625, 1.0908203125, -0.1439208984375, 3.642578125, -0.1181640625, -0.01568603515625, -0.160888671875, -0.1148681640625, 1.365234375, 2.84765625, 2.05859375, 2.06640625, -0.1236572265625, -0.03326416015625, -0.0943603515625, 0.0706787109375, -0.166259765625, 0.472412109375, 2.0859375, 0.1419677734375, 3.490234375, -0.10455322265625, 1.236328125, -0.01528167724609375, 2.068359375, 0.01180267333984375, -0.160888671875, -0.0703125, -0.0789794921875, -0.0364990234375, 1.611328125, -0.146728515625, 2.494140625, 5.109375, -0.168701171875, 1.759765625, -0.0400390625, -0.00025081634521484375, -0.05438232421875, 3.021484375, -0.0207366943359375, -0.1063232421875, -0.11358642578125, -0.0058135986328125, 0.51416015625, -0.09918212890625, 1.068359375, 0.0821533203125, -0.0067901611328125, 0.1885986328125, 1.1640625, -0.149658203125, 1.0126953125, 0.97607421875, -1.6391277313232422e-05, 1.1416015625, -5.608797073364258e-05, -0.0122222900390625, -0.1075439453125, -0.04754638671875, -0.0830078125, 2.1640625, -0.031524658203125, 2.314453125, 3.619140625, 2.759765625, -0.0266265869140625, -0.002399444580078125, 0.03564453125, 4.19140625, -0.1527099609375, -0.139404296875, -0.11737060546875, 1.3359375, 2.30078125, -0.03411865234375, -0.122314453125, 0.79052734375, 0.0008616447448730469, -0.166259765625, -0.00040030479431152344, 0.0985107421875, -0.12890625, -0.158935546875, -0.1680908203125, -0.060455322265625, 0.89013671875, 2.03515625, 0.83251953125, -0.11651611328125, 0.16845703125, 1.2119140625, -0.10693359375, 1.2099609375, -0.0049896240234375, -0.154296875, 2.806640625, 1.8818359375, -0.118408203125, 0.10736083984375, 4.2578125, 2.1015625, -0.1063232421875, 0.41357421875, -0.1622314453125, -0.1357421875, -0.01678466796875, 1.28125, 4.41796875, 1.6181640625, -0.0105133056640625, 0.884765625, -0.00437164306640625, 0.289794921875, 2.05859375, 2.947265625, 0.1416015625, 0.474365234375, 0.0117950439453125, -0.125244140625, 3.125, -0.05413818359375, -0.001964569091796875, 3.5078125, 4.53515625, -0.0019521713256835938, 1.8212890625, 1.53125, -0.1685791015625, -0.053863525390625, 0.2001953125, -0.168212890625, 1.5595703125, -0.1309814453125, 2.34375, -0.1314697265625, 3.263671875, 1.716796875, 2.2578125, -0.1700439453125, -0.147216796875, -0.076171875, 2.357421875, -0.1240234375, -0.0926513671875, -0.14599609375, 0.54931640625, -0.0672607421875, -0.05499267578125, -0.011016845703125, -0.134521484375, -0.03448486328125, 2.0625, -0.037200927734375, 1.2421875, -0.0108489990234375, -0.03411865234375, 0.57958984375, -0.03662109375, -0.127197265625, -0.1558837890625, 0.1815185546875, 0.401123046875, -0.005950927734375, 0.49169921875, 0.27490234375, 0.41748046875, -0.07073974609375, -0.06488037109375, 2.5390625, 2.265625, -0.1343994140625, 3.1953125, -0.168212890625, -0.06463623046875], [2.2109375, 0.47705078125, 4.57421875, -0.150146484375, 0.1572265625, 1.357421875, 0.364990234375, -0.1693115234375, -0.00014913082122802734, -0.02325439453125, -0.0212860107421875, -0.030548095703125, -0.0028057098388671875, 1.7119140625, -0.1331787109375, -0.1536865234375, -0.08984375, -0.1412353515625, 1.9013671875, 2.515625, 0.478515625, 0.2037353515625, -0.1551513671875, -0.1611328125, 0.318603515625, -0.11761474609375, -0.04754638671875, 0.0694580078125, -0.038116455078125, 4.14453125, 0.4482421875, -0.14599609375, 0.76904296875, -0.101318359375, -0.1549072265625, 3.1015625, 2.56640625, -0.0208282470703125, -0.0282745361328125, -0.1566162109375, 4.5078125, 0.0687255859375, 0.360595703125, -0.1573486328125, 2.646484375, 1.8779296875, -0.000728607177734375, -0.0765380859375, -0.00031638145446777344, -0.0231170654296875, -0.0841064453125, -0.009552001953125, -0.1290283203125, 1.2666015625, 2.2265625, 0.1324462890625, 1.34765625, -0.0085296630859375, 2.943359375, 4.58984375, -0.11224365234375, 0.99755859375, 0.5419921875, -1.2218952178955078e-05, 1.9951171875, 0.79296875, -0.0016012191772460938, -0.1683349609375, 2.29296875, 1.0029296875, -0.007511138916015625, -0.14404296875, -0.1153564453125, 0.1859130859375, -0.1109619140625, 0.048553466796875, -0.1280517578125, -0.003940582275390625, 0.63818359375, 0.3525390625, -0.11566162109375, 1.37109375, 4.578125, -0.0019779205322265625, 0.28515625, 1.5859375, 0.0282745361328125, -0.141357421875, 4.39453125, -0.1572265625, -0.1072998046875, -0.12322998046875, 2.40625, -1.5437602996826172e-05, -0.06402587890625, -0.1641845703125, 0.453125, 0.51171875, 1.9267578125, -0.1029052734375, -0.1024169921875, -0.06134033203125, -0.06964111328125, -0.1510009765625, -0.1612548828125, 0.437744140625, 1.5771484375, -0.0292510986328125, -0.1380615234375, -0.0281982421875, 0.00920867919921875, -0.15673828125, 1.0009765625, -0.14599609375, -0.0006113052368164062, 2.611328125, -0.0146484375, 3.259765625, 0.359375, 0.473388671875, -0.004344940185546875, 0.376220703125, -0.1416015625, -0.07757568359375, 0.939453125, 1.388671875, 2.345703125, -0.0171051025390625, -0.0909423828125, 2.7578125, 0.89013671875, 1.529296875, -0.1551513671875, 0.53955078125, 1.548828125, -0.0182952880859375, -0.12115478515625, 0.3564453125, 2.748046875, -0.1690673828125, 1.232421875, -0.0838623046875, -0.0228118896484375, 1.1494140625, -0.046234130859375, 0.06353759765625, -0.1527099609375, 0.90625, -0.0167083740234375, -0.0080718994140625, -0.00734710693359375, -0.130126953125, 2.546875, 1.57421875, 0.270263671875, -0.10089111328125, 2.404296875, -0.076416015625, 2.037109375, 2.685546875, 1.236328125, 0.6279296875, 0.61083984375, -0.0660400390625, 0.4033203125, -0.07305908203125, -0.081298828125, 3.08984375, -0.09613037109375, -0.0011587142944335938, 2.66015625, 2.662109375, 0.71923828125, -0.0178680419921875, 3.818359375, 2.09765625, 2.41015625, -0.1524658203125, 0.65234375, -0.159423828125, -0.149658203125, -0.0242767333984375, -0.0946044921875, -0.158203125, 0.0367431640625, 1.8837890625, 0.93408203125, -0.000492095947265625, 1.986328125, -0.0038700103759765625, 0.779296875, -0.1383056640625, 0.0615234375, -0.161865234375, -0.037353515625, 3.171875, -6.556510925292969e-07, -0.0673828125, 2.5625, -0.0272064208984375, -0.0005159378051757812, -0.07342529296875, -0.0261688232421875, 0.98681640625, -0.1326904296875, 0.46142578125, 1.3701171875, -0.000797271728515625, -0.037811279296875, -0.11712646484375, 0.26025390625, 1.2060546875, -0.037506103515625, -0.151611328125, -0.10577392578125, -0.106689453125, 3.033203125, -0.09783935546875, 0.161376953125, -2.205371856689453e-06, -0.10931396484375, 0.10150146484375, 1.2197265625, -0.1524658203125, -0.142578125, -0.0171051025390625, -0.1544189453125, 4.66015625, -0.0007452964782714844, -0.0007915496826171875, 0.07611083984375, -0.051910400390625, 2.39453125, -0.0031681060791015625, 3.21875, -0.163818359375, -0.001125335693359375, -6.4373016357421875e-06, -0.06005859375, -0.12103271484375, 0.50048828125, -0.01119232177734375, 6.015625, 1.537109375, -0.140625, -0.14404296875, 5.32421875, -0.0926513671875, 1.3662109375, 2.91015625, -0.056121826171875, 0.9892578125, -0.157470703125, 2.7421875, 2.876953125, 0.92236328125, 0.1944580078125, -0.0174407958984375, 4.8828125, 3.013671875, 1.3310546875, 2.990234375, 0.02142333984375, -0.0283966064453125, 1.8330078125, -0.052459716796875, -0.154052734375, 1.4130859375, -0.002338409423828125, 0.0340576171875, -0.1632080078125, -0.0234375, -0.018463134765625, -0.1435546875, -0.163330078125, 0.2044677734375, -0.0037746429443359375, 0.09857177734375, -0.01421356201171875, -0.08587646484375, 1.8896484375, -0.11865234375, -0.1597900390625, -0.0845947265625, -0.008209228515625, 1.5947265625, -0.1693115234375, 1.2099609375, -0.0113677978515625, 0.76025390625, -0.09033203125, 0.08221435546875, -0.0595703125, -0.1343994140625, 0.451416015625, -0.021026611328125, 1.4453125, -0.1668701171875, 0.74072265625, -0.0267333984375, -0.00157928466796875, 1.15234375, 1.255859375, -0.03985595703125, 1.1845703125, -0.01245880126953125, 1.197265625, 0.1802978515625, -0.05126953125, 2.181640625, -0.052276611328125, 4.51953125, -0.11279296875, -0.059173583984375, -0.142578125, 0.0738525390625, 0.1722412109375, 0.982421875, -0.1405029296875, -4.285573959350586e-05, -0.07720947265625, 0.72509765625, 2.232421875, -8.690357208251953e-05, 0.1387939453125, 2.236328125, -0.00710296630859375, 3.451171875, -0.09979248046875, -0.08282470703125, -0.1624755859375, 3.611328125, 1.779296875], [0.1949462890625, -0.1685791015625, -0.0011501312255859375, -0.159912109375, -0.1409912109375, -0.08856201171875, -0.00017547607421875, 2.564453125, 1.806640625, 3.24609375, -0.145263671875, -0.0170440673828125, -0.0182952880859375, 1.7529296875, -0.1390380859375, 3.400390625, 1.416015625, 1.5009765625, 1.001953125, -0.00469970703125, 1.408203125, -0.00199127197265625, 0.484130859375, 1.1005859375, -0.0257110595703125, -0.1358642578125, -0.11566162109375, 0.42236328125, 0.46826171875, 0.061553955078125, -0.07440185546875, 0.438720703125, 5.0390625, -0.1475830078125, 3.236328125, -0.0211181640625, -0.1318359375, 2.689453125, 1.6630859375, 3.259765625, 1.1015625, -0.13818359375, 0.0985107421875, -0.1671142578125, 2.0859375, 1.12109375, 2.51171875, -0.11773681640625, -0.1529541015625, -0.1683349609375, -0.1697998046875, -0.11566162109375, 2.568359375, 0.1885986328125, 2.36328125, 0.287353515625, -0.00047659873962402344, -0.007221221923828125, -0.126220703125, 2.3203125, 0.01451873779296875, -0.004241943359375, 0.1561279296875, -0.1531982421875, 1.044921875, 1.5966796875, 0.416748046875, -0.1280517578125, -0.03961181640625, -0.1695556640625, -0.061920166015625, 0.56591796875, -0.1654052734375, -0.1302490234375, 0.83544921875, 1.5673828125, 0.40771484375, -0.041168212890625, -0.1654052734375, 4.34765625, -0.03436279296875, -0.1697998046875, 1.7802734375, -0.052825927734375, -0.166015625, 2.5703125, 1.9169921875, -0.0004265308380126953, 1.080078125, -0.166748046875, -0.007305145263671875, 2.103515625, -0.155517578125, -0.03497314453125, -0.0755615234375, 1.5126953125, 1.767578125, 1.5244140625, 4.83203125, 1.1591796875, -0.1619873046875, 1.0166015625, 0.62646484375, 1.494140625, -3.2186508178710938e-06, 1.8330078125, 4.234375, 1.927734375, -0.159423828125, 4.578125, 1.796875, 4.4609375, 2.337890625, -0.08111572265625, -0.005985260009765625, 0.501953125, 1.455078125, -0.11322021484375, -0.034637451171875, 2.146484375, -0.1602783203125, 3.32421875, -0.163330078125, 1.146484375, -0.05462646484375, 1.2060546875, 1.5205078125, -0.0002593994140625, -0.167724609375, -0.166748046875, -0.1693115234375, 0.2110595703125, 2.7421875, -0.093017578125, 1.4609375, -0.1234130859375, -0.1629638671875, -0.0136566162109375, -0.1480712890625, -0.0682373046875, 1.1923828125, 3.453125, 1.3828125, 1.814453125, -0.134521484375, 1.7177734375, -0.12744140625, -0.1064453125, 0.7353515625, -0.16552734375, 1.5380859375, 0.496826171875, -0.0714111328125, -0.1697998046875, 0.5849609375, 0.7265625, 0.375732421875, 2.603515625, -0.1700439453125, -0.0208282470703125, -0.027435302734375, -0.09228515625, -0.107177734375, 0.052764892578125, 3.64453125, -0.0831298828125, -0.12548828125, -0.1690673828125, -0.089111328125, -0.1441650390625, 1.6982421875, 2.90234375, -0.155517578125, 0.1337890625, 2.15234375, 0.84130859375, -0.001270294189453125, 0.9169921875, 0.08843994140625, -0.160888671875, 1.2470703125, -0.166015625, -0.16650390625, 0.78125, -0.051727294921875, 1.6669921875, -0.150634765625, -0.0892333984375, 1.294921875, 1.5419921875, 2.177734375, 2.94921875, -0.084228515625, -0.07891845703125, -0.064453125, -0.14306640625, 1.943359375, 0.7333984375, 0.63720703125, 0.94580078125, -0.158203125, 2.740234375, -0.07861328125, 1.4853515625, -0.16650390625, -0.0970458984375, 4.1953125, -0.15087890625, 1.5048828125, -0.1265869140625, -0.1380615234375, 2.560546875, 0.8828125, 0.2342529296875, 0.990234375, 1.3212890625, 1.828125, 1.0244140625, 1.466796875, 0.6572265625, 0.33349609375, 2.369140625, -5.2928924560546875e-05, 1.751953125, 0.149658203125, 0.39404296875, -0.11639404296875, -0.1397705078125, 1.244140625, -0.03326416015625, -0.06805419921875, -0.06549072265625, 3.333984375, 0.063720703125, 0.03564453125, -0.0102386474609375, 0.1422119140625, 1.658203125, -0.041015625, 0.11126708984375, 0.305419921875, 1.1103515625, 1.8974609375, 3.884765625, 0.01318359375, 0.04327392578125, -0.1358642578125, 1.8369140625, -0.025146484375, 0.76171875, 1.25390625, -0.10821533203125, -0.1251220703125, 0.2008056640625, 0.17578125, 0.06597900390625, -0.110595703125, -0.037261962890625, 2.34765625, -0.142333984375, 0.1070556640625, 0.018402099609375, 2.0546875, -0.159912109375, -0.1444091796875, -0.162109375, 1.4951171875, 1.1376953125, 0.7060546875, 0.1944580078125, 4.390625, 0.467041015625, -0.00028896331787109375, -0.0069427490234375, -0.0098114013671875, -0.1693115234375, -0.004756927490234375, 0.01551055908203125, -0.007427215576171875, -0.04644775390625, -0.1353759765625, 0.8173828125, 0.1453857421875, -0.057098388671875, 3.484375, -0.0308837890625, -0.155517578125, -0.0084381103515625, 1.8291015625, 1.7568359375, -0.08221435546875, -9.5367431640625e-07, 1.392578125, 3.455078125, -0.07330322265625, 0.73486328125, 1.6591796875, 0.034149169921875, -0.0242767333984375, -0.13916015625, -0.1292724609375, 0.8974609375, -0.0007915496826171875, 0.83447265625, 0.06402587890625, -0.0028972625732421875, -0.046234130859375, 0.87841796875, 2.4609375, 1.5732421875, 5.66015625, 0.2149658203125, -0.0207366943359375, 0.0137786865234375, -0.1065673828125, -1.2516975402832031e-06, 1.986328125, -0.04876708984375, -0.059051513671875, 3.505859375, -0.125732421875, -0.075439453125, 1.533203125, 3.583984375, -0.00024890899658203125, -0.0311431884765625, -0.1298828125, 1.4033203125, 0.30615234375, 2.1640625, -0.004756927490234375, -0.10699462890625, 2.833984375], [-0.0003294944763183594, 2.046875, -0.07745361328125, 0.354736328125, -0.035614013671875, 1.064453125, 1.3984375, -0.1251220703125, 3.212890625, -0.043914794921875, 1.845703125, -0.0246124267578125, 1.802734375, 1.5400390625, -0.1060791015625, 3.087890625, -0.05126953125, -0.0079803466796875, -0.1680908203125, 1.70703125, 0.038970947265625, 1.7392578125, 2.794921875, 1.548828125, 0.9208984375, 0.280029296875, 0.7802734375, 2.876953125, -0.1422119140625, -0.130615234375, 1.5, -0.051727294921875, 1.9482421875, 0.03948974609375, 0.83837890625, -0.140380859375, -0.135986328125, 2.388671875, -0.0357666015625, 1.9970703125, 2.216796875, 0.125244140625, -0.0001595020294189453, -0.033111572265625, 0.63330078125, 2.08984375, -0.0080718994140625, 0.583984375, 0.34619140625, -0.004192352294921875, 0.2447509765625, -0.10430908203125, -0.00429534912109375, 3.56640625, -0.10345458984375, -0.0064849853515625, -0.0069427490234375, 3.84765625, 1.92578125, 0.9599609375, -0.084228515625, 0.609375, 0.09393310546875, 1.3740234375, -0.005451202392578125, 1.3515625, 1.9208984375, -0.06903076171875, -0.059967041015625, -0.09161376953125, -0.01007843017578125, -0.161376953125, -0.16259765625, 2.7265625, -0.0465087890625, -0.1666259765625, 2.61328125, -0.0008783340454101562, 2.01171875, -0.164794921875, 1.0185546875, -0.01024627685546875, 1.1376953125, -0.0, -0.1104736328125, -1.2218952178955078e-05, -0.1641845703125, -0.12451171875, 0.070556640625, 3.3359375, 0.90625, 0.64697265625, 1.158203125, -0.1490478515625, 3.03125, 1.2353515625, 0.06549072265625, 2.4609375, -0.1693115234375, -0.160400390625, -0.0748291015625, -0.007061004638671875, -0.019989013671875, 2.349609375, -0.1685791015625, -0.0263214111328125, 1.3583984375, -0.1129150390625, -0.050262451171875, -0.0784912109375, -0.166015625, 1.9951171875, -0.1534423828125, -0.14404296875, 0.87744140625, 0.0186614990234375, 0.058013916015625, 0.293212890625, 3.375, -0.1690673828125, -0.163330078125, 3.572265625, -0.0625, -0.0006961822509765625, 1.62109375, -0.1605224609375, -0.02752685546875, 0.08587646484375, -0.0233306884765625, 0.81689453125, -0.0178680419921875, 1.568359375, 2.220703125, -0.11260986328125, 2.12109375, 0.2100830078125, 3.64453125, 0.0153656005859375, -0.00039696693420410156, -0.000324249267578125, 1.6826171875, -0.1630859375, 0.5751953125, -0.0272064208984375, -0.140380859375, -6.139278411865234e-06, 0.5693359375, 1.4423828125, -0.0902099609375, 0.796875, -0.100830078125, 0.5068359375, -0.161376953125, -0.0631103515625, 0.69140625, -0.0001666545867919922, 3.56640625, -0.00157928466796875, -5.608797073364258e-05, 0.3095703125, -0.03204345703125, 0.97509765625, -0.0050811767578125, 2.8671875, 2.435546875, -0.030364990234375, 3.87890625, -0.161865234375, -0.0367431640625, -0.1141357421875, 0.31396484375, 0.525390625, -0.12384033203125, 0.364990234375, 0.10382080078125, 0.71142578125, -0.00881195068359375, -0.1634521484375, -0.0191650390625, -0.080078125, -0.0892333984375, -0.150146484375, -0.106689453125, 3.568359375, -0.0440673828125, 0.7978515625, -0.16015625, 2.859375, 2.306640625, 0.60546875, 1.1494140625, -0.06805419921875, -0.035186767578125, 1.3642578125, -0.01983642578125, -0.0020732879638671875, -0.1412353515625, -0.009246826171875, -0.004215240478515625, -0.08489990234375, 0.020050048828125, -0.1697998046875, -0.1510009765625, -0.0311431884765625, -0.154296875, -0.127197265625, -0.155029296875, 4.609375, -0.1685791015625, 0.07525634765625, -0.08221435546875, -0.146484375, 0.005702972412109375, 0.88134765625, -0.0, 1.9697265625, 0.83544921875, 0.267822265625, -0.03436279296875, -0.0003752708435058594, -0.0222015380859375, 1.1669921875, 1.0869140625, -0.039306640625, 1.24609375, -0.052734375, -0.027435302734375, 2.423828125, 2.119140625, 1.404296875, 2.111328125, -0.05328369140625, -2.562999725341797e-06, -0.1617431640625, 2.125, -0.1580810546875, 1.2373046875, 1.060546875, -0.061859130859375, 0.84033203125, 0.08087158203125, -0.004901885986328125, 2.064453125, -0.033111572265625, 0.7314453125, 0.81640625, -0.0950927734375, -0.05816650390625, 0.68212890625, 0.2978515625, 2.369140625, 0.2064208984375, 0.84423828125, 0.206787109375, 1.7998046875, -0.04547119140625, -0.00021195411682128906, 0.8056640625, -0.14111328125, -0.1285400390625, 3.3046875, -0.00601959228515625, -0.163330078125, 1.7177734375, -0.140380859375, -0.1676025390625, 0.4423828125, -0.1700439453125, -0.0008587837219238281, -0.01029205322265625, 1.201171875, 1.1708984375, -0.0207366943359375, 0.344482421875, 1.109375, 1.177734375, 2.841796875, 1.8701171875, -0.06341552734375, -0.1541748046875, 1.2109375, -0.0252685546875, -0.0016241073608398438, 1.498046875, 2.197265625, 1.04296875, 0.447265625, -0.002353668212890625, -0.00659942626953125, -0.13427734375, 3.279296875, -0.1629638671875, 0.2154541015625, -0.0916748046875, 2.630859375, -0.0002529621124267578, 3.2578125, 0.0112762451171875, -5.960464477539063e-08, 0.10174560546875, 1.0224609375, 1.013671875, 0.78466796875, -0.1627197265625, -0.1590576171875, 0.1484375, -0.140625, 3.427734375, -0.0228118896484375, 0.1868896484375, -0.1595458984375, -0.005680084228515625, 2.828125, 0.391357421875, -0.12176513671875, -0.07720947265625, -0.1551513671875, 3.140625, 0.3125, -0.06610107421875, 0.1842041015625, -0.1180419921875, 0.92626953125, 0.74853515625, 1.435546875, 0.5234375, 0.1619873046875, -0.0154571533203125, 0.383056640625, 2.248046875, -0.007061004638671875, 1.8466796875, -0.1474609375], [-0.141357421875, -0.14599609375, 1.380859375, 0.0911865234375, 0.310546875, -0.055267333984375, 3.21484375, -0.0653076171875, 2.673828125, 2.556640625, 1.443359375, -0.0207366943359375, 0.1287841796875, -0.11590576171875, 3.189453125, 0.4892578125, 0.82080078125, 0.408203125, 0.48779296875, 2.162109375, 4.37109375, 1.583984375, 2.609375, 1.408203125, 0.86865234375, -0.03533935546875, -0.1644287109375, -0.156982421875, 1.2294921875, -0.0, -3.743171691894531e-05, -0.1695556640625, 0.82275390625, -0.1553955078125, 0.007061004638671875, 1.546875, 3.0390625, 3.53515625, 1.2705078125, 2.37890625, -0.1484375, -0.105712890625, 1.3994140625, 3.39453125, 1.2783203125, -0.000827789306640625, 2.41796875, 4.12109375, 1.7587890625, -0.06231689453125, -0.128662109375, -0.06353759765625, 3.291015625, -0.06396484375, -0.0261688232421875, 0.03326416015625, 0.08441162109375, 0.99658203125, -0.1280517578125, 0.94921875, -0.166259765625, -0.1512451171875, -0.0033321380615234375, 0.497314453125, 2.97265625, 1.5537109375, 2.572265625, -0.1181640625, -0.1527099609375, -0.150634765625, -0.06890869140625, 1.9521484375, 2.869140625, -0.1619873046875, 1.1376953125, -0.022003173828125, 0.86962890625, 1.8994140625, -0.134765625, 2.830078125, -0.1495361328125, 0.74365234375, 3.640625, -0.0098114013671875, 1.189453125, -0.05914306640625, 3.287109375, 0.187744140625, 0.01422119140625, 1.2158203125, 3.8203125, -0.16943359375, -0.09942626953125, -0.1390380859375, 2.1875, -0.0018367767333984375, -0.04840087890625, -0.160888671875, -0.0966796875, -0.045166015625, -0.1414794921875, -0.00022494792938232422, 4.703125, 2.236328125, -0.059814453125, -0.166015625, -0.1419677734375, -0.0007619857788085938, -0.043243408203125, 0.397216796875, 0.56103515625, 2.984375, 0.09844970703125, -0.1551513671875, 1.9716796875, 1.25, -0.0218048095703125, -0.1356201171875, 1.880859375, -0.12225341796875, 0.227294921875, -0.06610107421875, 2.22265625, 4.2734375, -0.00011819601058959961, -0.055267333984375, 1.484375, 0.58251953125, 0.0220947265625, -0.0188140869140625, -0.0011835098266601562, -0.166015625, 2.1015625, 2.140625, 4.953125, -0.0015687942504882812, 0.467529296875, 0.60205078125, 4.08984375, -0.148193359375, 3.009765625, -0.0050201416015625, 2.384765625, -0.08612060546875, -0.05975341796875, 0.78662109375, -0.169677734375, 0.5380859375, -0.038421630859375, -0.10430908203125, 1.6962890625, -0.1346435546875, -0.130615234375, -0.1246337890625, -0.0002529621124267578, -0.01312255859375, 0.57080078125, 0.360107421875, -0.0791015625, 2.728515625, 0.04730224609375, 0.06353759765625, 1.0361328125, 0.625, 2.439453125, -0.16259765625, 0.1954345703125, 0.0799560546875, -0.1610107421875, -0.0112152099609375, 1.96484375, 3.17578125, 0.141845703125, 0.12298583984375, 0.537109375, 2.439453125, 1.3212890625, 3.51171875, 3.97265625, 2.521484375, -0.165771484375, -0.15087890625, -0.156982421875, -0.1688232421875, 2.216796875, -0.1571044921875, 1.15625, -0.160400390625, -0.10528564453125, 2.70703125, -0.01259613037109375, 2.978515625, -0.014862060546875, 1.1474609375, 0.6416015625, 0.6337890625, 0.051605224609375, -0.1524658203125, -0.10040283203125, 2.98828125, 2.171875, 4.7578125, 2.564453125, 1.4404296875, 0.5869140625, 0.0159454345703125, 3.07421875, -0.12152099609375, -0.09564208984375, -0.163330078125, 1.2958984375, 6.75, -0.1187744140625, -0.0026988983154296875, -0.141845703125, 1.93359375, -0.153564453125, -0.1700439453125, 1.5302734375, 0.0211639404296875, -0.0950927734375, 0.9296875, -0.0643310546875, -0.0240325927734375, 3.88671875, -1.4781951904296875e-05, 3.357421875, -0.0005078315734863281, 0.46240234375, -5.0008296966552734e-05, -0.143798828125, -0.12042236328125, -0.1466064453125, -0.0906982421875, 3.1328125, 2.056640625, -0.00396728515625, -0.0008912086486816406, -0.08734130859375, -0.0086669921875, 4.01171875, 2.9296875, 1.224609375, -0.0009946823120117188, -0.1246337890625, 2.041015625, -0.06732177734375, 1.4638671875, 0.11566162109375, 1.94140625, -0.004913330078125, -0.01555633544921875, -0.014007568359375, -0.0770263671875, -0.0810546875, 1.4150390625, -0.00726318359375, 1.7392578125, 2.0625, 1.2587890625, -0.00406646728515625, -0.016143798828125, 0.28759765625, 0.1534423828125, -0.1697998046875, -0.1348876953125, -0.037078857421875, -0.04522705078125, -0.035491943359375, -0.146728515625, -0.09808349609375, -0.1688232421875, -0.059967041015625, 2.0859375, 0.264404296875, -0.045745849609375, 5.3671875, -1.1920928955078125e-07, 1.1103515625, 0.505859375, -0.1436767578125, -0.039764404296875, -0.07183837890625, 2.08984375, 1.73828125, 0.81884765625, 0.282958984375, 0.0550537109375, 1.8203125, 0.1634521484375, 0.337158203125, 2.748046875, -0.1561279296875, 0.521484375, -0.050994873046875, -0.0811767578125, 2.90625, -0.056427001953125, -0.169677734375, -0.080322265625, 0.74267578125, 1.0556640625, 3.3046875, 3.32421875, 3.27734375, 0.95458984375, 2.04296875, -0.1456298828125, -0.0531005859375, 2.484375, -0.09759521484375, 1.302734375, 3.0703125, 0.0294189453125, 2.912109375, 0.167724609375, -0.0035762786865234375, 1.3828125, 3.818359375, -0.024505615234375, 0.541015625, -0.131103515625, 0.54833984375, 1.048828125, -0.009918212890625, -0.049560546875, 1.9072265625, 0.859375, -0.148193359375, 0.16015625, 1.96484375, 1.884765625, -7.092952728271484e-05], [-0.1634521484375, 3.453125, -0.1661376953125, -0.1700439453125, 0.26025390625, 0.62548828125, 0.425537109375, 0.69287109375, -0.10845947265625, 0.78662109375, 1.61328125, 0.192626953125, -0.0794677734375, -0.1302490234375, 0.3076171875, -0.0235443115234375, -0.1690673828125, -0.1123046875, -5.0067901611328125e-06, 0.331298828125, -0.0033740997314453125, -0.06890869140625, -0.0665283203125, -0.1689453125, -0.1590576171875, 0.8486328125, -0.0257110595703125, -0.1617431640625, 0.456787109375, -0.126953125, -0.0887451171875, 0.1802978515625, 1.0087890625, 0.712890625, -0.007724761962890625, 0.044464111328125, -0.1014404296875, 3.185546875, -0.09918212890625, -0.16943359375, 2.568359375, -0.0902099609375, 2.314453125, 2.314453125, -0.11663818359375, 0.281982421875, 1.0029296875, 4.27734375, 0.284423828125, -0.169921875, 1.2626953125, 0.1822509765625, -0.10595703125, 0.0794677734375, 0.70703125, 2.447265625, 1.0986328125, 1.041015625, -0.1610107421875, 2.07421875, -0.13232421875, -0.12164306640625, -0.1683349609375, -0.09918212890625, 0.17041015625, 1.29296875, 0.1201171875, -0.169189453125, 0.392822265625, -0.163818359375, 1.958984375, 2.958984375, -0.1375732421875, -0.1563720703125, 2.013671875, 4.5703125, -0.1163330078125, 2.78125, 1.03125, -0.090576171875, 0.08001708984375, 0.69873046875, -0.16796875, 3.919921875, -5.125999450683594e-06, -0.163818359375, -0.1597900390625, -0.10693359375, -0.125244140625, 0.06304931640625, 0.85693359375, 0.10894775390625, 1.1796875, -0.11224365234375, -0.0458984375, -0.12176513671875, 2.416015625, -0.1474609375, -0.1688232421875, 0.595703125, 0.9990234375, -0.0474853515625, 1.51953125, 2.806640625, 3.21875, 0.277587890625, -0.01113128662109375, -0.04132080078125, -0.041656494140625, 0.453125, 2.39453125, 0.78662109375, 1.7314453125, 5.96875, -0.0230255126953125, 2.61328125, -0.0008029937744140625, -0.1424560546875, -0.169921875, -0.161865234375, 3.876953125, 0.7021484375, 2.810546875, -0.01421356201171875, 3.06640625, 0.82373046875, 1.8994140625, -0.072509765625, 1.1025390625, 0.2344970703125, -0.11700439453125, 0.09527587890625, 0.333251953125, 0.62255859375, -0.101318359375, 1.5556640625, -0.0205535888671875, -0.057403564453125, -0.00152587890625, -0.0, 3.48828125, 3.498046875, -0.1630859375, -0.16845703125, -0.06298828125, 0.039886474609375, -0.033966064453125, -0.0297393798828125, -0.0675048828125, -0.007221221923828125, -0.153076171875, 2.2265625, 1.2412109375, 0.296875, -0.1610107421875, 3.865234375, -0.1480712890625, -1.4781951904296875e-05, -0.1680908203125, -3.3974647521972656e-06, -0.048095703125, -0.1475830078125, 2.87890625, 2.724609375, -0.1654052734375, -0.1370849609375, 1.341796875, -0.039764404296875, -0.0927734375, 2.884765625, -0.04473876953125, 0.295166015625, 4.88671875, -0.169921875, 3.869140625, -0.01259613037109375, -0.0806884765625, 2.8046875, 1.41796875, 0.0909423828125, -0.032501220703125, 2.212890625, 3.8203125, -0.030120849609375, -0.1676025390625, 1.9755859375, -0.14892578125, 1.990234375, -0.07305908203125, 2.927734375, 1.5810546875, 0.5439453125, 0.7392578125, 3.408203125, -0.03662109375, 0.65869140625, -0.10406494140625, -0.09808349609375, 3.02734375, 0.9599609375, 0.481689453125, 1.546875, -0.169189453125, 2.63671875, -0.1666259765625, 1.3759765625, 0.255615234375, 2.021484375, -0.091064453125, 1.6875, 1.76953125, 0.31103515625, -0.1697998046875, -0.05377197265625, -0.1688232421875, 0.69580078125, -0.1697998046875, -0.002323150634765625, 1.29296875, 0.9990234375, -0.1676025390625, -0.01678466796875, -0.1475830078125, -0.164794921875, -0.135498046875, 0.255615234375, -0.0001150369644165039, 0.04754638671875, -0.1685791015625, 1.623046875, -0.06634521484375, -0.1064453125, -0.169677734375, -5.930662155151367e-05, 4.484375, -0.165771484375, 0.73974609375, 0.08880615234375, -0.129638671875, -0.07244873046875, -0.0001100301742553711, 0.560546875, -0.1510009765625, -0.1536865234375, 2.216796875, 0.043121337890625, 3.9453125, 2.51171875, 1.1904296875, 0.7060546875, -0.0570068359375, 4.55078125, 3.576171875, -0.059783935546875, 0.2030029296875, -0.0063018798828125, 2.58203125, 0.58984375, 0.59716796875, -0.1397705078125, -0.039794921875, -0.1422119140625, -0.16943359375, -0.1700439453125, -0.1634521484375, 3.771484375, 4.84375, -0.00015676021575927734, -0.018585205078125, -0.1363525390625, -0.1351318359375, 1.3662109375, -0.0, 1.51171875, -0.00011819601058959961, 0.199462890625, -0.162841796875, -0.11163330078125, -0.07818603515625, -0.0280303955078125, 3.552734375, -0.0006070137023925781, -0.0011587142944335938, -0.14208984375, -0.036376953125, 2.541015625, -0.1376953125, -0.0171051025390625, -0.1329345703125, -0.07952880859375, -0.16845703125, 0.74853515625, -0.169921875, -0.11187744140625, 1.041015625, -0.0458984375, -0.08984375, 0.414306640625, -0.11212158203125, 2.015625, -0.1583251953125, 3.501953125, -0.1622314453125, -0.1512451171875, -0.0181121826171875, 0.055389404296875, 1.080078125, -0.002353668212890625, -0.0001386404037475586, -0.009246826171875, -0.06201171875, -0.0728759765625, -0.0435791015625, -0.156982421875, -0.10003662109375, 0.52978515625, -0.013458251953125, -0.00018799304962158203, -0.1524658203125, 3.35546875, -0.14306640625, -0.16259765625, 0.08935546875, -0.0005445480346679688, 1.1123046875, -0.0653076171875, -0.00017404556274414062, -0.1610107421875, -0.157470703125, 1.8515625, 0.79248046875, -0.158935546875, 1.6181640625]], "dtype": "float16"}}, {"a": [{"__nd__": [-60.0, -45.0, 45.0, 60.0, 300.0, -300.0], "dtype": "float32"}], "e": {"__nd__": [-0.0, -0.0, 45.0, 60.0, 300.0, -0.0], "dtype": "float32"}}, {"a": [{"__nd__": [-60.0, -45.0, 45.0, 60.0], "dtype": "float16"}], "e": {"__nd__": [-0.0, -0.0, 45.0, 60.0], "dtype": "float16"}}, {"a": [{"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [[-6.0, -5.98828125, -5.9765625, -5.96484375, -5.953125, -5.94140625, -5.9296875, -5.91796875, -5.90625, -5.89453125, -5.8828125, -5.87109375, -5.859375, -5.84765625, -5.8359375, -5.82421875, -5.8125, -5.80078125, -5.7890625, -5.77734375, -5.765625, -5.75390625, -5.7421875, -5.73046875, -5.71875, -5.70703125, -5.6953125, -5.68359375, -5.671875, -5.66015625, -5.6484375, -5.63671875, -5.625, -5.61328125, -5.6015625, -5.58984375, -5.578125, -5.56640625, -5.5546875, -5.54296875, -5.53125, -5.51953125, -5.5078125, -5.49609375, -5.484375, -5.47265625, -5.4609375, -5.44921875, -5.4375, -5.42578125, -5.4140625, -5.40234375, -5.390625, -5.37890625, -5.3671875, -5.35546875, -5.34375, -5.33203125, -5.3203125, -5.30859375, -5.296875, -5.28515625, -5.2734375, -5.26171875, -5.25, -5.23828125, -5.2265625, -5.21484375, -5.203125, -5.19140625, -5.1796875, -5.16796875, -5.15625, -5.14453125, -5.1328125, -5.12109375, -5.109375, -5.09765625, -5.0859375, -5.07421875, -5.0625, -5.05078125, -5.0390625, -5.02734375, -5.015625, -5.00390625, -4.9921875, -4.98046875, -4.96875, -4.95703125, -4.9453125, -4.93359375, -4.921875, -4.91015625, -4.8984375, -4.88671875, -4.875, -4.86328125, -4.8515625, -4.83984375, -4.828125, -4.81640625, -4.8046875, -4.79296875, -4.78125, -4.76953125, -4.7578125, -4.74609375, -4.734375, -4.72265625, -4.7109375, -4.69921875, -4.6875, -4.67578125, -4.6640625, -4.65234375, -4.640625, -4.62890625, -4.6171875, -4.60546875, -4.59375, -4.58203125, -4.5703125, -4.55859375, -4.546875, -4.53515625, -4.5234375, -4.51171875, -4.5, -4.48828125, -4.4765625, -4.46484375, -4.453125, -4.44140625, -4.4296875, -4.41796875, -4.40625, -4.39453125, -4.3828125, -4.37109375, -4.359375, -4.34765625, -4.3359375, -4.32421875, -4.3125, -4.30078125, -4.2890625, -4.27734375, -4.265625, -4.25390625, -4.2421875, -4.23046875, -4.21875, -4.20703125, -4.1953125, -4.18359375, -4.171875, -4.16015625, -4.1484375, -4.13671875, -4.125, -4.11328125, -4.1015625, -4.08984375, -4.078125, -4.06640625, -4.0546875, -4.04296875, -4.03125, -4.01953125, -4.0078125, -3.99609375, -3.984375, -3.97265625, -3.9609375, -3.94921875, -3.9375, -3.92578125, -3.9140625, -3.90234375, -3.890625, -3.87890625, -3.8671875, -3.85546875, -3.84375, -3.83203125, -3.8203125, -3.80859375, -3.796875, -3.78515625, -3.7734375, -3.76171875, -3.75, -3.73828125, -3.7265625, -3.71484375, -3.703125, -3.69140625, -3.6796875, -3.66796875, -3.65625, -3.64453125, -3.6328125, -3.62109375, -3.609375, -3.59765625, -3.5859375, -3.57421875, -3.5625, -3.55078125, -3.5390625, -3.52734375, -3.515625, -3.50390625, -3.4921875, -3.48046875, -3.46875, -3.45703125, -3.4453125, -3.43359375, -3.421875, -3.41015625, -3.3984375, -3.38671875, -3.375, -3.36328125, -3.3515625, -3.33984375, -3.328125, -3.31640625, -3.3046875, -3.29296875, -3.28125, -3.26953125, -3.2578125, -3.24609375, -3.234375, -3.22265625, -3.2109375, -3.19921875, -3.1875, -3.17578125, -3.1640625, -3.15234375, -3.140625, -3.12890625, -3.1171875, -3.10546875, -3.09375, -3.08203125, -3.0703125, -3.05859375, -3.046875, -3.03515625, -3.0234375, -3.01171875, -3.0, -2.98828125, -2.9765625, -2.96484375, -2.953125, -2.94140625, -2.9296875, -2.91796875, -2.90625, -2.89453125, -2.8828125, -2.87109375, -2.859375, -2.84765625, -2.8359375, -2.82421875, -2.8125, -2.80078125, -2.7890625, -2.77734375, -2.765625, -2.75390625, -2.7421875, -2.73046875, -2.71875, -2.70703125, -2.6953125, -2.68359375, -2.671875, -2.66015625, -2.6484375, -2.63671875, -2.625, -2.61328125, -2.6015625, -2.58984375, -2.578125, -2.56640625, -2.5546875, -2.54296875, -2.53125, -2.51953125, -2.5078125, -2.49609375, -2.484375, -2.47265625, -2.4609375, -2.44921875, -2.4375, -2.42578125, -2.4140625, -2.40234375, -2.390625, -2.37890625, -2.3671875, -2.35546875, -2.34375, -2.33203125, -2.3203125, -2.30859375, -2.296875, -2.28515625, -2.2734375, -2.26171875, -2.25, -2.23828125, -2.2265625, -2.21484375, -2.203125, -2.19140625, -2.1796875, -2.16796875, -2.15625, -2.14453125, -2.1328125, -2.12109375, -2.109375, -2.09765625, -2.0859375, -2.07421875, -2.0625, -2.05078125, -2.0390625, -2.02734375, -2.015625, -2.00390625, -1.9921875, -1.98046875, -1.96875, -1.95703125, -1.9453125, -1.93359375, -1.921875, -1.91015625, -1.8984375, -1.88671875, -1.875, -1.86328125, -1.8515625, -1.83984375, -1.828125, -1.81640625, -1.8046875, -1.79296875, -1.78125, -1.76953125, -1.7578125, -1.74609375, -1.734375, -1.72265625, -1.7109375, -1.69921875, -1.6875, -1.67578125, -1.6640625, -1.65234375, -1.640625, -1.62890625, -1.6171875, -1.60546875, -1.59375, -1.58203125, -1.5703125, -1.55859375, -1.546875, -1.53515625, -1.5234375, -1.51171875, -1.5, -1.48828125, -1.4765625, -1.46484375, -1.453125, -1.44140625, -1.4296875, -1.41796875, -1.40625, -1.39453125, -1.3828125, -1.37109375, -1.359375, -1.34765625, -1.3359375, -1.32421875, -1.3125, -1.30078125, -1.2890625, -1.27734375, -1.265625, -1.25390625, -1.2421875, -1.23046875, -1.21875, -1.20703125, -1.1953125, -1.18359375, -1.171875, -1.16015625, -1.1484375, -1.13671875, -1.125, -1.11328125, -1.1015625, -1.08984375, -1.078125, -1.06640625, -1.0546875, -1.04296875, -1.03125, -1.01953125, -1.0078125, -0.99609375, -0.984375, -0.97265625, -0.9609375, -0.94921875, -0.9375, -0.92578125, -0.9140625, -0.90234375, -0.890625, -0.87890625, -0.8671875, -0.85546875, -0.84375, -0.83203125, -0.8203125, -0.80859375, -0.796875, -0.78515625, -0.7734375, -0.76171875, -0.75, -0.73828125, -0.7265625, -0.71484375, -0.703125, -0.69140625, -0.6796875, -0.66796875, -0.65625, -0.64453125, -0.6328125, -0.62109375, -0.609375, -0.59765625, -0.5859375, -0.57421875, -0.5625, -0.55078125, -0.5390625, -0.52734375, -0.515625, -0.50390625, -0.4921875, -0.48046875, -0.46875, -0.45703125, -0.4453125, -0.43359375, -0.421875, -0.41015625, -0.3984375, -0.38671875, -0.375, -0.36328125, -0.3515625, -0.33984375, -0.328125, -0.31640625, -0.3046875, -0.29296875, -0.28125, -0.26953125, -0.2578125, -0.24609375, -0.234375, -0.22265625, -0.2109375, -0.19921875, -0.1875, -0.17578125, -0.1640625, -0.15234375, -0.140625, -0.12890625, -0.1171875, -0.10546875, -0.09375, -0.08203125, -0.0703125, -0.05859375, -0.046875, -0.03515625, -0.0234375, -0.01171875, 0.0, 0.01171875, 0.0234375, 0.03515625, 0.046875, 0.05859375, 0.0703125, 0.08203125, 0.09375, 0.10546875, 0.1171875, 0.12890625, 0.140625, 0.15234375, 0.1640625, 0.17578125, 0.1875, 0.19921875, 0.2109375, 0.22265625, 0.234375, 0.24609375, 0.2578125, 0.26953125, 0.28125, 0.29296875, 0.3046875, 0.31640625, 0.328125, 0.33984375, 0.3515625, 0.36328125, 0.375, 0.38671875, 0.3984375, 0.41015625, 0.421875, 0.43359375, 0.4453125, 0.45703125, 0.46875, 0.48046875, 0.4921875, 0.50390625, 0.515625, 0.52734375, 0.5390625, 0.55078125, 0.5625, 0.57421875, 0.5859375, 0.59765625, 0.609375, 0.62109375, 0.6328125, 0.64453125, 0.65625, 0.66796875, 0.6796875, 0.69140625, 0.703125, 0.71484375, 0.7265625, 0.73828125, 0.75, 0.76171875, 0.7734375, 0.78515625, 0.796875, 0.80859375, 0.8203125, 0.83203125, 0.84375, 0.85546875, 0.8671875, 0.87890625, 0.890625, 0.90234375, 0.9140625, 0.92578125, 0.9375, 0.94921875, 0.9609375, 0.97265625, 0.984375, 0.99609375, 1.0078125, 1.01953125, 1.03125, 1.04296875, 1.0546875, 1.06640625, 1.078125, 1.08984375, 1.1015625, 1.11328125, 1.125, 1.13671875, 1.1484375, 1.16015625, 1.171875, 1.18359375, 1.1953125, 1.20703125, 1.21875, 1.23046875, 1.2421875, 1.25390625, 1.265625, 1.27734375, 1.2890625, 1.30078125, 1.3125, 1.32421875, 1.3359375, 1.34765625, 1.359375, 1.37109375, 1.3828125, 1.39453125, 1.40625, 1.41796875, 1.4296875, 1.44140625, 1.453125, 1.46484375, 1.4765625, 1.48828125, 1.5, 1.51171875, 1.5234375, 1.53515625, 1.546875, 1.55859375, 1.5703125, 1.58203125, 1.59375, 1.60546875, 1.6171875, 1.62890625, 1.640625, 1.65234375, 1.6640625, 1.67578125, 1.6875, 1.69921875, 1.7109375, 1.72265625, 1.734375, 1.74609375, 1.7578125, 1.76953125, 1.78125, 1.79296875, 1.8046875, 1.81640625, 1.828125, 1.83984375, 1.8515625, 1.86328125, 1.875, 1.88671875, 1.8984375, 1.91015625, 1.921875, 1.93359375, 1.9453125, 1.95703125, 1.96875, 1.98046875, 1.9921875, 2.00390625, 2.015625, 2.02734375, 2.0390625, 2.05078125, 2.0625, 2.07421875, 2.0859375, 2.09765625, 2.109375, 2.12109375, 2.1328125, 2.14453125, 2.15625, 2.16796875, 2.1796875, 2.19140625, 2.203125, 2.21484375, 2.2265625, 2.23828125, 2.25, 2.26171875, 2.2734375, 2.28515625, 2.296875, 2.30859375, 2.3203125, 2.33203125, 2.34375, 2.35546875, 2.3671875, 2.37890625, 2.390625, 2.40234375, 2.4140625, 2.42578125, 2.4375, 2.44921875, 2.4609375, 2.47265625, 2.484375, 2.49609375, 2.5078125, 2.51953125, 2.53125, 2.54296875, 2.5546875, 2.56640625, 2.578125, 2.58984375, 2.6015625, 2.61328125, 2.625, 2.63671875, 2.6484375, 2.66015625, 2.671875, 2.68359375, 2.6953125, 2.70703125, 2.71875, 2.73046875, 2.7421875, 2.75390625, 2.765625, 2.77734375, 2.7890625, 2.80078125, 2.8125, 2.82421875, 2.8359375, 2.84765625, 2.859375, 2.87109375, 2.8828125, 2.89453125, 2.90625, 2.91796875, 2.9296875, 2.94140625, 2.953125, 2.96484375, 2.9765625, 2.98828125, 3.0, 3.01171875, 3.0234375, 3.03515625, 3.046875, 3.05859375, 3.0703125, 3.08203125, 3.09375, 3.10546875, 3.1171875, 3.12890625, 3.140625, 3.15234375, 3.1640625, 3.17578125, 3.1875, 3.19921875, 3.2109375, 3.22265625, 3.234375, 3.24609375, 3.2578125, 3.26953125, 3.28125, 3.29296875, 3.3046875, 3.31640625, 3.328125, 3.33984375, 3.3515625, 3.36328125, 3.375, 3.38671875, 3.3984375, 3.41015625, 3.421875, 3.43359375, 3.4453125, 3.45703125, 3.46875, 3.48046875, 3.4921875, 3.50390625, 3.515625, 3.52734375, 3.5390625, 3.55078125, 3.5625, 3.57421875, 3.5859375, 3.59765625, 3.609375, 3.62109375, 3.6328125, 3.64453125, 3.65625, 3.66796875, 3.6796875, 3.69140625, 3.703125, 3.71484375, 3.7265625, 3.73828125, 3.75, 3.76171875, 3.7734375, 3.78515625, 3.796875, 3.80859375, 3.8203125, 3.83203125, 3.84375, 3.85546875, 3.8671875, 3.87890625, 3.890625, 3.90234375, 3.9140625, 3.92578125, 3.9375, 3.94921875, 3.9609375, 3.97265625, 3.984375, 3.99609375, 4.0078125, 4.01953125, 4.03125, 4.04296875, 4.0546875, 4.06640625, 4.078125, 4.08984375, 4.1015625, 4.11328125, 4.125, 4.13671875, 4.1484375, 4.16015625, 4.171875, 4.18359375, 4.1953125, 4.20703125, 4.21875, 4.23046875, 4.2421875, 4.25390625, 4.265625, 4.27734375, 4.2890625, 4.30078125, 4.3125, 4.32421875, 4.3359375, 4.34765625, 4.359375, 4.37109375, 4.3828125, 4.39453125, 4.40625, 4.41796875, 4.4296875, 4.44140625, 4.453125, 4.46484375, 4.4765625, 4.48828125, 4.5, 4.51171875, 4.5234375, 4.53515625, 4.546875, 4.55859375, 4.5703125, 4.58203125, 4.59375, 4.60546875, 4.6171875, 4.62890625, 4.640625, 4.65234375, 4.6640625, 4.67578125, 4.6875, 4.69921875, 4.7109375, 4.72265625, 4.734375, 4.74609375, 4.7578125, 4.76953125, 4.78125, 4.79296875, 4.8046875, 4.81640625, 4.828125, 4.83984375, 4.8515625, 4.86328125, 4.875, 4.88671875, 4.8984375, 4.91015625, 4.921875, 4.93359375, 4.9453125, 4.95703125, 4.96875, 4.98046875, 4.9921875, 5.00390625, 5.015625, 5.02734375, 5.0390625, 5.05078125, 5.0625, 5.07421875, 5.0859375, 5.09765625, 5.109375, 5.12109375, 5.1328125, 5.14453125, 5.15625, 5.16796875, 5.1796875, 5.19140625, 5.203125, 5.21484375, 5.2265625, 5.23828125, 5.25, 5.26171875, 5.2734375, 5.28515625, 5.296875, 5.30859375, 5.3203125, 5.33203125, 5.34375, 5.35546875, 5.3671875, 5.37890625, 5.390625, 5.40234375, 5.4140625, 5.42578125, 5.4375, 5.44921875, 5.4609375, 5.47265625, 5.484375, 5.49609375, 5.5078125, 5.51953125, 5.53125, 5.54296875, 5.5546875, 5.56640625, 5.578125, 5.58984375, 5.6015625, 5.61328125, 5.625, 5.63671875, 5.6484375, 5.66015625, 5.671875, 5.68359375, 5.6953125, 5.70703125, 5.71875, 5.73046875, 5.7421875, 5.75390625, 5.765625, 5.77734375, 5.7890625, 5.80078125, 5.8125, 5.82421875, 5.8359375, 5.84765625, 5.859375, 5.87109375, 5.8828125, 5.89453125, 5.90625, 5.91796875, 5.9296875, 5.94140625, 5.953125, 5.96484375, 5.9765625, 5.98828125, 6.0]], "dtype": "float32"}], "e": {"__nd__": [[-8.43964897967453e-11, -9.39162705937413e-11, -1.0447273601776885e-10, -1.161741328137289e-10, -1.2914069813518836e-10, -1.4350384769379332e-10, -1.5940776476597307e-10, -1.7701214105603214e-10, -1.9649142035671474e-10, -2.1803753247340296e-10, -2.4186172509210735e-10, -2.681952437910695e-10, -2.972918022869919e-10, -3.294305939149922e-10, -3.64916513673208e-10, -4.040849599373786e-10, -4.4730222303890343e-10, -4.949707865797848e-10, -5.475299102997155e-10, -6.054617918138661e-10, -6.692931764362697e-10, -7.395998258274972e-10, -8.170122911543842e-10, -9.02219021714501e-10, -9.95972193607031e-10, -1.0990933718701967e-09, -1.2124796722190467e-09, -1.337109534205183e-09, -1.4740513254452026e-09, -1.624469447847332e-09, -1.789633330417928e-09, -1.970926755134883e-09, -2.1698567387318235e-09, -2.38806530106217e-09, -2.6273387909725443e-09, -2.88962187511288e-09, -3.1770301944789026e-09, -3.4918636870884256e-09, -3.836623463371325e-09, -4.2140269052026724e-09, -4.627026317649552e-09, -5.078827580717871e-09, -5.572910133366804e-09, -6.113050954326127e-09, -6.703346766556706e-09, -7.3482393503354615e-09, -8.052546185410847e-09, -8.821484875909391e-09, -9.660708677472485e-09, -1.057634158030396e-08, -1.1575009395414781e-08, -1.2663885051722445e-08, -1.3850727675901453e-08, -1.514392700130429e-08, -1.6552554882309778e-08, -1.8086414144136143e-08, -1.975609187354621e-08, -2.157302603222888e-08, -2.3549564076574825e-08, -2.5699028682879543e-08, -2.8035794130687464e-08, -3.057536446249287e-08, -3.333445519615452e-08, -3.6331076813667096e-08, -3.958463778985788e-08, -4.3116038739299256e-08, -4.694778610314643e-08, -5.1104098730547776e-08, -5.56110357763373e-08, -6.049663170415442e-08, -6.579102063142273e-08, -7.152659975417919e-08, -7.77381785610487e-08, -8.44631387053596e-08, -9.174162585168233e-08, -9.961674862779546e-08, -1.0813473494408754e-07, -1.1734520910522406e-07, -1.27301362340404e-07, -1.3806021570417215e-07, -1.4968289008265856e-07, -1.6223485488353617e-07, -1.757862406748245e-07, -1.90412123401984e-07, -2.0619282281586493e-07, -2.2321434300920373e-07, -2.415685855794436e-07, -2.6135390385206847e-07, -2.826753302542784e-07, -3.056451873817423e-07, -3.303833864265471e-07, -3.5701802403309557e-07, -3.85685808623748e-07, -4.165326288330107e-07, -4.4971415036343387e-07, -4.853963559980912e-07, -5.237562277216057e-07, -5.649823151543387e-07, -6.092757303122198e-07, -6.568503749804222e-07, -7.079341912685777e-07, -7.627697868883843e-07, -8.216153446483077e-07, -8.847454182614456e-07, -9.524520692139049e-07, -1.0250457762595033e-06, -1.1028563449144713e-06, -1.1862343853863422e-06, -1.2755522220686544e-06, -1.371205144096166e-06, -1.4736127695869072e-06, -1.5832201825105585e-06, -1.7004998653646908e-06, -1.8259523812957923e-06, -1.9601088752096985e-06, -2.1035316422057804e-06, -2.256817424495239e-06, -2.4205969566537533e-06, -2.595539172034478e-06, -2.7823516575153917e-06, -2.9817826998623786e-06, -3.1946249237080337e-06, -3.421715518925339e-06, -3.663940333353821e-06, -3.922234554920578e-06, -4.1975877138611395e-06, -4.49104391009314e-06, -4.803705905942479e-06, -5.136737399880076e-06, -5.491367119248025e-06, -5.8688906392490026e-06, -6.270674475672422e-06, -6.6981601776205935e-06, -7.152865691750776e-06, -7.636391273990739e-06, -8.150422218022868e-06, -8.696734766999725e-06, -9.277195204049349e-06, -9.893772585201077e-06, -1.0548533282417338e-05, -1.1243655535508879e-05, -1.1981425814155955e-05, -1.2764250641339459e-05, -1.3594656593340915e-05, -1.4475298485194799e-05, -1.540896664664615e-05, -1.6398587831645273e-05, -1.744723522278946e-05, -1.8558133888291195e-05, -1.9734665329451673e-05, -2.098037839459721e-05, -2.2298991098068655e-05, -2.369440335314721e-05, -2.5170695153065026e-05, -2.6732150217867456e-05, -2.8383243261487223e-05, -3.0128667276585475e-05, -3.197332989657298e-05, -3.3922362490557134e-05, -3.598114199121483e-05, -3.81552817998454e-05, -4.0450649976264685e-05, -4.287339106667787e-05, -4.542990427580662e-05, -4.812689439859241e-05, -5.097134271636605e-05, -5.397055429057218e-05, -5.713214341085404e-05, -6.046404814696871e-05, -6.397456309059635e-05, -6.767232116544619e-05, -7.156632636906579e-05, -7.566597196273506e-05, -7.998102228157222e-05, -8.452167094219476e-05, -8.929851173888892e-05, -9.43225750233978e-05, -9.960534953279421e-05, -0.00010515878238948062, -0.00011099529365310445, -0.00011712779814843088, -0.00012356971274130046, -0.00013033501454629004, -0.0001374381681671366, -0.00014489423483610153, -0.0001527188578620553, -0.00016092821897473186, -0.0001695391838438809, -0.0001785692002158612, -0.0001880363852251321, -0.00019795949629042298, -0.00020835800387430936, -0.00021925207693129778, -0.0002306625829078257, -0.0002426111896056682, -0.00025512027787044644, -0.00026821301435120404, -0.0002819134097080678, -0.00029624628950841725, -0.00031123729422688484, -0.0003269129665568471, -0.00034330078051425517, -0.000360429025022313, -0.00037832697853446007, -0.00039702493813820183, -0.00041655407403595746, -0.000436946633271873, -0.00045823585242033005, -0.00048045607400126755, -0.0005036426591686904, -0.000527832075022161, -0.0005530619528144598, -0.0005793710006400943, -0.0006067991489544511, -0.000635387550573796, -0.0006651784642599523, -0.000696215545758605, -0.0007285435567609966, -0.0007622085977345705, -0.0007972581661306322, -0.0008337409817613661, -0.0008717072196304798, -0.0009112083935178816, -0.000952297355979681, -0.000995028531178832, -0.001039457623846829, -0.0010856420267373323, -0.001133640413172543, -0.0011835130862891674, -0.0012353219790384173, -0.0012891304213553667, -0.0013450032565742731, -0.0014030073070898652, -0.00146321055945009, -0.0015256829792633653, -0.001590496045537293, -0.0016577229835093021, -0.0017274385318160057, -0.0017997194081544876, -0.0018746439600363374, -0.00195229216478765, -0.002032746095210314, -0.0021160889882594347, -0.0022024065256118774, -0.0022917857859283686, -0.002384315710514784, -0.002480087336152792, -0.002579193329438567, -0.0026817279867827892, -0.002787787700071931, -0.002897470723837614, -0.00301087717525661, -0.003128108801320195, -0.003249269677326083, -0.0033744650427252054, -0.0035038029309362173, -0.003637392073869705, -0.0037753439974039793, -0.003917771857231855, -0.004064790438860655, -0.0042165168561041355, -0.004373068921267986, -0.004534567706286907, -0.0047011347487568855, -0.0048728943802416325, -0.005049972329288721, -0.005232496187090874, -0.005420594941824675, -0.005614399444311857, -0.005814042408019304, -0.00601965794339776, -0.006231382023543119, -0.00644935155287385, -0.006673706229776144, -0.006904585752636194, -0.007142132613807917, -0.007386489771306515, -0.0076378025114536285, -0.007896216586232185, -0.008161879144608974, -0.00843493826687336, -0.008715544827282429, -0.00900384970009327, -0.009300004690885544, -0.009604162536561489, -0.009916476905345917, -0.010237102396786213, -0.010566195473074913, -0.010903911665081978, -0.011250408366322517, -0.011605842970311642, -0.01197037287056446, -0.01234415639191866, -0.012727353721857071, -0.013120122253894806, -0.013522620312869549, -0.013935009017586708, -0.014357444830238819, -0.01479008886963129, -0.015233096666634083, -0.015686629340052605, -0.016150841489434242, -0.016625892370939255, -0.01711193472146988, -0.0176091268658638, -0.018117621541023254, -0.01863757148385048, -0.01916912943124771, -0.01971244253218174, -0.020267663523554802, -0.02083493582904339, -0.02141440659761429, -0.022006217390298843, -0.022610507905483246, -0.023227421566843987, -0.02385708875954151, -0.024499645456671715, -0.025155222043395042, -0.02582394704222679, -0.02650594152510166, -0.027201330289244652, -0.027910226956009865, -0.028632748872041702, -0.02936900407075882, -0.030119096860289574, -0.03088313154876232, -0.03166120499372482, -0.03245340660214424, -0.03325982764363289, -0.03408054634928703, -0.03491564467549324, -0.03576519340276718, -0.03662925586104393, -0.03750789165496826, -0.03840115666389465, -0.039309099316596985, -0.04023176431655884, -0.0411691777408123, -0.04212137311697006, -0.043088365346193314, -0.044070176780223846, -0.04506680369377136, -0.04607824608683586, -0.047104496508836746, -0.048145536333322525, -0.04920133203268051, -0.05027185007929802, -0.05135704576969147, -0.05245686694979668, -0.053571246564388275, -0.05470011383295059, -0.05584338679909706, -0.057000964879989624, -0.05817274749279022, -0.0593586266040802, -0.060558468103408813, -0.061772141605615616, -0.06299949437379837, -0.06424037367105484, -0.065494604408741, -0.06676200777292252, -0.06804238259792328, -0.06933553516864777, -0.07064122706651688, -0.0719592496752739, -0.07328933477401733, -0.0746312364935875, -0.07598468661308289, -0.07734939455986023, -0.07872506231069565, -0.08011137694120407, -0.08150801062583923, -0.08291462808847427, -0.08433087170124054, -0.08575636893510818, -0.08719073981046677, -0.08863358199596405, -0.0900844857096672, -0.09154301136732101, -0.09300872683525085, -0.09448117017745972, -0.0959598571062088, -0.09744430333375931, -0.09893399477005005, -0.10042842477560043, -0.10192703455686569, -0.10342928767204285, -0.10493460297584534, -0.106442391872406, -0.1079520583152771, -0.10946298390626907, -0.11097452789545059, -0.11248603463172913, -0.11399684846401215, -0.11550627648830414, -0.11701361835002899, -0.1185181587934494, -0.12001916766166687, -0.12151589244604111, -0.12300756573677063, -0.12449341267347336, -0.12597262859344482, -0.12744440138339996, -0.12890790402889252, -0.13036228716373444, -0.13180670142173767, -0.1332402527332306, -0.13466206192970276, -0.13607120513916016, -0.13746678829193115, -0.13884785771369934, -0.14021345973014832, -0.14156262576580048, -0.14289440214633942, -0.14420776069164276, -0.14550170302391052, -0.14677521586418152, -0.14802725613117218, -0.14925678074359894, -0.15046271681785583, -0.1516440212726593, -0.1527995765209198, -0.15392829477787018, -0.1550290733575821, -0.156100794672966, -0.15714232623577118, -0.1581525206565857, -0.15913023054599762, -0.16007430851459503, -0.16098356246948242, -0.16185683012008667, -0.16269293427467346, -0.1634906530380249, -0.16424880921840668, -0.16496619582176208, -0.16564157605171204, -0.16627375781536102, -0.16686148941516876, -0.16740356385707855, -0.1678987294435501, -0.16834576427936554, -0.16874340176582336, -0.1690904200077057, -0.16938556730747223, -0.1696275770664215, -0.16981522738933563, -0.16994725167751312, -0.1700224131345749, -0.17003944516181946, -0.16999711096286774, -0.16989415884017944, -0.1697293519973755, -0.16950145363807678, -0.16920921206474304, -0.16885142028331757, -0.16842681169509888, -0.16793420910835266, -0.16737237572669983, -0.16674008965492249, -0.16603617370128632, -0.1652594357728958, -0.16440866887569427, -0.16348272562026978, -0.1624804139137268, -0.16140060126781464, -0.16024214029312134, -0.15900389850139618, -0.15768475830554962, -0.15628361701965332, -0.15479938685894012, -0.15323098003864288, -0.15157733857631683, -0.1498374342918396, -0.14801020920276642, -0.1460946649312973, -0.14408980309963226, -0.1419946551322937, -0.13980825245380402, -0.137529656291008, -0.13515795767307281, -0.13269223272800446, -0.13013161718845367, -0.12747526168823242, -0.12472231686115265, -0.12187197804450989, -0.11892344057559967, -0.11587595194578171, -0.11272875219583511, -0.10948113352060318, -0.1061323955655098, -0.10268186777830124, -0.09912890940904617, -0.09547289460897446, -0.09171322733163834, -0.08784935623407364, -0.0838807225227356, -0.07980682700872421, -0.07562718540430069, -0.07134132832288742, -0.06694883853197098, -0.06244931370019913, -0.057842377573251724, -0.05312768742442131, -0.048304930329322815, -0.04337382689118385, -0.03833411633968353, -0.03318557143211365, -0.027928002178668976, -0.022561240941286087, -0.017085149884223938, -0.011499624699354172, -0.0058045899495482445, 0.0, 0.0059141600504517555, 0.011937875300645828, 0.018071100115776062, 0.024313759058713913, 0.030665747821331024, 0.03712692856788635, 0.04369713366031647, 0.05037617310881615, 0.057163819670677185, 0.06405981630086899, 0.07106387615203857, 0.07817568629980087, 0.08539491146802902, 0.09272117167711258, 0.10015406459569931, 0.10769317299127579, 0.1153380274772644, 0.12308814376592636, 0.13094301521778107, 0.13890211284160614, 0.14696484804153442, 0.15513062477111816, 0.1633988469839096, 0.17176885902881622, 0.1802399903535843, 0.1888115555047989, 0.19748280942440033, 0.2062530219554901, 0.21512143313884735, 0.22408723831176758, 0.23314963281154633, 0.24230776727199554, 0.2515608072280884, 0.2609078288078308, 0.2703479826450348, 0.2798803448677063, 0.28950393199920654, 0.2992178499698639, 0.3090210556983948, 0.3189125657081604, 0.328891396522522, 0.3389565050601959, 0.3491068482398987, 0.3593413829803467, 0.3696589767932892, 0.3800585865974426, 0.39053910970687866, 0.40109938383102417, 0.4117383360862732, 0.4224547743797302, 0.43324756622314453, 0.4441155791282654, 0.4550575613975525, 0.4660724103450775, 0.47715887427330017, 0.48831579089164734, 0.4995419383049011, 0.5108360648155212, 0.5221970081329346, 0.5336235165596008, 0.5451143980026245, 0.5566683411598206, 0.5682841539382935, 0.5799605846405029, 0.5916963219642639, 0.6034902334213257, 0.6153410077095032, 0.6272473931312561, 0.639208197593689, 0.6512220501899719, 0.6632878184318542, 0.6754042506217957, 0.6875700354576111, 0.6997839212417603, 0.7120447754859924, 0.7243512272834778, 0.7367021441459656, 0.7490963339805603, 0.7615324258804321, 0.7740093469619751, 0.7865258455276489, 0.7990806698799133, 0.8116726875305176, 0.8243007063865662, 0.8369635343551636, 0.8496599793434143, 0.8623889088630676, 0.8751491904258728, 0.8879396915435791, 0.900759220123291, 0.9136067032814026, 0.9264810085296631, 0.9393810033798218, 0.9523057341575623, 0.965254008769989, 0.9782248139381409, 0.9912170767784119, 1.0042297840118408, 1.0172618627548218, 1.030312418937683, 1.0433802604675293, 1.056464672088623, 1.0695644617080688, 1.0826787948608398, 1.0958067178726196, 1.1089472770690918, 1.12209951877594, 1.1352627277374268, 1.1484358310699463, 1.1616181135177612, 1.1748086214065552, 1.1880066394805908, 1.2012112140655518, 1.2144216299057007, 1.2276370525360107, 1.2408568859100342, 1.2540801763534546, 1.2673062086105347, 1.2805343866348267, 1.2937639951705933, 1.3069941997528076, 1.3202245235443115, 1.3334541320800781, 1.3466825485229492, 1.359909176826477, 1.3731331825256348, 1.3863542079925537, 1.3995715379714966, 1.4127846956253052, 1.4259932041168213, 1.4391963481903076, 1.452393889427185, 1.4655849933624268, 1.4787695407867432, 1.491946816444397, 1.5051164627075195, 1.5182780027389526, 1.5314310789108276, 1.5445753335952759, 1.5577104091644287, 1.5708357095718384, 1.5839511156082153, 1.5970561504364014, 1.6101505756378174, 1.6232340335845947, 1.6363062858581543, 1.649366855621338, 1.6624157428741455, 1.675452470779419, 1.6884769201278687, 1.7014888525009155, 1.7144880294799805, 1.7274740934371948, 1.7404471635818481, 1.7534067630767822, 1.766352891921997, 1.7792853116989136, 1.7922039031982422, 1.8051085472106934, 1.817999005317688, 1.8308753967285156, 1.8437373638153076, 1.8565850257873535, 1.8694181442260742, 1.8822367191314697, 1.8950406312942505, 1.9078298807144165, 1.9206044673919678, 1.9333642721176147, 1.9461092948913574, 1.9588394165039062, 1.9715548753738403, 1.9842554330825806, 1.996941089630127, 2.0096120834350586, 2.022268295288086, 2.034909725189209, 2.0475363731384277, 2.0601484775543213, 2.0727458000183105, 2.0853285789489746, 2.0978968143463135, 2.1104507446289062, 2.122990131378174, 2.1355154514312744, 2.14802622795105, 2.1605231761932373, 2.1730058193206787, 2.1854746341705322, 2.197929859161377, 2.2103710174560547, 2.2227985858917236, 2.235212802886963, 2.2476136684417725, 2.2600009441375732, 2.2723753452301025, 2.2847366333007812, 2.2970850467681885, 2.3094208240509033, 2.3217437267303467, 2.3340542316436768, 2.3463525772094727, 2.3586385250091553, 2.370912551879883, 2.3831746578216553, 2.3954248428344727, 2.407663583755493, 2.419890880584717, 2.4321067333221436, 2.4443116188049316, 2.456505298614502, 2.468688488006592, 2.480860710144043, 2.4930224418640137, 2.505173921585083, 2.517314910888672, 2.5294461250305176, 2.541567325592041, 2.5536789894104004, 2.5657808780670166, 2.577873468399048, 2.589956760406494, 2.6020307540893555, 2.614096164703369, 2.626152515411377, 2.638200283050537, 2.6502397060394287, 2.6622707843780518, 2.6742937564849854, 2.6863086223602295, 2.6983156204223633, 2.710314989089966, 2.722306966781616, 2.7342913150787354, 2.7462685108184814, 2.7582385540008545, 2.7702016830444336, 2.7821578979492188, 2.794107437133789, 2.8060505390167236, 2.8179874420166016, 2.8299179077148438, 2.8418421745300293, 2.8537607192993164, 2.865673065185547, 2.877579927444458, 2.88948130607605, 2.9013772010803223, 2.9132676124572754, 2.9251530170440674, 2.937033176422119, 2.948908567428589, 2.9607789516448975, 2.972644805908203, 2.9845058917999268, 2.9963626861572266, 3.0082149505615234, 3.0200629234313965, 3.031907081604004, 3.0437469482421875, 3.0555827617645264, 3.0674149990081787, 3.0792434215545654, 3.0910682678222656, 3.1028895378112793, 3.1147074699401855, 3.1265218257904053, 3.138333320617676, 3.1501412391662598, 3.1619462966918945, 3.17374849319458, 3.1855475902557373, 3.1973440647125244, 3.2091376781463623, 3.220928907394409, 3.232717275619507, 3.2445032596588135, 3.256286859512329, 3.2680680751800537, 3.2798469066619873, 3.291623830795288, 3.303398370742798, 3.315171003341675, 3.32694149017334, 3.338710069656372, 3.3504767417907715, 3.362241744995117, 3.374005079269409, 3.3857665061950684, 3.397526264190674, 3.4092845916748047, 3.421041250228882, 3.4327964782714844, 3.4445502758026123, 3.4563026428222656, 3.4680538177490234, 3.4798035621643066, 3.4915521144866943, 3.5032994747161865, 3.515045642852783, 3.5267906188964844, 3.538534641265869, 3.5502777099609375, 3.5620195865631104, 3.573760509490967, 3.585500478744507, 3.5972397327423096, 3.608978033065796, 3.620715379714966, 3.6324520111083984, 3.6441879272460938, 3.6559231281280518, 3.6676576137542725, 3.6793911457061768, 3.691124439239502, 3.7028567790985107, 3.7145886421203613, 3.7263197898864746, 3.738050699234009, 3.7497806549072266, 3.7615103721618652, 3.7732396125793457, 3.784968137741089, 3.796696424484253, 3.808424234390259, 3.8201515674591064, 3.831878423690796, 3.8436050415039062, 3.8553314208984375, 3.8670570850372314, 3.8787827491760254, 3.890507936477661, 3.9022326469421387, 3.913957357406616, 3.9256815910339355, 3.937405586242676, 3.949129343032837, 3.960852861404419, 3.972576379776001, 3.984299421310425, 3.9960222244262695, 4.007744789123535, 4.019467353820801, 4.031189441680908, 4.042911529541016, 4.054633617401123, 4.066355228424072, 4.0780768394470215, 4.089798450469971, 4.101519584655762, 4.113240718841553, 4.124961853027344, 4.136682987213135, 4.148403644561768, 4.1601243019104, 4.171844959259033, 4.183565139770508, 4.195285797119141, 4.207005977630615, 4.21872615814209, 4.2304463386535645, 4.242166519165039, 4.253886699676514, 4.26560640335083, 4.2773261070251465, 4.289046287536621, 4.3007659912109375, 4.312485694885254, 4.324204921722412, 4.3359246253967285, 4.347644329071045, 4.359363555908203, 4.3710832595825195, 4.382802486419678, 4.394522190093994, 4.406241416931152, 4.4179606437683105, 4.429679870605469, 4.441399097442627, 4.453118324279785, 4.464837551116943, 4.476556777954102, 4.488275527954102, 4.49999475479126, 4.511713981628418, 4.523433208465576, 4.535151958465576, 4.546871185302734, 4.558589935302734, 4.570309162139893, 4.582027912139893, 4.593747138977051, 4.605465888977051, 4.617185115814209, 4.628903865814209, 4.640622615814209, 4.652341842651367, 4.664060592651367, 4.675779342651367, 4.687498092651367, 4.699217319488525, 4.710936069488525, 4.722654819488525, 4.734373569488525, 4.746092796325684, 4.757811546325684, 4.769530296325684, 4.781249046325684, 4.792967796325684, 4.804686546325684, 4.816405296325684, 4.828124523162842, 4.839843273162842, 4.851562023162842, 4.863280773162842, 4.874999523162842, 4.886718273162842, 4.898437023162842, 4.910155773162842, 4.921874523162842, 4.933593273162842, 4.945312023162842, 4.957030773162842, 4.968749523162842, 4.980468273162842, 4.992187023162842, 5.00390625, 5.015625, 5.02734375, 5.0390625, 5.05078125, 5.0625, 5.07421875, 5.0859375, 5.09765625, 5.109375, 5.12109375, 5.1328125, 5.14453125, 5.15625, 5.16796875, 5.1796875, 5.19140625, 5.203125, 5.21484375, 5.2265625, 5.23828125, 5.25, 5.26171875, 5.2734375, 5.28515625, 5.296875, 5.30859375, 5.3203125, 5.33203125, 5.34375, 5.35546875, 5.3671875, 5.37890625, 5.390625, 5.40234375, 5.4140625, 5.42578125, 5.4375, 5.44921875, 5.4609375, 5.47265625, 5.484375, 5.49609375, 5.5078125, 5.51953125, 5.53125, 5.54296875, 5.5546875, 5.56640625, 5.578125, 5.58984375, 5.6015625, 5.61328125, 5.625, 5.63671875, 5.6484375, 5.66015625, 5.671875, 5.68359375, 5.6953125, 5.70703125, 5.71875, 5.73046875, 5.7421875, 5.75390625, 5.765625, 5.77734375, 5.7890625, 5.80078125, 5.8125, 5.82421875, 5.8359375, 5.84765625, 5.859375, 5.87109375, 5.8828125, 5.89453125, 5.90625, 5.91796875, 5.9296875, 5.94140625, 5.953125, 5.96484375, 5.9765625, 5.98828125, 6.0]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(gelu_tanh, json.loads(_TESTS), 'triton', *[0.002, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")